# Libraries

In [29]:
import os
# Set up the download directory to be the same as the Python script directory
try:
    #Python file
    download_directory = os.path.dirname(os.path.realpath(__file__))
except:
    #Jupyter notebook
    download_directory = os.getcwd()

# Define the path for the new folder
download_directory = os.path.join(download_directory, "Download_folder")


# Create the folder if it does not exist
if not os.path.exists(download_directory):
    os.makedirs(download_directory)

download_directory

'h:\\Shared drives\\DevOps Projects\\Python projecs\\python-automations\\Amz - Wmt Pricing\\Download_folder'

In [30]:
import sys

def is_python_script():
    try:
        os.path.basename(__file__)
        print("Running python file...")
        return True
    except NameError:
        return False
    
# Navigate up from the current directory and then into the "Modules" directory

is_python_script = is_python_script()

if is_python_script:
    # Get the directory of the current Python file
    current_directory = os.path.dirname(os.path.realpath(__file__))

    # Navigate up from the current directory and then into the "Modules" directory
    path_to_append = os.path.join(current_directory, '..', 'modules')

else:
    path_to_append = os.path.join(os.getcwd(), '..', 'modules')

# Normalize the path to resolve any ".."
normalized_path = os.path.normpath(path_to_append)

# Append the normalized path to sys.path
sys.path.append(normalized_path)

In [31]:
#Import libraries
import pandas as pd
pd.set_option('display.max_columns', None)

import GoogleAPI_functions as gAPI #Here we also have all required libraries for google API.
import hybris as hybris
import pricing_module as pricing
# import math
import DW_connection as dw
import FTP_Connection as ftp_server

import numpy as np
from datetime import datetime, date, timedelta
from dateutil.relativedelta import relativedelta

import json
import xml.etree.ElementTree as ET
import re
pd.options.mode.copy_on_write = True

# Params

In [32]:
days_before = 7

test = False
min_margin_update_prices = 0.059
max_margin_update_prices = 0.203

#New params to implement as checks: if any of these is met, then stop the code and let me know.
threshold_new_nodes_flag = 20000
threshold_price_increases_flag = 10000
threshold_price_decreases_flag = 10000

#This is used in the errors part
perc_failure_flag = 0.015

# Dates calcs

In [33]:
today_str = pd.to_datetime('today').strftime('%Y-%m-%d')

date_str = today_str

start_date_str = (pd.to_datetime(date_str) - relativedelta(days=days_before)).strftime('%Y-%m-%d')
end_date_str = date_str
dates_str_list = pd.date_range(start=start_date_str, end=end_date_str).strftime('%Y-%m-%d').tolist()
print(dates_str_list)
yesterday_str = (pd.to_datetime(date_str) - relativedelta(days=1)).strftime('%Y-%m-%d')
last_week_str = (pd.to_datetime(date_str) - relativedelta(days=7)).strftime('%Y-%m-%d')

['2026-03-11', '2026-03-12', '2026-03-13', '2026-03-14', '2026-03-15', '2026-03-16', '2026-03-17', '2026-03-18']


# Input files

## Drive connection

In [34]:
#Connect to google drive

SCOPES = ["https://www.googleapis.com/auth/drive.metadata.readonly", "https://www.googleapis.com/auth/drive.readonly",
          "https://www.googleapis.com/auth/drive",'https://www.googleapis.com/auth/spreadsheets']

file_name = "google_credentials.json"
folder_name = "credentials"

# credentials_file = os.path.join('..', folder_name, file_name)
credentials_file = gAPI.create_path_cred()

service, service_sheets = gAPI.connect_drive(SCOPES,credentials_file)

## Buybox

In [35]:
sqlQuery = f"""SELECT *
FROM pricing_tests.walmart_item_report WHERE date = (SELECT MAX(date) FROM pricing_tests.walmart_item_report WHERE date <= '{date_str}') 
    AND "1p_offer_status" = 'PUBLISHED'"""

dfWmtItemReport = dw.runQuery(sqlQuery, newCredentials=True)
dfWmtItemReport.rename(columns={"item_id": "Item ID", "product_code": "Product Code"}, inplace=True)
dfWmtItemReport

,walmart_item_number,Item ID,gtin,product_code_wmt,unit_cost,customer_review_count,content_quality_score,1p_offer_status,walmart_buybox_winner,offer_price,walmart_dot_com_price,date,Product Code,unpublished_reason,average_customer_rating
0,683896256,15670401081,03651414172967,ADVN0581624575E~wm,147.38,NaN,0.963346,PUBLISHED,Yes,191.59,191.59,2026-03-17,ADVN0581624575E,None,NaN
1,663735834,250706329,00844402098537,ADVN0581626575E,208.13,NaN,0.981000,PUBLISHED,No,250.07,169.02,2026-03-17,ADVN0581626575E,None,NaN
2,672160033,991738091,08997020617023,ACCE0041523570E,92.27,3.0,0.989500,PUBLISHED,Yes,105.48,105.48,2026-03-17,ACCE0041523570E,None,5.0
3,664909698,428656653,08997020617030,ACCE0071523575E,81.98,9.0,0.983650,PUBLISHED,No,110.09,115.47,2026-03-17,ACCE0071523575E,None,4.8
4,672805389,943271635,08997020614015,ACCE0131318560V,62.35,22.0,0.975792,PUBLISHED,No,71.07,79.93,2026-03-17,ACCE0131318560V,None,4.6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72472,668944630,5180569485,00746573263809,ACHI0021722555V,69.39,165.0,0.983500,PUBLISHED,Yes,82.45,82.45,2026-03-17,ACHI0021722555V,None,4.5
72473,668944614,5142541577,00746573262567,ACHI0021722565H,74.42,115.0,0.983500,PUBLISHED,Yes,91.13,91.13,2026-03-17,ACHI0021722565H,None,4.5
72474,669408072,5433454236,00746573262406,ACHI0021723545V,55.05,9.0,0.983500,PUBLISHED,No,68.93,63.29,2026-03-17,ACHI0021723545V,None,4.9
72475,669408075,5444924898,00746573262512,ACHI0021723550V,71.89,32.0,0.983500,PUBLISHED,Yes,69.24,69.24,2026-03-17,ACHI0021723550V,None,4.2


In [36]:
dfAvgWmtPrice = dfWmtItemReport.groupby("Product Code").agg({"offer_price":"mean","unit_cost":"mean"}).reset_index()
dfAvgWmtPrice

,Product Code,offer_price,unit_cost
0,ACCE0041523570E,105.480,92.270
1,ACCE0071523575E,110.090,81.980
2,ACCE0131318560V,71.070,62.350
3,ACCE0131419550H,66.750,39.290
4,ACCE0131820535YXL,68.360,61.790
...,...,...,...
55095,ZETA0411823565HXL,98.900,80.740
55096,ZETA0411825555VXL,100.440,79.570
55097,ZETA0411826560V,111.045,87.095
55098,ZETA0411826565H,118.285,96.270


## MAP

In [37]:
queryMAP= f"""  SELECT map_prices.sku,
    map_prices.map
   FROM pricing_tests.map_prices
  WHERE map_prices.date = (SELECT MAX(date) FROM pricing_tests.map_prices WHERE date <= '{date_str}')"""

dfMAP = dw.runQuery(queryMAP, newCredentials = True)

dfMAP = dfMAP.rename(columns={'sku': 'Product Code', 'map': 'MAP'})

dfMAP

,Product Code,MAP
0,ATTU0531527560W,239.00
1,ATTU0531727540WXL,249.00
2,ATTU0531730545W,309.00
3,ATTU0531731535YXL,279.00
4,ATTU0531831540YXL,429.00
...,...,...
16167,TOYO3371728575E,379.13
16168,TOYO3371828575E,435.41
16169,TOYO3371629575E,315.19
16170,TOYO3371631575E,361.68


## Current DSV

In [38]:
compare_dsvs = False

In [39]:
def read_dsv(read_date = None):
    folderWmtDSV = "1piuawZRpppmoD-Qdkd1IUj3x4rs-LKny"
    readCols = ["SKU", "Price", "Source"]
    dtype = {"SKU": str, "Price": float,"Source": str}
    dfWmatDSVFiles = gAPI.get_all_files_folder(service, folderWmtDSV)

    dfWmatDSVFiles["Date"] = dfWmatDSVFiles["Name"].apply(lambda x: re.search(r"\d{4}-\d{2}-\d{2}", x).group())
    dfWmatDSVFiles["Date"] = pd.to_datetime(dfWmatDSVFiles["Date"], format = "%Y-%m-%d")

    dfWmatDSVFiles = dfWmatDSVFiles[(dfWmatDSVFiles["Date"] >= '2025-12-30')]

    if read_date == None:
        read_date = max(dfWmatDSVFiles["Date"])
        print(f"Max DSV date: {read_date}")

    dfWmatDSVFiles = dfWmatDSVFiles[dfWmatDSVFiles["Date"] == read_date]

    idWmtFile = dfWmatDSVFiles["ID"].iloc[0]

    dfCurrDSVOriginal = gAPI.get_df_of_file(service, idWmtFile, "csv", read_cols = readCols, dtype = dtype)

    #Drop duplicates
    dfCurrDSVOriginal  = dfCurrDSVOriginal.drop_duplicates(subset=["SKU", "Source"], keep="first")

    dfCurrDSVOriginal = dfCurrDSVOriginal.rename(columns={"SKU": "sku", "Price": "walmart_dsv_price","Source":"source"})


    # queryCurrDSV = f"""SELECT *
    # FROM pricing_tests.walmart_dsv WHERE date = (SELECT MAX(date) FROM pricing_tests.walmart_dsv WHERE date <= '{date_str}')"""
    # dfCurrDSVOriginal = dw.runQuery(queryCurrDSV, newCredentials=True)

    dfCurrDSVNLC = dfCurrDSVOriginal[dfCurrDSVOriginal['source'].notna()].copy()
    dfCurrDSVNLC = dfCurrDSVNLC.rename(columns={"sku": "Product Code", "source": "Identifier","walmart_dsv_price":"current_nlc_price"})#.drop(columns=["date"])

    dfCurrDSV = dfCurrDSVOriginal[dfCurrDSVOriginal['source'].isna()].copy()

    # dfCurrDSV = dfCurrDSV.drop_duplicates(subset=["SKU", "Vendor Code"], keep="last")
    dfCurrDSV = dfCurrDSV.rename(columns={"sku": "SKU"})
    return dfCurrDSVOriginal, dfCurrDSVNLC, dfCurrDSV

In [40]:
def dsv_changes(dfCurrDSVOriginal, dfPrevDSVOriginal):

    dfCurrDSVOriginalCheck = dfCurrDSVOriginal.copy()
    dfCurrDSVOriginalCheck["source"] =  dfCurrDSVOriginalCheck["source"].fillna("National")
    dfCurrDSVOriginalCheck["SKU-Node"] = dfCurrDSVOriginalCheck["sku"] + "-" + dfCurrDSVOriginalCheck["source"]

    dfPrevDSVOriginalCheck = dfPrevDSVOriginal.copy()
    dfPrevDSVOriginalCheck["source"] =  dfPrevDSVOriginalCheck["source"].fillna("National")
    dfPrevDSVOriginalCheck["SKU-Node"] = dfPrevDSVOriginalCheck["sku"] + "-" + dfPrevDSVOriginalCheck["source"]

    dfDSVComp = dfCurrDSVOriginalCheck.merge(dfPrevDSVOriginalCheck, how="left", on=["SKU-Node"], suffixes=("_current", "_previous"))
    dfDSVComp["Price change %"] = (dfDSVComp["walmart_dsv_price_current"] - dfDSVComp["walmart_dsv_price_previous"]) / dfDSVComp["walmart_dsv_price_previous"]

    dfDSVComp["Price change % category"] = np.where(dfDSVComp["Price change %"] >0,"Increase",
                                            np.where(dfDSVComp["Price change %"] <0,"Decrease", np.where(dfDSVComp["Price change %"] == 0, "No change", "New")))

    dfDSVComp["Price change % category"].value_counts()

    #drop sku_previous and source_previous + rename sku_current to sku and source_current to source
    dfDSVComp = dfDSVComp.drop(columns=["sku_previous", "source_previous"]).rename(columns={"sku_current": "sku", "source_current": "source"})

    # dfDSVChanges = dfDSVComp[dfDSVComp["Price change % category"] != "No change"].copy()
    # dfDSVChanges = dfDSVChanges[["SKU-Node", "walmart_dsv_price_current","Price change %", "Price change % category","sku", "source"]]
    return dfDSVComp

In [41]:
dfCurrDSVOriginal, dfCurrDSVNLC, dfCurrDSV = read_dsv()
dfCurrDSVOriginal

Max DSV date: 2026-03-17 00:00:00
Download 87.
Download 100.


,sku,walmart_dsv_price,source
0,IRON0061417565H,40.11,62832635
1,IRON0081417565H,40.11,628326503
2,IRON0061417565H,40.11,628326106
3,IRON0061417565H,40.11,6283261141
4,APLS0031518555V,40.11,628326916
...,...,...,...
3278564,YOKO39618331250E,264.04,628326944
3278565,ZEET0012028555E,144.88,6283261112
3278566,ZEET0012028555E,144.88,628326584
3278567,ZETA0411726570S,104.49,6283261079


In [42]:
if compare_dsvs:
    dfPrevDSVOriginal1, dfPrevDSVNLC1, dfPrevDSV1 = read_dsv('2026-03-05')
    dfPrevDSVOriginal, dfPrevDSVNLC, dfPrevDSV = read_dsv('2026-03-06')

    dfDSVChanges = dsv_changes(dfPrevDSVOriginal, dfPrevDSVOriginal1)


## Rollbacks (to exclude from NLCs)

In [43]:
path = r"H:\Shared drives\07_Finance\07_10_Pricing_department\07_11_01_WalmartB2B\07_10_01_03 Rollbacks\2026-01 RBs\Approved RBs\Approved RBs start date 2026-02-01.xlsx"

dfRollbacksAll = pd.read_excel(path)
dfRollbacks = dfRollbacksAll[dfRollbacksAll["End date"]> pd.to_datetime('today')].copy()
print(f"Number of rollbacks active as of today: {len(dfRollbacks['Product Code'].unique())}")
dfRollbacks

Number of rollbacks active as of today: 577


,Item ID,Brand,Product Code,Final cost used,Time,Start date,End date,Planned end date,Unit cost,Source,Walmart SKU
0,15674650903,Advanta,ADVN0611620565H,65.46,90 days,2026-01-30,2026-04-30,2026-04-30,65.46,Sole Source,ADVN0611620565H~wm
1,15691958892,Advanta,ADVN0611622560H,88.50,90 days,2026-01-30,2026-04-30,2026-04-30,88.50,Sole Source,ADVN0611622560H~wm
2,5317408156,Advanta,ADVN0611721545WXL,62.88,90 days,2026-01-30,2026-04-30,2026-04-30,62.88,Dual Source,ADVN0611721545WXL
3,15701902922,Advanta,ADVN0611721545WXL,76.61,90 days,2026-01-30,2026-04-30,2026-04-30,76.61,Sole Source,ADVN0611721545WXL~wm
4,8616201882,Advanta,ADVN0611723555VXL,73.60,90 days,2026-01-30,2026-04-30,2026-04-30,73.60,Sole Source,ADVN0611723555VXL~wm
...,...,...,...,...,...,...,...,...,...,...,...
777,14888903045,Achilles,ACHI0361624575S,87.86,90 days,2026-01-30,2026-04-30,2026-04-30,87.86,Sole Source,ACHI0361624575S~wm
778,5178959122,Achilles,ACHI0361722560H,61.41,90 days,2026-01-30,2026-04-30,2026-04-30,61.41,Sole Source,ACHI0361722560H
779,5429311514,Achilles,ACHI0361823565H,75.59,90 days,2026-01-30,2026-04-30,2026-04-30,75.59,Sole Source,ACHI0361823565H
780,5439168730,Achilles,ACHI0361826570H,94.91,90 days,2026-01-30,2026-04-30,2026-04-30,94.91,Sole Source,ACHI0361826570H


In [44]:
# days_check = ["2026-03-13","2026-03-15"]
# days_check_date = [pd.to_datetime(day) for day in days_check]
# days_check_date
# dfRollbacksEnded = dfRollbacksAll[dfRollbacksAll["End date"].isin(days_check_date)].copy()
# dfRollbacksEnded

In [45]:
# dfRollbacksEndedSKUs = dfRollbacksEnded[["Product Code"]].drop_duplicates()
# dfRollbacksEndedSKUs = dfRollbacksEndedSKUs[~dfRollbacksEndedSKUs["Product Code"].isin(dfRollbacks["Product Code"])]
# dfRollbacksEndedSKUs

In [46]:
# dfNewDSV[(dfNewDSV["SKU"].isin(dfRollbacksEndedSKUs["Product Code"])) & (dfNewDSV["Source"].notna())]

In [47]:
dfRollbacksSKUPrice = dfRollbacks.groupby(["Product Code"]).agg({"Unit cost": "min"}).reset_index()
dfRollbacksSKUPrice = dfRollbacksSKUPrice.rename(columns={"Product Code": "SKU", "Unit cost": "RB price"})
dfRollbacksSKUPrice

,SKU,RB price
0,ACHI0031726570E,136.30
1,ACHI0361623585S,92.40
2,ACHI0361624575S,87.86
3,ACHI0361722560H,61.41
4,ACHI0361823565H,75.59
...,...,...
572,ZEET0251621560HXL,48.62
573,ZENN0012027555VXL,102.42
574,ZENN0012226535WXL,86.81
575,ZENN0012228545WXL,106.37


In [48]:
# dfRollbacksSKUPriceWithDSV = dfRollbacksSKUPrice.merge(dfCurrDSV[["SKU", "walmart_dsv_price"]], how="left", on="SKU")
# dfRollbacksSKUPriceWithDSV["Delta %"] = round((dfRollbacksSKUPriceWithDSV["Price"] - dfRollbacksSKUPriceWithDSV["walmart_dsv_price"]) / dfRollbacksSKUPriceWithDSV["walmart_dsv_price"],4)
# dfRollbacksSKUPriceWithDSV["Delta %"].value_counts()

## Wh node mapping

In [49]:
idFolderWhStatus = "1Y4drFI84j2eNQM_XTNb9nY9vBCngszZq"
dtype = {"Identifier": str}

dfWHnodeMappingAll = gAPI.get_last_df(service, idFolderWhStatus, sep = ";", dtype=dtype)
dfWHnodeMappingAll = dfWHnodeMappingAll[dfWHnodeMappingAll["Channel"]== "WalmartB2B"]

dfWHnodeMapping = dfWHnodeMappingAll[(dfWHnodeMappingAll["Warehouse Status"] == "ENABLED") &
                                      (dfWHnodeMappingAll["Identifier Status"] == "ENABLED") &
                                       (dfWHnodeMappingAll["Inventory Enabled"] ==1)].copy()
# keepCols = ["Warehouse Code","Identifier", "Type", "Inventory Threshold"]
# dfWHnodeMapping = dfWHnodeMapping[keepCols]
dfWHnodeMapping = dfWHnodeMapping.rename(columns={"Type": "Node type", "Inventory Threshold": "Zero Out Threshold"})

# dfWHnodeMapping["Wh-Identifier"] = dfWHnodeMapping["Warehouse Code"] + "-" + dfWHnodeMapping["Identifier"]

dfWHnodeMapping

Last modified time 2026-03-18T10:00:33.438Z
Download 100.


,Channel,Warehouse Code,Warehouse Status,Identifier Status,Source,Identifier,Node type,Inventory Enabled,Inventory Type,Zero Out Threshold
0,WalmartB2B,5019,ENABLED,ENABLED,NaN,62832654,HUB,1,SUM,7
7,WalmartB2B,502,ENABLED,ENABLED,NaN,62832637,HUB,1,SUM,4
37,WalmartB2B,5022,ENABLED,ENABLED,NaN,628326670,HUB,1,SUM,4
71,WalmartB2B,5024,ENABLED,ENABLED,NaN,628326514,HUB,1,SUM,7
77,WalmartB2B,5024,ENABLED,ENABLED,NaN,6283261190,HUB,1,SUM,7
...,...,...,...,...,...,...,...,...,...,...
23155,WalmartB2B,5980-Vans,ENABLED,ENABLED,NaN,628326944,SPOKE,1,AGGREGATE,4
23157,WalmartB2B,5128-Vans,ENABLED,ENABLED,NaN,628326686,SPOKE,1,AGGREGATE,4
23159,WalmartB2B,5520-Vans,ENABLED,ENABLED,NaN,628326685,SPOKE,1,AGGREGATE,4
23160,WalmartB2B,5520-Vans,ENABLED,ENABLED,NaN,6283261052,SPOKE,1,AGGREGATE,4


In [50]:
# idFolderWhStatus = "1Y4drFI84j2eNQM_XTNb9nY9vBCngszZq"
# dtype = {"Identifier": str}
# # https://drive.google.com/open?id=&usp=drive_fs
# dfWHnodeMappingAllPrev = gAPI.get_df_of_file(service, "1mXEz6Dn4s-GF4mRsFeIgUxsh4fhC5W5j","csv", sep = ";", dtype=dtype)
# dfWHnodeMappingAllPrev = dfWHnodeMappingAllPrev[dfWHnodeMappingAllPrev["Channel"]== "WalmartB2B"]

# dfWHnodeMappingAllPrev["Wh-Identifier"] = dfWHnodeMappingAllPrev["Warehouse Code"] + "-" + dfWHnodeMappingAllPrev["Identifier"]
# dfWHnodeMappingAllPrev = dfWHnodeMappingAllPrev.rename(columns={"Type": "Node type", "Inventory Threshold": "Zero Out Threshold"})


# dfWHnodeMappingPrev = dfWHnodeMappingAllPrev[(dfWHnodeMappingAllPrev["Warehouse Status"] == "ENABLED") &
#                                       (dfWHnodeMappingAllPrev["Identifier Status"] == "ENABLED") &
#                                        (dfWHnodeMappingAllPrev["Inventory Enabled"] ==1)].copy()
# # keepCols = ["Warehouse Code","Identifier", "Type", "Inventory Threshold"]
# # dfWHnodeMappingPrev = dfWHnodeMappingPrev[keepCols]
# dfWHnodeMappingPrev = dfWHnodeMappingPrev.rename(columns={"Type": "Node type", "Inventory Threshold": "Zero Out Threshold"})
# dfWHnodeMappingPrev["Wh-Identifier"] = dfWHnodeMappingPrev["Warehouse Code"] + "-" + dfWHnodeMappingPrev["Identifier"]

# dfWHnodeMappingPrev

In [51]:
# dfVendor = dw.get_vendor_code_table()
# dfVendor = dfVendor.rename(columns={"warehouse_code": "Warehouse Code"})
# dfVendorRef = dfVendor[["Warehouse Code", "Vendor Group", "vendor_code","warehouse_name"]].copy()

# # dfVendor = dfVendor[["Warehouse Code", "vendor_code"]].copy()
# dfVendorRef

In [52]:
# dfWHnodeMappingNew = dfWHnodeMapping[dfWHnodeMapping["Wh-Identifier"].isin(dfWHnodeMappingPrev["Wh-Identifier"]) == False].copy()
# dfWHnodeMappingNew = dfWHnodeMappingNew.merge(dfVendorRef, on="Warehouse Code", how="left")
# dfWHnodeMappingNew.to_excel("New active WH-identifier combinations.xlsx", index=False)
# dfWHnodeMappingWithPrevStatus = dfWHnodeMapping.merge(dfWHnodeMappingAllPrev, how = "left", on="Wh-Identifier", suffixes=("", "_Prev")).merge(dfVendorRef, on="Warehouse Code", how="left")
# dfWHnodeMappingWithPrevStatus["Was fully enabled before?"] = np.where(dfWHnodeMappingWithPrevStatus["Wh-Identifier"].isin(dfWHnodeMappingPrev["Wh-Identifier"]), "Yes", "No")
# dfWHnodeMappingWithPrevStatus["Was fully enabled before?"].value_counts() 
# dfWHnodeMappingWithPrevStatus.to_excel("WH-identifier combinations with previous status.xlsx", index=False)
# dfWHnodeMappingNew["Vendor Group"].value_counts()

In [53]:
# OLD:
# # id = "1uQKncC10D0cVgLOXWQcIxNnogNrIKzKZ" #10-09-2025 file
# # id = "14Ahcn-M7G6fBWJH4PbbIN3tOiOnTCxyT" #10-30-2025 file 
# # id = "15zrneTcAiUJJv7YJO-_gqia8QBGHqPd6" #11-13-2025 file
# # id = "1MtBERWyq3kvcQfEZyiQKmzXAm0U8y0Cs" #11-24-2025 file 
# # id = "1ELvSFQ-pqhb7j70BQJLuar6N68Cn_6ya" #12-04-2025 file 
# # id = "1NOh38WnyG-sFtG_EGX52UCWzL7dB3I-z" #12-11-2025 file
# # id = "1SoKqC_odyUj6t4NwjAjAh-31UQsNG7lv" #12-30-2025 file
# # id = "1LUQS3droPFVV56kmn7HIpth-pNHDIg9I" #1-14-2026 file
# id = "1fxZOdTyygUkLipYQO7ty3xJks0hvnmIn" #1-19-2026 file including SPOKES



# usecols = ["Warehouse", "Identifier", "Inventory Enabled","Inventory Type","Zero Out Threshold","Type"]
# dtype = {"Identifier": str}
# dfWHnodeMappingAll = gAPI.get_df_of_file(service, id, "csv", sep = ";", dtype=dtype, read_cols=usecols)

# dfWHnodeMapping = dfWHnodeMappingAll[dfWHnodeMappingAll["Inventory Enabled"] == True].copy()
# dfWHnodeMapping =dfWHnodeMapping.drop(columns=["Inventory Enabled","Inventory Type"]).copy()

# dfWHnodeMapping["Warehouse"] = dfWHnodeMapping["Warehouse"].str.split(" [", n=1, regex=False).str[0]
# dfWHnodeMapping.rename(columns={"Warehouse": "Warehouse Code", "Type": "Node type"}, inplace=True)
# dfWHnodeMapping

## Shipping costs

In [54]:
path = r"H:\Shared drives\07_Finance\07_10_Pricing_department\07_11_01_WalmartB2B\07_10_01_01 Node level costs\Config files\Costs to add by node.xlsx"
dfCostNode = pd.read_excel(path, dtype = {"Identifier": str}, usecols=["Identifier", "Shipping cost"])
dfCostNode

,Identifier,Shipping cost
0,628326761,10
1,6283261135,7
2,6283261137,7
3,6283261207,7
4,6283261132,7
5,6283261134,7
6,6283261147,7
7,6283261109,7
8,6283261235,7


In [55]:
dfCostNodeCheck = dfCostNode[dfCostNode["Identifier"]!= "628326761"]

## Inventory

In [56]:
def process_inventory_nlc(dfInventoryRelevant, metricGroup = "min", min_units = 4):

    dfInventoryRelevant["Brand code"] = dfInventoryRelevant["Product Code"].str[:4]

    dfInventoryRelevant = dfInventoryRelevant[dfInventoryRelevant["Available"] >= min_units].copy()

    dfInventoryRelevantFiltered = dfInventoryRelevant[dfInventoryRelevant["Product Code"].isin(dfCurrDSV["SKU"])].copy()

    dfInventoryRelevantFilteredWithNodes = dfInventoryRelevantFiltered.merge(dfWHnodeMapping, how="left", on="Warehouse Code")
    dfInvAfterThreshold= dfInventoryRelevantFilteredWithNodes[dfInventoryRelevantFilteredWithNodes["Available"] >= dfInventoryRelevantFilteredWithNodes["Zero Out Threshold"]].copy()
    dfInvAfterThreshold = dfInvAfterThreshold[dfInvAfterThreshold["Node type"].notna()].copy()
    dfInvMinPrices = dfInvAfterThreshold.groupby(['Product Code', 'Identifier',"date"]).agg({'Purchase Price+FET':metricGroup})
    dfInvNLC = dfInvAfterThreshold.merge(dfInvMinPrices, how="inner", on=['Product Code', 'Identifier', 'Purchase Price+FET',"date"])
    dfInvNLC = dfInvNLC.drop_duplicates(subset=["Product Code", "Identifier", "Purchase Price+FET","date"], keep="first")
    dfInvNLC = dfInvNLC[["Product Code", "Identifier", "Warehouse Code", "Available", "Purchase Price+FET","date","Node type"]].copy()
    
    dfInvFinal = (
        dfInvNLC.sort_values('date', ascending=False)
        .groupby(['Product Code', 'Identifier'], as_index=False)
        .first()
    )

    return dfInvFinal

In [57]:
dfInvAll = pd.DataFrame()
for date_i in dates_str_list:
    dfInvDate = pricing.get_inventory(date_i, add_rebates = False, amazon = False, greater3=True)
    dfInvDate["date"] = pd.to_datetime(date_i)
    print(f"Date {date_i} done, rows retrieved: {len(dfInvDate)}")

    dfInvAll = pd.concat([dfInvAll, dfInvDate], ignore_index=True)

dfInvAll = dfInvAll[~dfInvAll["Product Code"].isin(dfRollbacks["Product Code"])]

dfInvAll

Download 100.
Last modified time 2026-03-11T10:01:18.275Z
Download 93.
Download 100.
Date 2026-03-11 done, rows retrieved: 2245372
Download 100.
Last modified time 2026-03-12T10:01:48.663Z
Download 92.
Download 100.
Date 2026-03-12 done, rows retrieved: 2246294
Download 100.
Last modified time 2026-03-13T20:06:51.587Z
Download 90.
Download 100.
Date 2026-03-13 done, rows retrieved: 2295802
Download 100.
Last modified time 2026-03-14T11:01:28.638Z
Download 90.
Download 100.
Date 2026-03-14 done, rows retrieved: 2296413
Download 100.
Last modified time 2026-03-15T11:01:18.282Z
Download 90.
Download 100.
Date 2026-03-15 done, rows retrieved: 2294127
Download 100.
Last modified time 2026-03-16T11:01:16.646Z
Download 90.
Download 100.
Date 2026-03-16 done, rows retrieved: 2291317
Download 100.
Last modified time 2026-03-17T11:01:42.267Z
Download 94.
Download 100.
Date 2026-03-17 done, rows retrieved: 2187651
Download 100.
Last modified time 2026-03-18T10:01:24.060Z
Download 95.
Download 100

,Product Code,Warehouse Code,Purchase Price,Available,FET,Purchase Price+FET,name,State_Code,Postal code,Zone,closed,walmartb2b_enabled,walmartb2b_threshold,Can show inv?,date
0,ACCE0131419550H,5266,45.90,4,0.0,45.90,Wheel Mart Sacramento,US-CA,95836,2,No,Yes,4.0,True,2026-03-11
1,ACCE0131820535YXL,5614,56.95,87,0.0,56.95,Quality Tire Distributors,US-FL,33610,4,No,Yes,4.0,True,2026-03-11
2,ACCE0221316580T,5232,33.35,16,0.0,33.35,Hesselbein Tire - San Antonio,US-TX,78219,3,No,Yes,4.0,True,2026-03-11
3,ACCE0221316580T,5233,33.35,16,0.0,33.35,Hesselbein Tire Southwest - Mission,US-TX,78572,3,No,Yes,4.0,True,2026-03-11
4,ACCE0221316580T,5234,33.35,10,0.0,33.35,Hesselbein Tire - Houston,US-TX,77040,3,No,Yes,4.0,True,2026-03-11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18050419,ZETA0411826565H,583,99.00,47,0.0,99.00,"Rudolph Tire - Murray, KY",US-KY,42071,8,No,Yes,4.0,True,2026-03-18
18050420,ZETA0411826565H,6081,103.64,92,0.0,103.64,Centerline Auto,US-CA,95825,2,No,Yes,7.0,True,2026-03-18
18050421,ZETA0411827565H,568,90.86,50,0.0,90.86,Freedom Tire Distributing,US-CA,92882,2,No,Yes,7.0,True,2026-03-18
18050422,ZETA0411827565H,8186,99.00,32,0.0,99.00,"Tires Depot Inc Memphis, TN",US-TN,38134,8,No,Yes,4.0,True,2026-03-18


## Sales

### Get data

In [58]:
days_sales = 90
today_minus_x = (pd.to_datetime(date_str) - relativedelta(days=days_sales)).strftime('%Y-%m-%d')

print("start date", today_minus_x)

query = f"""SELECT order_id, sku, externalwarehouseid, order_date, supplier_id,
       quantity, total_inv_amount, profit, external_price, order_type
FROM warehouse.vw_virtual_node_tracker
WHERE shop = 'WalmartB2B'
  AND order_type IN ('Sale')
  AND orderstatus NOT IN ('WILL_CALL_NOT_PAID', 'CANCELLED_BEFORE_SHIPPING', 'ERROR')
  AND order_date >= '{today_minus_x}'
  AND order_date < CURRENT_DATE;
"""

dfSalesSKUNode = dw.runQuery(query, newCredentials=False)
dfSalesSKUNode["SKU-Node"] = dfSalesSKUNode["sku"] + "-" + dfSalesSKUNode["externalwarehouseid"].astype(str)
dfSalesSKUNode["order_date"] = pd.to_datetime(dfSalesSKUNode["order_date"])


dfSales = dfSalesSKUNode.groupby(["order_date", "sku"]).agg({"quantity":"sum", "total_inv_amount":"sum", "profit":"sum"}).reset_index()
dfSales = dfSales.rename(columns={ "total_inv_amount": "revenue", "sku": "Product Code"})
dfSKURevenue = dfSales.copy().groupby("Product Code").agg({"revenue": "sum"}).reset_index()
dfSKURevenue = dfSKURevenue.rename(columns={"revenue": "sku_revenue"})


dfSKUNodeRevenue = dfSalesSKUNode.groupby(["SKU-Node"]).agg({"total_inv_amount":"sum"}).reset_index()
total_revenue = dfSKUNodeRevenue["total_inv_amount"].sum()
print(f"Total revenue from sales with node-level costs in the last 90 days: {total_revenue}")
dfSKUNodeRevenue = dfSKUNodeRevenue.rename(columns={"total_inv_amount": "sku_node_revenue"})
dfSKUNodeRevenue

start date 2025-12-18
Total revenue from sales with node-level costs in the last 90 days: 89693190.94000003


,SKU-Node,sku_node_revenue
0,ACCE0131318560V-6283261264,141.52
1,ACCE0131418555V-6283261117,454.05
2,ACCE0131418555V-628326864,100.90
3,ACCE0131419550H-6283261117,196.00
4,ACCE0131419550H-628326433,222.24
...,...,...
116455,ZETA0411823555V-628326297,152.80
116456,ZETA0411823555V-628326704,452.04
116457,ZETA0411823560VXL-628326985,178.94
116458,ZETA0411823565HXL-6283261094,93.26


In [59]:
# How many Product Code are in the dfSales (use dfSales['Product Code'].nunique()) over the last 14 days, 30 days, 90 days and 150 days?
days = [days_sales]
for day in days:
    date_threshold = pd.to_datetime(date_str) - timedelta(days=day)
    count_product_codes = dfSales[dfSales['order_date'] >= date_threshold]['Product Code'].nunique()
    print(f"Number of unique Product Codes in the last {day} days: {count_product_codes}")

#Use the above to calculate how many skus represent 20% of sales, 50% of sales and 80% of sales for the same days in the list
for day in days:
    date_threshold = pd.to_datetime(date_str) - timedelta(days=day)
    dfSalesFiltered = dfSales[dfSales['order_date'] >= date_threshold].copy()
    dfSalesFiltered = dfSalesFiltered.groupby("Product Code").agg({"revenue": "sum"}).reset_index()
    dfSalesFiltered = dfSalesFiltered.sort_values("revenue", ascending=False).reset_index(drop=True)
    dfSalesFiltered["cumulative_revenue"] = dfSalesFiltered["revenue"].cumsum()
    total_revenue_filt = dfSalesFiltered["revenue"].sum()
    dfSalesFiltered["cumulative_revenue_pct"] = dfSalesFiltered["cumulative_revenue"] / total_revenue_filt
    skus_20_pct = (dfSalesFiltered[dfSalesFiltered["cumulative_revenue_pct"] <= 0.20].shape[0])
    skus_50_pct = (dfSalesFiltered[dfSalesFiltered["cumulative_revenue_pct"] <= 0.50].shape[0])
    skus_80_pct = (dfSalesFiltered[dfSalesFiltered["cumulative_revenue_pct"] <= 0.80].shape[0])
    
    print(f"In the last {day} days, {skus_20_pct} SKUs represent 20% of sales, {skus_50_pct} SKUs represent 50% of sales, and {skus_80_pct} SKUs represent 80% of sales.")
    #Also create a column "Sales category" that is top 20%, top 50% top 80% and the rest, and count how many skus are in each category.
    dfSalesFiltered["sku_sales_category"] = pd.cut(dfSalesFiltered["cumulative_revenue_pct"], bins=[0, 0.05,0.1,0.2,0.3,0.4, 0.50,0.6,0.7,0.8,0.9,0.95,1], labels=["0-5%","5-10%","10-20%","20-30%","30-40%","40-50%","50-60%","60-70%","70-80%","80-90%","90-95%","95-100%"])
    sales_category_counts = dfSalesFiltered["sku_sales_category"].value_counts()
    print(f"Sales category counts in the last {day} days:\n{sales_category_counts}\n")

dfSKURevenue = dfSKURevenue.merge(dfSalesFiltered[["Product Code", "sku_sales_category"]], on="Product Code", how="left")
dfSKURevenue

Number of unique Product Codes in the last 90 days: 22680
In the last 90 days, 114 SKUs represent 20% of sales, 870 SKUs represent 50% of sales, and 4480 SKUs represent 80% of sales.
Sales category counts in the last 90 days:
sku_sales_category
95-100%    10711
80-90%      3779
90-95%      3710
70-80%      1899
60-70%      1073
50-60%       638
40-50%       387
30-40%       231
20-30%       138
10-20%        79
5-10%         22
0-5%          13
Name: count, dtype: int64



,Product Code,sku_revenue,sku_sales_category
0,ACCE0131318560V,141.52,95-100%
1,ACCE0131418555V,554.95,95-100%
2,ACCE0131419550H,418.24,95-100%
3,ACCE0131820535YXL,137.98,95-100%
4,ACCE0221316580T,309.02,95-100%
...,...,...,...
22675,ZETA0411723560SXL,126.44,95-100%
22676,ZETA0411823555V,834.04,95-100%
22677,ZETA0411823560VXL,178.94,95-100%
22678,ZETA0411823565HXL,93.26,95-100%


### Stats less sales

In [60]:
# df = dfSales.copy()
# df['order_date'] = pd.to_datetime(df['order_date'])

# # --- anchors (same logic as before) ---
# as_of = df['order_date'].max().normalize()

# l2w_start = as_of - pd.Timedelta(days=13)   # 14 days inclusive
# p90_end   = l2w_start - pd.Timedelta(days=1)
# p90_start = p90_end - pd.Timedelta(days=89) # 90 days inclusive

# # --- slice periods ---
# l2w = df[df['order_date'].between(l2w_start, as_of)]
# p90 = df[df['order_date'].between(p90_start, p90_end)]

# # --- global totals ---
# l2w_qty = l2w['quantity'].sum()
# p90_qty = p90['quantity'].sum()

# l2w_rev = l2w['revenue'].sum()
# p90_rev = p90['revenue'].sum()

# l2w_prof = l2w['profit'].sum()
# p90_prof = p90['profit'].sum()

# # --- global metrics (fixed days) ---
# global_avg_daily_qty_L2W = l2w_qty / 14
# global_avg_daily_qty_P90 = p90_qty / 90

# global_margin_L2W = l2w_prof / l2w_rev
# global_margin_P90 = p90_prof / p90_rev

# # --- comparisons ---
# global_qty_pct_change_L2W_vs_P90 = global_avg_daily_qty_L2W / global_avg_daily_qty_P90 - 1
# global_margin_delta_L2W_vs_P90 = global_margin_L2W - global_margin_P90

# # --- nice output dataframe ---
# overall = pd.DataFrame([{
#     'as_of': as_of.date(),
#     'L2W_start': l2w_start.date(),
#     'P90_start': p90_start.date(),
#     'P90_end': p90_end.date(),
#     'avg_daily_qty_L2W': global_avg_daily_qty_L2W,
#     'avg_daily_qty_P90': global_avg_daily_qty_P90,
#     'qty_pct_change_L2W_vs_P90': global_qty_pct_change_L2W_vs_P90,
#     'margin_L2W': global_margin_L2W,
#     'margin_P90': global_margin_P90,
#     'margin_delta_L2W_vs_P90': global_margin_delta_L2W_vs_P90
# }])

# # round
# num_cols = overall.select_dtypes(include='number').columns
# overall[num_cols] = overall[num_cols].round(4)

# overall


In [61]:
# rb = r"H:\Shared drives\07_Finance\07_10_Pricing_department\07_11_01_WalmartB2B\07_10_01_03 Rollbacks\All RBs tracker.xlsx"
# dfRb = pd.read_excel(rb, usecols = ["Product Code", "Start date", "End date"]).rename(columns = {"Start date":"Rb start date", "End date":"Rb end date"})
# # dfRb["Ended RB"] = np.where(dfRb["Rb end date"] < pd.to_datetime('today'), "Yes", "No")
# # dfRb["Active RB"] = np.where(dfRb["Rb end date"] >= pd.to_datetime('today'), "Yes", "No")

# # dfRb["RB Status"] = np.where(dfRb["Ended RB"] == "Yes", "Ended", np.where(dfRb["Active RB"] == "Yes", "Active", "Unknown"))

# # dfRb = dfRb.drop(columns=["Rb start date", "Rb end date", "Ended RB", "Active RB"])

# dfRb

In [62]:
# threshold = -0.0
# totalTreshold = threshold + global_qty_pct_change_L2W_vs_P90

# marginTreshold = 0.08

In [63]:
# df = dfSales.copy()
# df['order_date'] = pd.to_datetime(df['order_date'])

# # -----------------------
# # 1) L2W vs P90 (fixed days)
# # -----------------------
# as_of = df['order_date'].max().normalize()

# l2w_start = as_of - pd.Timedelta(days=13)   # 14 days inclusive
# p90_end   = l2w_start - pd.Timedelta(days=1)
# p90_start = p90_end - pd.Timedelta(days=89) # 90 days inclusive

# df['period'] = np.select(
#     [
#         df['order_date'].between(l2w_start, as_of),
#         df['order_date'].between(p90_start, p90_end)
#     ],
#     ['L2W', 'P90'],
#     default=None
# )

# agg = (
#     df[df['period'].notna()]
#       .groupby(['Product Code', 'period'], as_index=False)
#       .agg(qty=('quantity', 'sum'),
#            rev=('revenue', 'sum'),
#            prof=('profit', 'sum'))
# )

# agg['days'] = agg['period'].map({'L2W': 14, 'P90': 90}).astype(int)
# agg['avg_daily_qty'] = agg['qty'] / agg['days']
# agg['margin'] = agg['prof'] / agg['rev']
# agg = agg.drop(columns=['qty', 'rev', 'prof', 'days'])

# wide = agg.pivot(index='Product Code', columns='period', values=['avg_daily_qty', 'margin'])
# wide.columns = [f'{m}_{p}' for (m, p) in wide.columns]
# wide = wide.reset_index()

# wide['qty_pct_change_L2W_vs_P90'] = wide['avg_daily_qty_L2W'] / wide['avg_daily_qty_P90'] - 1
# wide["qty_abs_change_L2W_vs_P90"] = wide['avg_daily_qty_L2W'] - wide['avg_daily_qty_P90']

# wide['margin_delta_L2W_vs_P90'] = wide['margin_L2W'] - wide['margin_P90']

# wide = wide.merge(dfRb, how="left", left_on="Product Code", right_on="Product Code")

# wide["Sales below threshold"] = np.where(wide["qty_pct_change_L2W_vs_P90"] < totalTreshold, "Yes", "No")
# wide["margin_L2W > target"] = np.where(wide["margin_L2W"] >= marginTreshold, "Yes", "No")
# wide["Rb active in analysis period"] = np.where((wide["Rb end date"] >= p90_start), "Yes", "No")
# wide["Target less sales"] = np.where((wide["Sales below threshold"] == "Yes") & 
#                                      (wide["margin_L2W > target"] == "Yes") &
#                                      (wide["Rb active in analysis period"]!="Yes"), "Yes", "No")

# # -----------------------
# # 2) Monthly reference for ALL months in data
# #    (avg_daily_qty = monthly qty / # calendar days in that month)
# # -----------------------
# df['month'] = df['order_date'].dt.to_period('M')

# monthly = (
#     df.groupby(['Product Code', 'month'], as_index=False)
#       .agg(qty=('quantity', 'sum'),
#            rev=('revenue', 'sum'),
#            prof=('profit', 'sum'))
# )

# monthly['days_in_month'] = monthly['month'].dt.days_in_month
# monthly['avg_daily_qty'] = monthly['qty'] / monthly['days_in_month']
# monthly['margin'] = monthly['prof'] / monthly['rev']

# monthly_wide = monthly.pivot(index='Product Code', columns='month', values=['avg_daily_qty', 'margin'])
# monthly_wide.columns = [f'{m}_{str(mon)}' for (m, mon) in monthly_wide.columns]  # e.g. avg_daily_qty_2025-10
# monthly_wide = monthly_wide.reset_index()

# # -----------------------
# # 3) Merge monthly reference to the right
# # -----------------------
# final = wide.merge(monthly_wide, on='Product Code', how='left')

# # -----------------------
# # 4) Round to 2 decimals
# # -----------------------
# num_cols = final.select_dtypes(include='number').columns
# final[num_cols] = final[num_cols].round(4)


# finalTarget = final[final["Target less sales"] == "Yes"].copy()

# final


In [64]:
# targetVolume = finalTarget["avg_daily_qty_L2W"].sum()
# totalVolume = final["avg_daily_qty_L2W"].sum()
# percVolume = targetVolume / totalVolume
# print(percVolume)

In [65]:
# # finalTarget categories for margin_L2W for 11% to 13%, 13 to 15, 15 to 17 and 17+
# finalTarget["margin_L2W cat"] = pd.cut(finalTarget["margin_L2W"], bins=[-np.inf,.08, 0.11, 0.13, 0.15, 0.17, np.inf], labels=["<8%", "8% to <11%", "11% to <13%", "13% to <15%", "15% to <17%", ">=17%"])
# finalTarget["margin_L2W cat"].value_counts()

In [66]:
# final["qty_abs_change_L2W_vs_P90"].sum()

In [67]:
# final[final["Target less sales"] == "Yes"]["qty_abs_change_L2W_vs_P90"].sum()

In [68]:
# final.to_excel("Walmart B2B SKU-level sales and margin analysis.xlsx", index=False)

<!-- ### Assign groups -->

In [69]:
# import random
# from typing import Dict, List, Optional, Sequence, Tuple

# import pandas as pd


# def split_skus_into_groups(
#     df: pd.DataFrame,
#     n_groups: int,
#     code_col: str = "Product Code",
#     qty_col: str = "Total quantity",
#     seed: int = 42,
#     max_iters: int = 200_000,
#     patience: int = 40_000,
#     allow_equal_sum_only_if_sizes_met: bool = True,
# ) -> pd.DataFrame:
#     """
#     Partition rows into exactly `n_groups` groups such that:
#       - group sizes are as equal as possible (fixed targets)
#       - group total quantities are as close as possible

#     Returns: df copy with new column 'group' in [1..n_groups].

#     Algorithm (fast + good quality):
#       1) Size-constrained greedy assignment (largest qty first -> lowest current sum among non-full groups)
#       2) Local search with pairwise swaps across groups to reduce sum imbalance

#     Notes:
#       - Size targets are fixed: e.g. n=10, k=3 -> [4,3,3]; n=10, k=4 -> [3,3,2,2]
#       - Handles non-numeric qty by coercing to 0.
#     """
#     if n_groups < 2:
#         raise ValueError("n_groups must be >= 2")
#     if code_col not in df.columns or qty_col not in df.columns:
#         raise ValueError(f"df must contain columns: '{code_col}' and '{qty_col}'")

#     work = df[[code_col, qty_col]].copy()
#     work[qty_col] = pd.to_numeric(work[qty_col], errors="coerce").fillna(0)

#     n = len(work)
#     out = df.copy()
#     if n == 0:
#         out["group"] = pd.Series(dtype="int64")
#         return out

#     # --- Target sizes: as equal as possible (largest groups first) ---
#     base = n // n_groups
#     r = n % n_groups
#     target_sizes = [base + (1 if i < r else 0) for i in range(n_groups)]

#     rng = random.Random(seed)

#     # --- 1) Greedy init (largest qty first, tie-broken by shuffle) ---
#     idxs = list(work.index)
#     rng.shuffle(idxs)
#     work = work.loc[idxs].sort_values(qty_col, ascending=False)

#     groups: List[List[int]] = [[] for _ in range(n_groups)]
#     sums: List[float] = [0.0] * n_groups

#     for row_idx, qty in zip(work.index, work[qty_col].tolist()):
#         eligible = [g for g in range(n_groups) if len(groups[g]) < target_sizes[g]]
#         # Choose lowest sum; tie-break on smaller size; then random
#         eligible.sort(key=lambda g: (sums[g], len(groups[g]), rng.random()))
#         g = eligible[0]
#         groups[g].append(row_idx)
#         sums[g] += float(qty)

#     qty_map: Dict[int, float] = work[qty_col].to_dict()

#     def spread(s: Sequence[float]) -> float:
#         return float(max(s) - min(s))

#     def sse(s: Sequence[float]) -> float:
#         # sum of squared error vs mean (helps overall balance, not just min/max)
#         m = sum(s) / len(s)
#         return float(sum((x - m) ** 2 for x in s))

#     # Objective: prioritize lowering max-min spread; use SSE as tie-breaker
#     def objective(s: Sequence[float]) -> Tuple[float, float]:
#         return (spread(s), sse(s))

#     best_groups = [g[:] for g in groups]
#     best_sums = sums[:]
#     best_obj = objective(best_sums)

#     # --- 2) Local search via swaps (keeps sizes exactly fixed) ---
#     group_of = {idx: g for g in range(n_groups) for idx in groups[g]}
#     group_lists = [g[:] for g in groups]
#     current_sums = sums[:]
#     current_obj = objective(current_sums)

#     # Precompute which groups are currently highest / lowest sum often helps,
#     # but we keep it simple + fast with random sampling.
#     no_improve = 0

#     def try_swap(a: int, b: int) -> bool:
#         nonlocal current_obj, best_obj, best_groups, best_sums

#         ga, gb = group_of[a], group_of[b]
#         if ga == gb:
#             return False

#         qa = float(qty_map[a])
#         qb = float(qty_map[b])

#         new_sums = current_sums[:]
#         new_sums[ga] = new_sums[ga] - qa + qb
#         new_sums[gb] = new_sums[gb] - qb + qa

#         new_obj = objective(new_sums)

#         # Accept if strictly better objective
#         if new_obj < current_obj:
#             # apply swap
#             group_lists[ga].remove(a)
#             group_lists[gb].remove(b)
#             group_lists[ga].append(b)
#             group_lists[gb].append(a)
#             group_of[a], group_of[b] = gb, ga
#             current_sums[:] = new_sums
#             current_obj = new_obj

#             if new_obj < best_obj:
#                 best_obj = new_obj
#                 best_sums = new_sums[:]
#                 best_groups = [g[:] for g in group_lists]
#             return True

#         return False

#     for _ in range(max_iters):
#         # Pick two different groups; bias toward swapping from high-sum to low-sum groups.
#         # This usually improves convergence vs totally random.
#         order = sorted(range(n_groups), key=lambda g: current_sums[g])
#         low = rng.choice(order[: max(1, n_groups // 3)])
#         high = rng.choice(order[-max(1, n_groups // 3) :])

#         if low == high:
#             continue
#         if not group_lists[low] or not group_lists[high]:
#             continue

#         a = rng.choice(group_lists[high])  # item from high-sum group
#         b = rng.choice(group_lists[low])   # item from low-sum group

#         improved = try_swap(a, b)

#         if improved:
#             no_improve = 0
#         else:
#             no_improve += 1
#             if no_improve >= patience:
#                 break

#     # --- Build assignment column ---
#     assignment = {idx: (g + 1) for g in range(n_groups) for idx in best_groups[g]}
#     out["group"] = out.index.map(assignment).astype("int64")

#     return out


# # --- Example ---
# # df_with_groups = split_skus_into_groups(dfSalesSKUJanuary, n_groups=5,
# #                                        code_col="Product Code", qty_col="January quantity")


In [70]:
# # #use functino split_skus_into_3_groups to finalTarget dataframe based on "Product Code" and "avg_daily_qty_L2W"
# finalTargetWithGroups = split_skus_into_groups(df = finalTarget, n_groups=3, code_col="Product Code", qty_col="avg_daily_qty_L2W", seed=10)
# finalTargetWithGroupsMerge = finalTargetWithGroups[["Product Code", "group","qty_pct_change_L2W_vs_P90"]].copy()
# display(finalTargetWithGroups)
# finalTargetWithGroupsCheck = finalTargetWithGroups.groupby("group").agg({"Product Code": "count", "avg_daily_qty_L2W": "sum"})
# finalTargetWithGroupsCheck

### Prev

In [71]:
# dfSales["Brand code"] = dfSales["Product Code"].str[:4]

# target_brands = ["BLHK", "IRON", "LION", "MICH", "ARMS", "WATF", "FIRE"]


# dfSalesBrands = dfSales[dfSales["Brand code"].isin(target_brands)].copy()
# dfSalesBrands

In [72]:
# dfSalesSKU = dfSalesBrands.groupby(["Product Code","Brand code"]).agg({"quantity": "sum"}).reset_index()
# dfSalesSKU = dfSalesSKU.rename(columns={"quantity": "Total quantity"})
# dfSalesSKU

In [73]:
# dfSalesSKUJanuary = dfSalesBrands[dfSalesBrands["order_date"].dt.month == 1].groupby(["Product Code","Brand code"]).agg({"quantity": "sum"}).reset_index()
# dfSalesSKUJanuary = dfSalesSKUJanuary.rename(columns={"quantity": "Total quantity"})
# dfSalesSKUJanuary

In [74]:
# dfSalesSKUNotJan = dfSalesSKU[dfSalesSKU["Product Code"].isin(dfSalesSKUJanuary["Product Code"]) == False].copy()
# dfSalesSKUNotJan

In [75]:
# def selection_by_brand(dfSales: pd.DataFrame, target_brands: List[str]) -> pd.DataFrame:

#     selection = pd.DataFrame()
#     for brand in target_brands:
#         # print(f"Processing brand: {brand}")
#         dfBrand = dfSales[dfSales["Brand code"] == brand].copy()
#         dfBrandWithGroups = split_skus_into_3_groups(dfBrand, seed=42)
#         # print(dfBrandWithGroups[["Product Code", "Total quantity", "group"]].to_string(index=False))
#         # print("\n")

#         selection = pd.concat([selection, dfBrandWithGroups], ignore_index=True)

#     #BY GROUP DISPLAY # SKUS AND TOTAL QTY
#     display(
#         selection.groupby("group", as_index=False)
#         .agg(
#             sku_count=("Product Code", "count"),
#             total_qty=("Total quantity", "sum"),
#         )
#         .sort_values("group")
#     )

#     return selection

In [76]:
# selectionJan = selection_by_brand(dfSalesSKUJanuary, target_brands)
# selectionJan

In [77]:
# selectionNotJan = selection_by_brand(dfSalesSKUNotJan, target_brands)
# selectionNotJan

In [78]:
# finalGroups = pd.concat([selectionJan, selectionNotJan], ignore_index=True).drop(columns=["Total quantity","Brand code"])
# finalGroups

In [79]:
# dfSalesSKUWithGroups = dfSalesSKU.merge(finalGroups, how="left", on="Product Code")
# dfSalesSKUWithGroups["group"].value_counts()

# # dfSalesSKUWithGroups sum sales by group
# dfSalesByGroup = dfSalesSKUWithGroups.groupby("group").agg({"Total quantity": "sum","Product Code": "count"}).reset_index()
# dfSalesByGroup

## Current tests tracker

In [150]:
# id = "12jQyVDfPnR9qUxfbwmFPWjHxTMZNk59I"
# dfCurrentTestsAll = gAPI.get_df_of_file(service, id, "csv", dtype = {"Identifier": str})

pathTracker = r"H:\Shared drives\07_Finance\07_10_Pricing_department\07_11_01_WalmartB2B\07_10_01_01 Node level costs\Final node level costs tracker.csv"

dfCurrentTestsAll = pd.read_csv(pathTracker, dtype = {"Identifier": str})

dfCurrentTestsAll["SKU-Node"] = dfCurrentTestsAll["Product Code"] + "-" + dfCurrentTestsAll["Identifier"]


dfCurrentTest = dfCurrentTestsAll.copy()
dfCurrentTests = dfCurrentTest[["SKU-Node", "Final target","Final price category","Start date", "Sub-group"]].copy()
dfCurrentTestsAll["Final target"].value_counts()

C:\Users\franc\AppData\Local\Temp\ipykernel_26900\4003682945.py:6: DtypeWarning: Columns (3,6,7,11,12,14,15,16,17) have mixed types. Specify dtype option on import or set low_memory=False.
  dfCurrentTestsAll = pd.read_csv(pathTracker, dtype = {"Identifier": str})


Final target
Added                       1258584
Normal strategy              498692
Margin test                  430874
Updated                      350334
No sales test                291844
Updates test                 148696
Decreased - margin > 20%     133485
Wm margin split test          95983
Shipping cost added           31921
Less sales                    15861
Less sales back               15556
Name: count, dtype: int64

In [151]:
if compare_dsvs:
    dfDSVChanges["Product Code"] = dfDSVChanges["SKU-Node"].str.split("-", n=1, regex=False).str[0]
    dfDSVChanges["Identifier"] = dfDSVChanges["SKU-Node"].str.split("-", n=1, regex=False).str[1]
    dfDSVChanges = dfDSVChanges.merge(dfCurrentTests[["Final target", "SKU-Node"]], on=["SKU-Node"], how="left")

    dfDSVChanges["Final target"] = np.where(dfDSVChanges["Price change % category"] == "Increase", "Updated",
                                                np.where((dfDSVChanges["Price change % category"] == "Decrease") & (dfDSVChanges["Final target"]!= "Wm margin split test"), "Decreased - margin > 20%", 
                                                        dfDSVChanges["Final target"]))

    dfDSVChanges["Final price category"] = np.where(dfDSVChanges["Price change % category"] == "Decrease", "20%","11%")
    dfDSVChanges = dfDSVChanges.rename(columns={"Price change %": "Final delta vs current %",
                                                    "walmart_dsv_price_current":"Final price",
                                                    "Price change % category": "Final price change category final"})
    dfDSVChanges["Node type"] = "HUB"
    dfDSVChanges["Min units"] = 8
    dfDSVChanges["Start date"] = "2026-03-13"
    display(dfDSVChanges["Final target"].value_counts())                        

In [152]:
# dfCurrentTestsAll["Final price category"] = (
#     dfCurrentTestsAll["Final price category"]
#     .astype(str)
#     .str.strip()
#     .str.replace(r"%$", "", regex=True)   # remove existing trailing %
#     .str.extract(r"(\d+)", expand=False)  # keep just the number
#     .add("%")
# )

In [153]:
dfCurrentTestsAll[dfCurrentTestsAll["Final target"] == "Margin test"]["Sub-group"].value_counts()

Sub-group
11%    164587
13%    128393
14%     65160
15%     51168
12%       156
Name: count, dtype: int64

In [154]:
# dfCurrentTestsAll["Sub-group"] = dfCurrentTestsAll["Sub-group"].fillna(dfCurrentTestsAll["Final price category"])

# Initial output

## Function

In [85]:
def calculate_new_node_level_cost(dfInvAll, dfAvgWmtPrice, dfMAP, minWmMargin=0, distanceMAP=0.05, min_units = 4):

    dfInvFinal = process_inventory_nlc(dfInvAll, metricGroup="min", min_units = min_units)

    dfOutput = dfInvFinal.merge(dfAvgWmtPrice, how="left", on="Product Code").merge(dfMAP, how="left", on="Product Code").merge(dfCostNode, how="left", on=["Identifier"])
    dfOutput["Shipping cost"] = dfOutput["Shipping cost"].fillna(0)
    dfOutput["Cost+Shipping"] = dfOutput["Purchase Price+FET"] + dfOutput["Shipping cost"]
    dfOutput["Is MAP now?"] = np.where(dfOutput["MAP"].isna(), "No", "Yes")

    margins = [0.06, 0.08,0.11,0.12,0.13,0.14,0.15, 0.2]
    minWmMargin = 0 #-.05
    distanceMAP = 0.05

    for margin in margins:
        margin_suffix = f" - {int(margin*100)}% margin"
        dfOutput["Node level cost" + margin_suffix] = round(dfOutput["Purchase Price+FET"] /(1-margin),2) 
        dfOutput["Walmart Margin" + margin_suffix] = round((dfOutput["offer_price"] - dfOutput["Node level cost" + margin_suffix]) / dfOutput["offer_price"],4)
        dfOutput["Node cost < MAP - min margin%" + margin_suffix] = np.where(dfOutput["Node level cost" + margin_suffix] < dfOutput["MAP"] *(1-distanceMAP), "Yes", "No")
        dfOutput["Node cost - MAP" + margin_suffix] = dfOutput["Node level cost" + margin_suffix] - dfOutput["MAP"]
        dfOutput["Target for node level cost?" + margin_suffix] = np.where(dfOutput["Is MAP now?"] == "Yes",
                                                                        np.where(dfOutput["Node cost < MAP - min margin%" + margin_suffix]== "No", "No", "Yes"), #This happens if it is MAP now
                                                                                np.where(dfOutput["Walmart Margin" + margin_suffix]> minWmMargin, "Yes", "No")) #This happens if it is not MAP now
             
        dfOutput["Node level cost" + margin_suffix] = dfOutput["Node level cost" + margin_suffix] + dfOutput["Shipping cost"]

    dfOutput = dfOutput.merge(dfCurrDSVNLC, how="left", on=["Product Code", "Identifier"])


    dfOutput["Final node level cost category"] = np.where(dfOutput["Target for node level cost? - 11% margin"] == "Yes", "11%",
                                                np.where(dfOutput["Target for node level cost? - 8% margin"] == "Yes", "8%",
                                                        np.where(dfOutput["Target for node level cost? - 6% margin"] == "Yes", "6%", "N/A")))

    dfOutput["Final node level cost"] = np.where(dfOutput["Target for node level cost? - 11% margin"] == "Yes", dfOutput["Node level cost - 11% margin"],
                                                np.where(dfOutput["Target for node level cost? - 8% margin"] == "Yes", dfOutput["Node level cost - 8% margin"],
                                                        np.where(dfOutput["Target for node level cost? - 6% margin"] == "Yes", dfOutput["Node level cost - 6% margin"], np.nan)))


    
    dfOutput["Final walmart margin"] = np.where(dfOutput["Target for node level cost? - 11% margin"] == "Yes", dfOutput["Walmart Margin - 11% margin"],
                                                np.where(dfOutput["Target for node level cost? - 8% margin"] == "Yes", dfOutput["Walmart Margin - 8% margin"],
                                                        np.where(dfOutput["Target for node level cost? - 6% margin"] == "Yes", dfOutput["Walmart Margin - 6% margin"], np.nan)))


    dfOutput["Final price change %"] = round((dfOutput["Final node level cost"] - dfOutput["current_nlc_price"])/ dfOutput["current_nlc_price"], 3)
    dfOutput["Final price change category"] = np.where(dfOutput["Final price change %"] < 0, "Decrease",
                                                     np.where(dfOutput["Final price change %"] > 0, "Increase", "No change"))

    dfOutput["SKU-Node"] = dfOutput["Product Code"] + "-" + dfOutput["Identifier"]

    dfOutput["Brand code"] = dfOutput["Product Code"].str[:4]

    dfOutput["New NLC"] = np.where(dfOutput["current_nlc_price"].isna(), "Yes", "No")

    dfOutput["Min units"] = min_units

    dfOutput["current_nlc_margin"] = round((dfOutput["current_nlc_price"] - dfOutput["Cost+Shipping"])/dfOutput["current_nlc_price"],4)  

    #dfOutput["current_nlc_margin category" make categories: 0 to 6%, 6% to 8%, 8% to 10.8%,10.8% to 11.2%, 11.2% to 13%, 13 to 15, 15 to 17 and 17+%
    dfOutput["current_nlc_margin category"] = pd.cut(dfOutput["current_nlc_margin"], bins=[-np.inf, 0, 0.06, 0.08, 0.108, 
                                                                                            0.112, 0.13, 0.15, 0.17, 0.198, np.inf], 
                                                                                            labels=["<0%", "0% to <6%", "6% to <8%", "8% to <10.8%", "10.8% to <11.2%", 
                                                                                                     "11.2% to <13%", "13% to <15%", "15% to <17%", "17% to <20%", ">=20%"])


    dfOutput["Current walmart margin at National"] = round((dfOutput["offer_price"] - dfOutput["unit_cost"])/dfOutput["offer_price"],4)
    dfOutput["Current walmart margin at NLC"] = round((dfOutput["offer_price"] - dfOutput["current_nlc_price"])/dfOutput["offer_price"],4)
    dfOutput["Current walmart margin at NLC category"] = pd.cut(dfOutput["Current walmart margin at NLC"], bins=[-np.inf, -.1, 0,0.1, 0.15, 0.2, 0.25, 0.3, np.inf], labels=["<-10%", "-10% to 0%", "0% to 10%", "10% to 15%", "15% to 20%", "20% to 25%", "25% to 30%", ">=30%"])

    dfOutput["Walmart margin at new NLC category"] = pd.cut(dfOutput["Final walmart margin"], bins=[-np.inf, -.1, 0,0.1, 0.15, 0.2, 0.25, 0.3, np.inf], labels=["<-10%", "-10% to 0%", "0% to 10%", "10% to 15%", "15% to 20%", "20% to 25%", "25% to 30%", ">=30%"])


    dfOutput["Total margin"] = round(dfOutput["Current walmart margin at NLC"] + dfOutput["current_nlc_margin"],4)
    margin_splits = [0.50, 0.60]

    for margin in margin_splits:
        dfOutput[f"Wmt margin {int(margin * 100)}% group"] = round(dfOutput["Total margin"] * (1-margin), 4)
        dfOutput[f"Price margin split {int(margin * 100)}%"] = round(dfOutput["offer_price"] * (1- dfOutput[f"Wmt margin {int(margin * 100)}% group"]),2)
        dfOutput[f"Price margin split {int(margin * 100)}%"] = np.minimum(dfOutput[f"Price margin split {int(margin * 100)}%"], dfOutput["Node level cost - 20% margin"])

    dfOutput = dfOutput.merge(dfSKUNodeRevenue, how = "left", on = "SKU-Node").merge(dfSKURevenue, how = "left", on = "Product Code")

    dfOutputAll = dfOutput.copy()
    print("Total of rows in dfOutputAll:", len(dfOutputAll))

    dfOutput = dfOutput[dfOutput["Final node level cost category"] != "N/A"].copy()

    print("Rows we can work with (not N/A):", len(dfOutput))

    display(dfOutput["New NLC"].value_counts())

    return dfOutputAll

## Df Output

In [86]:
dfOutputAll4 = calculate_new_node_level_cost(dfInvAll, dfAvgWmtPrice, dfMAP, minWmMargin=0, distanceMAP=0.05, min_units=4)
print("---- Done with min_units = 4 ----")
dfOutputAll8 = calculate_new_node_level_cost(dfInvAll, dfAvgWmtPrice, dfMAP, minWmMargin=0, distanceMAP=0.05, min_units=8)
print("---- Done with min_units = 8 ----")

dfOutput4 = dfOutputAll4[dfOutputAll4["Final node level cost category"] != "N/A"].copy()
dfOutput8 = dfOutputAll8[dfOutputAll8["Final node level cost category"] != "N/A"].copy()

dfOutput4Notin8 = dfOutput4[~dfOutput4["SKU-Node"].isin(dfOutput8["SKU-Node"])].copy()
dfOutputInitial = pd.concat([dfOutput8, dfOutput4Notin8], ignore_index=True)
print(len(dfOutputInitial))

dfOutput = dfOutputInitial.merge(dfCurrentTests[["SKU-Node","Final target","Start date","Sub-group"]], how="left", left_on="SKU-Node", right_on="SKU-Node")

# dfOutput = dfOutput[dfOutput["Product Code"].isin(dfRollbacks["Product Code"]) == False].copy()

dfOutput

Total of rows in dfOutputAll: 2403810
Rows we can work with (not N/A): 2292521


New NLC
No     2282946
Yes       9575
Name: count, dtype: int64

---- Done with min_units = 4 ----
Total of rows in dfOutputAll: 1600702
Rows we can work with (not N/A): 1523611


New NLC
No     1517412
Yes       6199
Name: count, dtype: int64

---- Done with min_units = 8 ----
2292714


,Product Code,Identifier,Warehouse Code,Available,Purchase Price+FET,date,Node type,offer_price,unit_cost,MAP,Shipping cost,Cost+Shipping,Is MAP now?,Node level cost - 6% margin,Walmart Margin - 6% margin,Node cost < MAP - min margin% - 6% margin,Node cost - MAP - 6% margin,Target for node level cost? - 6% margin,Node level cost - 8% margin,Walmart Margin - 8% margin,Node cost < MAP - min margin% - 8% margin,Node cost - MAP - 8% margin,Target for node level cost? - 8% margin,Node level cost - 11% margin,Walmart Margin - 11% margin,Node cost < MAP - min margin% - 11% margin,Node cost - MAP - 11% margin,Target for node level cost? - 11% margin,Node level cost - 12% margin,Walmart Margin - 12% margin,Node cost < MAP - min margin% - 12% margin,Node cost - MAP - 12% margin,Target for node level cost? - 12% margin,Node level cost - 13% margin,Walmart Margin - 13% margin,Node cost < MAP - min margin% - 13% margin,Node cost - MAP - 13% margin,Target for node level cost? - 13% margin,Node level cost - 14% margin,Walmart Margin - 14% margin,Node cost < MAP - min margin% - 14% margin,Node cost - MAP - 14% margin,Target for node level cost? - 14% margin,Node level cost - 15% margin,Walmart Margin - 15% margin,Node cost < MAP - min margin% - 15% margin,Node cost - MAP - 15% margin,Target for node level cost? - 15% margin,Node level cost - 20% margin,Walmart Margin - 20% margin,Node cost < MAP - min margin% - 20% margin,Node cost - MAP - 20% margin,Target for node level cost? - 20% margin,current_nlc_price,Final node level cost category,Final node level cost,Final walmart margin,Final price change %,Final price change category,SKU-Node,Brand code,New NLC,Min units,current_nlc_margin,current_nlc_margin category,Current walmart margin at National,Current walmart margin at NLC,Current walmart margin at NLC category,Walmart margin at new NLC category,Total margin,Wmt margin 50% group,Price margin split 50%,Wmt margin 60% group,Price margin split 60%,sku_node_revenue,sku_revenue,sku_sales_category,Final target,Start date,Sub-group
0,ACCE0131820535YXL,628326659,5614,87,56.95,2026-03-18,HUB,68.360,61.79,NaN,0.0,56.95,No,60.59,0.1137,No,NaN,Yes,61.90,0.0945,No,NaN,Yes,63.99,0.0639,No,NaN,Yes,64.72,0.0532,No,NaN,Yes,65.46,0.0424,No,NaN,Yes,66.22,0.0313,No,NaN,Yes,67.00,0.0199,No,NaN,Yes,71.19,-0.0414,No,NaN,No,63.99,11%,63.99,0.0639,0.000,No change,ACCE0131820535YXL-628326659,ACCE,No,8,0.1100,10.8% to <11.2%,0.0961,0.0639,0% to 10%,0% to 10%,0.1739,0.0870,62.41,0.0696,63.60,NaN,137.98,95-100%,Updates test,2026-02-12,Update
1,ACCE0221316580T,6283261113,5237,16,38.11,2026-03-18,HUB,45.930,37.94,NaN,0.0,38.11,No,40.54,0.1174,No,NaN,Yes,41.42,0.0982,No,NaN,Yes,42.82,0.0677,No,NaN,Yes,43.31,0.0570,No,NaN,Yes,43.80,0.0464,No,NaN,Yes,44.31,0.0353,No,NaN,Yes,44.84,0.0237,No,NaN,Yes,47.64,-0.0372,No,NaN,No,43.25,11%,42.82,0.0677,-0.010,Decrease,ACCE0221316580T-6283261113,ACCE,No,8,0.1188,11.2% to <13%,0.1740,0.0583,0% to 10%,0% to 10%,0.1771,0.0886,41.86,0.0708,42.68,NaN,309.02,95-100%,Normal strategy,2026-01-05,11%
2,ACCE0221316580T,6283261117,5266,20,33.90,2026-03-18,HUB,45.930,37.94,NaN,0.0,33.90,No,36.06,0.2149,No,NaN,Yes,36.85,0.1977,No,NaN,Yes,38.09,0.1707,No,NaN,Yes,38.52,0.1613,No,NaN,Yes,38.97,0.1515,No,NaN,Yes,39.42,0.1417,No,NaN,Yes,39.88,0.1317,No,NaN,Yes,42.37,0.0775,No,NaN,Yes,39.21,11%,38.09,0.1707,-0.029,Decrease,ACCE0221316580T-6283261117,ACCE,No,8,0.1354,13% to <15%,0.1740,0.1463,10% to 15%,15% to 20%,0.2817,0.1408,39.46,0.1127,40.75,NaN,309.02,95-100%,Normal strategy,2026-01-05,11%
3,ACCE0221316580T,6283261134,5234,8,33.35,2026-03-18,HUB,45.930,37.94,NaN,7.0,40.35,No,42.48,0.2275,No,NaN,Yes,43.25,0.2108,No,NaN,Yes,44.47,0.1842,No,NaN,Yes,44.90,0.1748,No,NaN,Yes,45.33,0.1655,No,NaN,Yes,45.78,0.1557,No,NaN,Yes,46.24,0.1457,No,NaN,Yes,48.69,0.0923,No,NaN,Yes,41.69,11%,44.47,0.1842,0.067,Increase,ACCE0221316580T-6283261134,ACCE,No,8,0.0321,0% to <6%,0.1740,0.0923,0% to 10%,15% to 20%,0.1244,0.0622,43.07,0.0498,43.64,177.88,309.02,95-100%,Shipp

In [87]:
dfOutput["Final target"].value_counts()

Final target
Added                       732431
Margin test                 376738
Normal strategy             330893
Updated                     279448
No sales test               205943
Decreased - margin > 20%    130208
Updates test                104817
Wm margin split test         92898
Shipping cost added          25442
Less sales back               2116
Less sales                    2102
Name: count, dtype: int64

In [88]:
dfOutput["current_nlc_margin category"].value_counts()

current_nlc_margin category
10.8% to <11.2%    707992
11.2% to <13%      377706
8% to <10.8%       293760
13% to <15%        274363
6% to <8%          163825
15% to <17%        161512
17% to <20%        135760
>=20%              114170
0% to <6%           49051
<0%                  5000
Name: count, dtype: int64

In [89]:
dfAnalyze = dfOutput[dfOutput["current_nlc_margin"] >0.205].copy()#["Final target"].value_counts()
dfAnalyze

,Product Code,Identifier,Warehouse Code,Available,Purchase Price+FET,date,Node type,offer_price,unit_cost,MAP,Shipping cost,Cost+Shipping,Is MAP now?,Node level cost - 6% margin,Walmart Margin - 6% margin,Node cost < MAP - min margin% - 6% margin,Node cost - MAP - 6% margin,Target for node level cost? - 6% margin,Node level cost - 8% margin,Walmart Margin - 8% margin,Node cost < MAP - min margin% - 8% margin,Node cost - MAP - 8% margin,Target for node level cost? - 8% margin,Node level cost - 11% margin,Walmart Margin - 11% margin,Node cost < MAP - min margin% - 11% margin,Node cost - MAP - 11% margin,Target for node level cost? - 11% margin,Node level cost - 12% margin,Walmart Margin - 12% margin,Node cost < MAP - min margin% - 12% margin,Node cost - MAP - 12% margin,Target for node level cost? - 12% margin,Node level cost - 13% margin,Walmart Margin - 13% margin,Node cost < MAP - min margin% - 13% margin,Node cost - MAP - 13% margin,Target for node level cost? - 13% margin,Node level cost - 14% margin,Walmart Margin - 14% margin,Node cost < MAP - min margin% - 14% margin,Node cost - MAP - 14% margin,Target for node level cost? - 14% margin,Node level cost - 15% margin,Walmart Margin - 15% margin,Node cost < MAP - min margin% - 15% margin,Node cost - MAP - 15% margin,Target for node level cost? - 15% margin,Node level cost - 20% margin,Walmart Margin - 20% margin,Node cost < MAP - min margin% - 20% margin,Node cost - MAP - 20% margin,Target for node level cost? - 20% margin,current_nlc_price,Final node level cost category,Final node level cost,Final walmart margin,Final price change %,Final price change category,SKU-Node,Brand code,New NLC,Min units,current_nlc_margin,current_nlc_margin category,Current walmart margin at National,Current walmart margin at NLC,Current walmart margin at NLC category,Walmart margin at new NLC category,Total margin,Wmt margin 50% group,Price margin split 50%,Wmt margin 60% group,Price margin split 60%,sku_node_revenue,sku_revenue,sku_sales_category,Final target,Start date,Sub-group
419,ACCE0221522560V,628326984,6081,18,60.41901,2026-03-18,HUB,82.725,63.07,NaN,0.0,60.41901,No,64.28,0.2230,No,NaN,Yes,65.67,0.2062,No,NaN,Yes,67.89,0.1793,No,NaN,Yes,68.66,0.1700,No,NaN,Yes,69.45,0.1605,No,NaN,Yes,70.25,0.1508,No,NaN,Yes,71.08,0.1408,No,NaN,Yes,75.52,0.0871,No,NaN,Yes,81.39,11%,67.89,0.1793,-0.166,Decrease,ACCE0221522560V-628326984,ACCE,No,8,0.2577,>=20%,0.2376,0.0161,0% to 10%,15% to 20%,0.2738,0.1369,71.40,0.1095,73.67,543.84,4797.62,70-80%,Updated,2026-03-10,8%
1800,ACCE0322231530WXL,628326984,6081,24,147.85000,2026-03-18,HUB,215.330,158.13,NaN,0.0,147.85000,No,157.29,0.2695,No,NaN,Yes,160.71,0.2537,No,NaN,Yes,166.12,0.2285,No,NaN,Yes,168.01,0.2198,No,NaN,Yes,169.94,0.2108,No,NaN,Yes,171.92,0.2016,No,NaN,Yes,173.94,0.1922,No,NaN,Yes,184.81,0.1417,No,NaN,Yes,186.06,11%,166.12,0.2285,-0.107,Decrease,ACCE0322231530WXL-628326984,ACCE,No,8,0.2054,>=20%,0.2656,0.1359,10% to 15%,20% to 25%,0.3413,0.1706,178.59,0.1365,184.81,NaN,NaN,NaN,Decreased - margin > 20%,2026-03-13,20%
4596,ADVA08822529575G,628326959,6013,16,230.38000,2026-03-18,HUB,353.000,253.77,NaN,0.0,230.38000,No,245.09,0.3057,No,NaN,Yes,250.41,0.2906,No,NaN,Yes,258.85,0.2667,No,NaN,Yes,261.80,0.2584,No,NaN,Yes,264.80,0.2499,No,NaN,Yes,267.88,0.2411,No,NaN,Yes,271.04,0.2322,No,NaN,Yes,287.97,0.1842,No,NaN,Yes,290.85,11%,258.85,0.2667,-0.110,Decrease,ADVA08822529575G-628326959,ADVA,No,8,0.2079,>=20%,0.2811,0.1761,15% to 20%,25% to 30%,0.3840,0.1920,285.22,0.1536,287.97,NaN,NaN,NaN,Decreased - margin > 20%,2026-03-13,20%
4657,ADVA21722511H,628326959,6013,16,241.63000,2026-03-18,HUB,329.040,259.18,NaN,0.0,241.63000,No,257.05,0.2188,No,NaN,Yes,262.64,0.2018,No,NaN,Yes,271.49,0.1749,No,NaN,Yes,274.58,0.1655,No,NaN,Yes,277.74,0.1559,No,NaN,Yes,280.97,0.1461,No,NaN,Yes,284.27,0.1361,No,NaN,Yes,302.04,0.0821,No,NaN,Yes,308.22,11%,271.49,0.1749,-0.119,Decrease,ADVA21722511H-628326959,ADVA,No,8,0.2160,>=20%,0.2123,0.0633,0% to 10%,15% to 20%,0.2793,0.139

## New sku-nodes + checks

In [90]:
dfOutputNew = dfOutput[dfOutput["New NLC"] == "Yes"].copy()
display(dfOutputNew["Min units"].value_counts())
dfOutputNew["Brand code"].value_counts()

Min units
8    6199
4    3376
Name: count, dtype: int64

Brand code
IRHD    2162
PIRE    1677
NEXE     638
CONT     614
FALK     457
        ... 
AMPO       1
LANV       1
VERC       1
TRAV       1
POWE       1
Name: count, Length: 114, dtype: int64

In [91]:
# dfRollbacksEnded = dfRollbacksAll[dfRollbacksAll["End date"] == '2026-03-13'].copy()


In [92]:
# dfOutputNew[dfOutputNew["Product Code"].isin(dfRollbacksEnded["Product Code"])]["Product Code"].value_counts()

In [93]:
dfOutputNew["Identifier"].value_counts()

Identifier
6283261024    539
628326472     539
6283261060    473
628326750     473
628326846     294
             ... 
6283261536      1
628326599       1
628326985       1
6283261354      1
628326148       1
Name: count, Length: 760, dtype: int64

In [94]:
dfOutputNew["Warehouse Code"].value_counts()

Warehouse Code
5548         1078
5776          946
6020          424
8184          417
5251          332
             ... 
5263            1
5576            1
5592            1
5391-Vans       1
5418-Vans       1
Name: count, Length: 717, dtype: int64

## Update current nlc margins + if is in stock or not

In [ ]:
update_cols = ["current_nlc_margin_date", "current_nlc_margin", "Current walmart margin at NLC category","sku_sales_category"]
dfCurrentTestsAll = dfCurrentTestsAll.drop(columns=update_cols)
dfCurrentTestsAll = dfCurrentTestsAll.merge(dfOutput[["SKU-Node","current_nlc_margin", "Current walmart margin at NLC category","sku_sales_category"]], how="left", left_on="SKU-Node", right_on="SKU-Node")
dfCurrentTestsAll["current_nlc_margin_date"] = today_str
dfCurrentTestsAll["sku_sales_category"] = dfCurrentTestsAll["sku_sales_category"].astype(str).replace("nan", "No sales")
dfCurrentTestsAll["Is in stock?"] = np.where(dfCurrentTestsAll["current_nlc_margin"].isna(), "No", "Yes")
dfCurrentTestsAll

,Product Code,Identifier,Final target,Final price category,Final price,Final delta vs current %,Final price change category final,Category inventory,Node type,Start date,Min units,Sub-group,Is in stock?,Last price update,current_nlc_margin,Current walmart margin at NLC category,sku_sales_category,current_nlc_margin_date
0,ACCE0131318560V,6283261261,Normal strategy,11%,70.06,0.0000,No change,NaN,HUB,2026-01-05,8,11%,No,NaN,NaN,NaN,No sales,2026-03-18
1,ACCE0131318560V,6283261264,Normal strategy,11%,70.76,0.0000,No change,NaN,HUB,2026-01-05,8,11%,No,NaN,NaN,NaN,No sales,2026-03-18
2,ACCE0131318560V,6283261391,Normal strategy,11%,67.87,NaN,New,NaN,HUB,2026-01-05,8,11%,No,NaN,NaN,NaN,No sales,2026-03-18
3,ACCE0131418555V,6283261391,Normal strategy,11%,56.13,-0.0200,Decrease,NaN,HUB,2026-01-05,8,11%,No,NaN,NaN,NaN,No sales,2026-03-18
4,ACCE0221316580T,6283261113,Normal strategy,11%,43.25,0.0300,Increase,NaN,HUB,2026-01-05,8,11%,Yes,NaN,0.1188,0% to 10%,95-100%,2026-03-18
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3271825,NEXE1182025550VXL,6283261120,Margin test,14%,208.60,0.0545,Increase,NaN,HUB,2026-03-12,4,14%,Yes,2026-03-17,0.1400,10% to 15%,60-70%,2026-03-18
3271826,NEXE1182025550VXL,62832640,Margin test,14%,208.60,0.0545,Increase,NaN,HUB,2026-03-12,4,14%,Yes,2026-03-17,0.1400,10% to 15%,60-70%,2026-03-18
3271827,NEXE1182027555H,6283261120,Margin test,11%,202.69,0.0122,Increase,NaN,HUB,2026-03-12,4,11%,Yes,2026-03-17,0.1100,-10% to 0%,80-90%,2026-03-18
3271828,NEXE1182027555H,62832640,Margin test,11%,202.69,0.0122,Increase,NaN,HUB,2026-03-12,4,11%,Yes,2026-03-17,0.1100,-10% to 0%,80-90%,2026-03-18


## Inv check

In [96]:
# dfInvDate = pricing.get_inventory('2026-03-16', add_rebates = False, amazon = False, greater3=True)
# dfInvDate1 = pricing.get_inventory('2026-03-09', add_rebates = False, amazon = False, greater3=True)

In [97]:
dfInvDate = pricing.get_inventory(today_str, add_rebates = False, amazon = False, greater3=True)
dfInvDate1 = pricing.get_inventory(yesterday_str, add_rebates = False, amazon = False, greater3=True)

dfVendor = dw.get_vendor_code_table()
dfVendor = dfVendor.rename(columns={"warehouse_code": "Warehouse Code"})
dfVendorRef = dfVendor[["Warehouse Code", "vendorReference", "vendor_code","warehouse_name"]].copy()

dfVendor = dfVendor[["Warehouse Code", "vendor_code"]].copy()

dfInvComp = dfInvDate.merge(dfInvDate1, how="outer", on=["Product Code", "Warehouse Code"], suffixes=("_current", "_prev")).merge(dfVendor, how="left", on="Warehouse Code")
dfInvComp["Delta price%"] = round((dfInvComp["Purchase Price+FET_current"] - dfInvComp["Purchase Price+FET_prev"])/ dfInvComp["Purchase Price+FET_prev"],4)
dfInvComp["Delta price category"] = np.where(dfInvComp["Delta price%"] < 0, "Decrease",
                                              np.where(dfInvComp["Delta price%"] > 0, "Increase", np.where(dfInvComp["Delta price%"] == 0, "No change", 
                                                                                                           np.where(dfInvComp["Purchase Price+FET_prev"].isna(), "Only in current file", 
                                                                                                                     np.where(dfInvComp["Purchase Price+FET_current"].isna(), "Only in prev file", "Unknown")))))
#keep only cols Product Code, Warehouse Code, Purchase Price+FET_13th, Purchase Price+FET_12th, Delta price%, Delta price category, Available_13th, Available_12th
dfInvComp = dfInvComp[["Product Code", "Warehouse Code", "Purchase Price+FET_current", "Purchase Price+FET_prev", "Delta price%", "Delta price category", "Available_current", "Available_prev","vendor_code"]].copy()
dfInvComp["Brand code"] = dfInvComp["Product Code"].str[:4]
dfInvCompAnalysis = dfInvComp.groupby(["Delta price category"]).agg({"Product Code": "count", "Delta price%": "mean"}).reset_index().rename(columns={"Product Code": "Count SKU-Whs", "Delta price%": "Avg price change %"})
dfInvCompAnalysis["Avg price change %"] = dfInvCompAnalysis["Avg price change %"].round(4)
dfInvCompAnalysis

Download 100.
Last modified time 2026-03-18T10:01:24.060Z
Download 95.
Download 100.
Download 100.
Last modified time 2026-03-17T11:01:42.267Z
Download 94.
Download 100.


,Delta price category,Count SKU-Whs,Avg price change %
0,Decrease,6768,-0.0352
1,Increase,30165,0.1011
2,No change,2121132,0.0000
3,Only in current file,35383,NaN
4,Only in prev file,29586,NaN


In [145]:
vendor_col = "vendor_code"
brand_col = "Brand code"
col_group = vendor_col
category = "Increase"

dfInvCompTotalGrouped = dfInvComp.groupby(col_group).agg({"Product Code": "count"}).rename(columns={"Product Code": "Total wh-sku lines"}).reset_index()
dfInvCompIncr = dfInvComp[dfInvComp["Delta price category"] ==category].copy()

dfInvCompIncrBrand = dfInvCompIncr.groupby(col_group).agg({"Product Code": "count","Delta price%": "mean"}).reset_index().rename(columns={"Product Code": f"Count of wh-sku price {category}","Delta price%": f"Avg price {category} %"})
dfInvCompIncrBrand = dfInvCompIncrBrand.sort_values(f"Count of wh-sku price {category}", ascending=False)
dfInvCompIncrBrand[f"Avg price {category} %"] = dfInvCompIncrBrand[f"Avg price {category} %"].round(3)
dfInvCompIncrBrand = dfInvCompIncrBrand.merge(dfInvCompTotalGrouped, how="left", on=col_group)
dfInvCompIncrBrand[f"% Lines {category}"] = round((dfInvCompIncrBrand[f"Count of wh-sku price {category}"] / dfInvCompIncrBrand["Total wh-sku lines"]),3)
# dfInvCompIncrBrand = dfInvCompIncrBrand.sort_values(f"% Lines {category}", ascending=False)
dfInvCompIncrBrand[dfInvCompIncrBrand["Total wh-sku lines"]>1000]

,vendor_code,Count of wh-sku price Increase,Avg price Increase %,Total wh-sku lines,% Lines Increase
0,TFS,25540,0.109,43110,0.592
1,CLA,2187,0.074,14615,0.150
2,GTC,445,0.006,10760,0.041
3,ATD,303,0.022,164573,0.002
4,MIT,236,0.047,1688,0.140
6,KMT,146,0.040,22061,0.007
7,5673,132,0.016,16282,0.008
8,TSW,105,0.060,12058,0.009
9,NGT,88,0.031,4378,0.020
10,PTA,84,0.476,1228,0.068


# Tests (no)

## Test less sales

### Original

In [99]:
# dfOutputLessSales = dfOutput.merge(finalTargetWithGroupsMerge, how="left", on="Product Code")
# dfOutputLessSales = dfOutputLessSales[dfOutputLessSales["group"].notna()].copy()
# dfOutputLessSalesFilt = dfOutputLessSales[dfOutputLessSales["current_nlc_margin"]>=0.109].copy()
# dfOutputLessSalesFilt= dfOutputLessSalesFilt.merge(dfCurrentTests[["SKU-Node","Final target"]], how="left", left_on="SKU-Node", right_on="SKU-Node")
# dfOutputLessSalesFilt = dfOutputLessSalesFilt[dfOutputLessSalesFilt["Final target"] != "Margin test"].copy()
# print("Total rows with less sales:", len(dfOutputLessSales))
# display(dfOutputLessSales["current_nlc_margin category"].value_counts())
# print("Filtered:", len(dfOutputLessSalesFilt))
# display(dfOutputLessSalesFilt["current_nlc_margin category"].value_counts())
# display(dfOutputLessSalesFilt["group"].value_counts())

# dfOutputLessSalesFiltCheck = dfOutputLessSalesFilt.groupby(["current_nlc_margin category", "group"]).agg({"Product Code": "count"})

# dfOutputLessSalesFiltCheck = dfOutputLessSalesFiltCheck.rename(columns={"Product Code": "count"}).reset_index()
# # dfOutputLessSalesFiltCheck pivot by group
 
# dfOutputLessSalesFiltCheck = dfOutputLessSalesFiltCheck.pivot(index="current_nlc_margin category", columns="group", values="count").fillna(0).astype(int)

# dfOutputLessSalesFiltCheck

In [100]:
# dfOutputLessSalesFilt["New NLC"].value_counts()

In [101]:
# dfTest = dfOutputLessSalesFilt.copy()
# dfTest = dfTest[(dfTest["Target for node level cost? - 11% margin"] == "Yes") &
#                                       (dfTest["group"].notna()) &
#                                       (dfTest["current_nlc_price"].notna())].copy()

# print("Total rows in test:", len(dfTest))

# dfTest["Group"] = np.where(dfTest["group"] == 1, "10%",
#                                       np.where(dfTest["group"] == 2, "8%",
#                                               np.where(dfTest["group"] == 3, "6%", "No group")))
# dfTest["Test price"]= np.where(dfTest["group"] == 1, dfTest["Node level cost - 10% margin"],
#                                       np.where(dfTest["group"] == 2, dfTest["Node level cost - 8% margin"],
#                                               np.where(dfTest["group"] == 3, dfTest["Node level cost - 6% margin"], np.nan)))

# dfTest["Delta test to current NLC %"] = round((dfTest["Test price"] - dfTest["current_nlc_price"])/dfTest["current_nlc_price"],4)
# dfTest["Delta test to current NLC category"] = np.where(dfTest["Delta test to current NLC %"] < 0, "Decrease",
#                                                                 np.where(dfTest["Delta test to current NLC %"] > 0, "Increase", "No change"))

# relevantCols = ["Product Code", "Identifier","SKU-Node", "Group", "Test price", "Delta test to current NLC %", "Delta test to current NLC category",
#                     "Node type", "Min units","current_nlc_margin","qty_pct_change_L2W_vs_P90"]



# dfTest = dfTest[relevantCols].copy().rename(columns = {"current_nlc_margin": "NLC margin before update"})

# # Now display # of increases, decreases, no change by each Group + average of "Delta test to current NLC %" by group
# summary = (
#     dfTest
#         .groupby("Group")
#         .agg(
#             count=("Delta test to current NLC category", "count"),
#             increases=("Delta test to current NLC category", lambda x: (x == "Increase").sum()),
#             decreases=("Delta test to current NLC category", lambda x: (x == "Decrease").sum()),
#             no_change=("Delta test to current NLC category", lambda x: (x == "No change").sum()),
#             avg_delta_test_to_current_nlc_pct=("Delta test to current NLC %", "mean"),
#         )
#         .reset_index()
# )
# summary

In [102]:
# dfTestDSV= dfTest[["Product Code", "Identifier","Test price","SKU-Node"]].rename(columns = {"Product Code": "SKU", "Identifier": "Source", "Test price": "Price"})
# dfTestDSV

In [103]:
# usecols = ["Product Code", "Identifier","Group", "Test price", "Delta test to current NLC category", "Delta test to current NLC %", 
#            "Node type", "Min units","NLC margin before update","qty_pct_change_L2W_vs_P90"]
# dfTestTracker = dfTest[usecols].copy()

# dfTestTracker["Final target"] = "Less sales"

# dfTestTracker = dfTestTracker.rename(columns={
#     "Group": "Final price category",
#     "Delta test to current NLC category": "Final price change category final",
#     "Delta test to current NLC %": "Final delta vs current %",
#     "Test price": "Final price"})
# dfTestTracker["SKU-Node"] = dfTestTracker["Product Code"] + "-" + dfTestTracker["Identifier"].astype(str)
# dfTestTracker["Min units"] = dfTestTracker["Min units"].astype(int)
# dfTestTracker["Start date"] = date_str
# dfTestTracker

In [104]:
# dfTestTracker.to_csv("Test.csv", index=False)

In [105]:
# dfCurrentTestsAllCheck = dfCurrentTestsAll.copy()
# dfCurrentTestsAllCheck["SKU-Node"] = dfCurrentTestsAllCheck["Product Code"] + "-" + dfCurrentTestsAllCheck["Identifier"]

# df = dfCurrentTestsAllCheck[dfCurrentTestsAllCheck["SKU-Node"].isin(dfOutputLessSalesFilt["SKU-Node"])].copy()#["Final target"].value_counts()
# dfMargin = df[df["Final target"] == "Added"].copy()
# dfMargin["Start date"].value_counts()

### Pivot back

In [106]:
# dfPrevDSVOriginal = dfPrevDSVOriginal.rename(columns={"sku": "Product Code",  "source":"Identifier", "walmart_dsv_price": "prev_nlc_price"}).drop(columns=["date"])
# dfPrevDSVOriginal

In [107]:
# dfTestLessSales = dfCurrentTestsAll[(dfCurrentTestsAll["Final target"] == "Less sales")& (dfCurrentTestsAll["Start date"] == "2026-02-11")].copy()
# dfTest = dfTestLessSales[dfTestLessSales["Final price category"].isin(["6%", "8%"])].copy()
# dfTest = dfTest.merge(dfPrevDSVOriginal, how="left", on=["Product Code", "Identifier"])
# dfTest["SKU-Node"] = dfTest["Product Code"] + "-" + dfTest["Identifier"]
# dfTest

In [108]:
# dfTestDSV = dfTest[["Product Code", "Identifier","prev_nlc_price"]].copy().rename(columns={"Product Code": "SKU", "Identifier": "Source", "prev_nlc_price": "Price"})
# dfTestDSV

In [109]:
# usecols = ["Product Code", "Identifier","prev_nlc_price", "Final price change category final","Final price category",
#            "Node type", "Min units","qty_pct_change_L2W_vs_P90"]
# dfTestTracker = dfTest[usecols].copy()

# dfTestTracker["Final target"] = "Less sales back"
# dfTestTracker["Final price change category final"] = "No change"

# dfTestTracker = dfTestTracker.rename(columns={
#     "prev_nlc_price": "Final price"})
# dfTestTracker["SKU-Node"] = dfTestTracker["Product Code"] + "-" + dfTestTracker["Identifier"].astype(str)
# dfTestTracker["Min units"] = dfTestTracker["Min units"].astype(int)
# dfTestTracker["Start date"] = date_str
# dfTestTracker

### Revert test

In [110]:
# change = ["Less sales","Less sales back"]

# dfOutputChange = dfOutput[dfOutput["Final target"].isin(change) == True].copy()

# display(dfOutputChange["Final price change category"].value_counts())


In [111]:
# #dfOutputChange have an idea by each Final target, how many are Increase, Decrease, No change in terms of price change category, and average of price change % by Final target
# summary = (
#     dfOutputChange.groupby("Final target")
#                  .agg(
#                      count=("Final price change category", "count"),
#                      increases=("Final price change category", lambda x: (x == "Increase").sum()),
#                      decreases=("Final price change category", lambda x: (x == "Decrease").sum()),
#                      no_change=("Final price change category", lambda x: (x == "No change").sum()),
#                      avg_price_change_pct=("Final price change %", "mean")
#                  )
#                  .reset_index()
# )
# summary

In [112]:
# dfTest = dfOutputChange.copy()
# # dfTest["Test price"] where Final price change category = Increase then Test price = Final node level cost, else = current_nlc_price
# dfTest["Final target"] = "Normal strategy"
# dfTest["Final price"] = dfTest.apply(lambda row: row["Final node level cost"] if row["Final price change category"] == "Increase" else row["current_nlc_price"], axis=1)
# dfTest["Final delta vs current %"] = round((dfTest["Final price"] - dfTest["current_nlc_price"])/dfTest["current_nlc_price"],4)
# dfTest["Final price change category final"] = np.where(dfTest["Final delta vs current %"] < 0, "Decrease", np.where(dfTest["Final delta vs current %"] > 0, "Increase", "No change"))
# relevantCols = ["Product Code", "Identifier","Final target","Final price", "Final delta vs current %", "Final price change category final", "Node type", "Min units"]

# dfTest = dfTest[relevantCols].copy()
# dfTest

In [113]:
# dfTestDSV = dfTest[["Product Code", "Identifier","Final price"]].copy().rename(columns={"Product Code": "SKU", "Identifier": "Source", "Final price": "Price"})
# dfTestDSV["SKU-Node"] = dfTestDSV["SKU"] + "-" + dfTestDSV["Source"].astype(str)
# dfTestDSV

In [114]:
# dfTestTracker = dfTest.copy()

# dfTestTracker["SKU-Node"] = dfTestTracker["Product Code"] + "-" + dfTestTracker["Identifier"].astype(str)
# dfTestTracker["Min units"] = dfTestTracker["Min units"].astype(int)
# dfTestTracker["Start date"] = date_str
# dfTestTracker

## Test high margin by brand

In [115]:
# pathAssignments = r"H:\Shared drives\07_Finance\07_10_Pricing_department\07_11_01_WalmartB2B\07_10_01_09 Documentation\Price tests docs & analysis\Incr margin brand phase 2\pricing_test_assignments.xlsx"
# read_cols = ["brand_code","price_level_pct","allocation_pct","test_group"]
# dfAssignments = pd.read_excel(pathAssignments, sheet_name="assignments_long", dtype={"SKU-Node": str}, usecols = read_cols)
# dfAssignments = dfAssignments.sort_values(["brand_code","allocation_pct","price_level_pct","test_group"], ascending=[True, False, True, True]).reset_index(drop=True)
# dfAssignments

In [116]:
#create dataframe where just 2 colums, allocation_pct and group, where for allocation_pct = 0.5 then group = "group_1", for allocation_pct = 0.3 then group = "Group 2", for allocation_pct = 0.2 then group = "Group 3"

In [117]:
# dfOutputTarget = dfOutput[dfOutput["Brand code"].isin(dfAssignments["brand_code"].unique())].copy()

# # dfOutputTarget = dfOutput.copy()


# exclude_targets =["Shipping cost added", "Wm margin split test"]
# dfOutputTarget = dfOutputTarget[~dfOutputTarget["Final target"].isin(exclude_targets)]


# target_col = "Target for node level cost? - 14% margin"
# dfOutputTarget = dfOutputTarget[dfOutputTarget[target_col] == "Yes"].copy()
# dfOutputTarget = dfOutputTarget[dfOutputTarget["current_nlc_margin"] < 0.16].copy()
# dfOutputTarget = dfOutputTarget[dfOutputTarget["sku_revenue"] > 500].copy()

# # dfOutputTarget = dfOutputTarget[dfOutputTarget["current_nlc_margin"] < 0.15].copy()
# print("Total sku-nodes", len(dfOutputTarget))

# dfOutputTargetAnalysis = dfOutputTarget.groupby("Brand code").agg({"SKU-Node": "count", "sku_node_revenue": "sum"}).reset_index().rename(columns={"SKU-Node": "Count SKU-Node", "sku_node_revenue": "Total inv amount"})
# dfOutputTargetAnalysis["Total inv amount"] = dfOutputTargetAnalysis["Total inv amount"].round(2)
# dfOutputTargetAnalysis["% inv amount"] = round(dfOutputTargetAnalysis["Total inv amount"] / total_revenue,3)
# print("Total % inv amt sku-node:", dfOutputTargetAnalysis["% inv amount"].sum().round(3))
# perc_rev_sku = dfSKURevenue[dfSKURevenue["Product Code"].isin(dfOutputTarget["Product Code"].unique()) == True]["sku_revenue"].sum()/total_revenue
# print("Total % inv amt sku:", round(perc_rev_sku,3))
# dfOutputTargetAnalysis.sort_values("% inv amount", ascending=False)

In [118]:
# import pandas as pd
# import numpy as np

# # Copy
# df = dfOutputTarget.copy()

# # ----------------------------
# # 1) Build SKU-level table
# # ----------------------------
# def mode_or_nan(x):
#     m = x.mode(dropna=True)
#     return m.iloc[0] if len(m) > 0 else np.nan

# sku_summary = (
#     df.groupby("Product Code")
#       .apply(lambda g: pd.Series({
#           "Brand code": mode_or_nan(g["Brand code"]),
#           "sku_sales_category": mode_or_nan(g["sku_sales_category"]),
#           "Current walmart margin at NLC category": mode_or_nan(g["Current walmart margin at NLC category"]),
#           "current_nlc_margin category": mode_or_nan(g["current_nlc_margin category"]),
#           "num_sku_nodes": g["SKU-Node"].nunique()
#       }))
#       .reset_index()
# )

# # ----------------------------
# # 2) Assign groups within each Brand code + sku_sales_category
# # ----------------------------
# # For every Brand code, split each sku_sales_category bucket:
# #   50% -> group_1
# #   25% -> group_2
# #   25% -> group_3
# # Then concatenate all brands together.

# def assign_groups_within_brand_and_sales_category(g):
#     g = g.sample(frac=1, random_state=42).copy()

#     n = len(g)
#     n_g1 = int(round(n * 0.50))
#     n_g2 = int(round(n * 0.25))
#     n_g3 = n - n_g1 - n_g2

#     groups = (
#         ["group_1"] * n_g1 +
#         ["group_2"] * n_g2 +
#         ["group_3"] * n_g3
#     )

#     g["test_group"] = groups
#     return g

# sku_assignment = (
#     sku_summary
#       .groupby(["Brand code", "sku_sales_category"], group_keys=False)
#       .apply(assign_groups_within_brand_and_sales_category)
#       .reset_index(drop=True)
# )

# # ----------------------------
# # 3) Merge assignment back to SKU-Nodes
# # ----------------------------
# dfTestSelectionGrouped = df.merge(
#     sku_assignment[["Product Code", "test_group"]],
#     on="Product Code",
#     how="left"
# )

# # ----------------------------
# # 4) Checks
# # ----------------------------
# print("SKU counts by group:")
# print(sku_assignment["test_group"].value_counts(dropna=False))

# print("\nSKU share by group:")
# print(sku_assignment["test_group"].value_counts(normalize=True, dropna=False))

# print("\nSplit of Brand code across groups:")
# print(
#     pd.crosstab(
#         sku_assignment["Brand code"],
#         sku_assignment["test_group"],
#         margins=True
#     )
# )

# print("\nRow-wise share of Brand code across groups:")
# print(
#     pd.crosstab(
#         sku_assignment["Brand code"],
#         sku_assignment["test_group"],
#         normalize="index"
#     )
# )

# print("\nSplit of sku_sales_category across groups:")
# print(
#     pd.crosstab(
#         [sku_assignment["Brand code"], sku_assignment["sku_sales_category"]],
#         sku_assignment["test_group"]
#     )
# )

# print("\nOther category balance check:")
# print(
#     pd.crosstab(
#         [
#             sku_assignment["Brand code"],
#             sku_assignment["Current walmart margin at NLC category"],
#             sku_assignment["current_nlc_margin category"]
#         ],
#         sku_assignment["test_group"]
#     )
# )

# # Optional revenue check
# sku_assignment = sku_assignment.merge(
#     dfSKURevenue[["Product Code", "sku_revenue"]],
#     how="left",
#     on="Product Code"
# )

# sku_assignment = sku_assignment.sort_values("sku_revenue", ascending=False)

# dfTestSelectionGrouped

In [119]:
# # sku_assignment group by test_group and get the counts for the different columns, e.g. sku_sales_category, Current walmart margin at NLC category, current_nlc_margin category, num_sku_nodes
# summary = (
#     sku_assignment.groupby("test_group")
#                .agg(
#                     total_revenue =("sku_revenue", "sum"),
#                    count_skus=("Product Code", "count"),
#                    avg_num_sku_nodes=("num_sku_nodes", "mean"),
#                    sales_category_dist=("sku_sales_category", lambda x: x.value_counts(normalize=True).to_dict()),
#                    walmart_margin_dist=("Current walmart margin at NLC category", lambda x: x.value_counts(normalize=True).to_dict()),
#                    nlc_margin_dist=("current_nlc_margin category", lambda x: x.value_counts(normalize=True).to_dict())
#                )
#                .reset_index()
# )

# summary 

In [120]:
# # dfTest = dfOutputAll.merge(finalGroups, how="left", on="Product Code")
# # dfTest = dfTest[(dfTest["Target for node level cost? - 15% margin"] == "Yes") &
# #                                       (dfTest["group"].notna()) &
# #                                       (dfTest["current_nlc_price"].notna())].copy()
# # dfTest
# # dfTest["Group"] = np.where(dfTest["group"] == 1, "11%",
# #                                       np.where(dfTest["group"] == 2, "13%",
# #                                               np.where(dfTest["group"] == 3, "15%", "No group")))
# # dfTest["Test price"]= np.where(dfTest["group"] == 1, dfTest["Node level cost - 11% margin"],
# #                                       np.where(dfTest["group"] == 2, dfTest["Node level cost - 13% margin"],
# #                                               np.where(dfTest["group"] == 3, dfTest["Node level cost - 15% margin"], np.nan)))


# dfTest = dfOutputTarget.merge(
#     sku_assignment[["Product Code", "test_group"]],
#     on="Product Code",
#     how="left"
# )
# dfTest["brand_code"] = dfTest["Product Code"].str[:4]

# dfTest = dfTest.merge(dfAssignments[["brand_code", "test_group","price_level_pct"]], how="left", on=["brand_code", "test_group"])

# dfTest['price_level_pct'] = dfTest['price_level_pct'].astype(str)

# dfTest["Test price"] =  np.where(dfTest['price_level_pct'] == "11", dfTest["Node level cost - 11% margin"],
#                                        np.where(dfTest['price_level_pct'] == "12", dfTest["Node level cost - 12% margin"],
#                                                np.where(dfTest['price_level_pct'] == "13", dfTest["Node level cost - 13% margin"], 
#                                                         np.where(dfTest['price_level_pct'] == "14", dfTest["Node level cost - 14% margin"], np.nan))))

# dfTest["Delta test to current NLC %"] = round((dfTest["Test price"] - dfTest["current_nlc_price"])/dfTest["current_nlc_price"],4)
# dfTest["Delta test to current NLC category"] = np.where(dfTest["Delta test to current NLC %"] < 0, "Decrease",
#                                                                 np.where(dfTest["Delta test to current NLC %"] > 0, "Increase", "No change"))

# # relevantCols = ["Product Code", "Identifier","SKU-Node", "price_level_pct", "Test price", "Delta test to current NLC %", "Delta test to current NLC category",
# #                     "Node type", "Min units"]
# # dfTest = dfTest[relevantCols].copy()



In [121]:

# summary = (
#     dfTest
#         .groupby(["price_level_pct"])
#         .agg(
#             count=("Delta test to current NLC category", "count"),
#             increases=("Delta test to current NLC category", lambda x: (x == "Increase").sum()),
#             decreases=("Delta test to current NLC category", lambda x: (x == "Decrease").sum()),
#             no_change=("Delta test to current NLC category", lambda x: (x == "No change").sum()),
#             avg_delta_test_to_current_nlc_pct=("Delta test to current NLC %", "mean"),
#         )
#         .reset_index()
# )

# # Now display # of increases, decreases, no change by each Group + average of "Delta test to current NLC %" by group
# summary_brand = (
#     dfTest
#         .groupby(["brand_code","price_level_pct"])
#         .agg(
#             count=("Delta test to current NLC category", "count"),
#             increases=("Delta test to current NLC category", lambda x: (x == "Increase").sum()),
#             decreases=("Delta test to current NLC category", lambda x: (x == "Decrease").sum()),
#             no_change=("Delta test to current NLC category", lambda x: (x == "No change").sum()),
#             avg_delta_test_to_current_nlc_pct=("Delta test to current NLC %", "mean"),
#         )
#         .reset_index()
# )
# display(summary)
# display(summary_brand)

In [122]:
# dfTestDSV= dfTest[["Product Code", "Identifier","Test price","SKU-Node"]].rename(columns = {"Product Code": "SKU", "Identifier": "Source", "Test price": "Price"})
# dfTestDSV

In [123]:
# dfTest

In [124]:
# dfCurrentTestsAll

In [125]:
# usecols = ["Product Code", "Identifier","price_level_pct", "Test price", "Delta test to current NLC category", "Delta test to current NLC %", "Node type", "Min units"]
# dfTestTracker = dfTest[usecols].copy()

# dfTestTracker["Final target"] = "Margin test"

# dfTestTracker = dfTestTracker.rename(columns={
#     "price_level_pct": "Final price category",
#     "Delta test to current NLC category": "Final price change category final",
#     "Delta test to current NLC %": "Final delta vs current %",
#     "Test price": "Final price"})
# dfTestTracker["SKU-Node"] = dfTestTracker["Product Code"] + "-" + dfTestTracker["Identifier"].astype(str)
# dfTestTracker["Min units"] = dfTestTracker["Min units"].astype(int)
# dfTestTracker["Start date"] = date_str
# dfTestTracker

In [126]:
# dfTestTracker["Final price category"].value_counts()

## Margin test by walmart margin

In [127]:
# dfOutputSalesAll = dfOutput.merge(dfSalesFiltered[["Product Code", "Sales category","revenue"]], how="left", on="Product Code")
# dfOutputSales = dfOutputSalesAll[dfOutputSalesAll["Sales category"].isin(["Top 20%", "Top 50%", "Top 80%"])].copy()
# dfOutputSalesTest = dfOutputSales[dfOutputSales["Current walmart margin at NLC"] > 0.13].copy()
# dfOutputSalesTest["Total margin"] = round(dfOutputSalesTest["Current walmart margin at NLC"] + dfOutputSalesTest["current_nlc_margin"],4)
# dfOutputSalesTest["New Walmart margin test"] = round(dfOutputSalesTest["Total margin"]/2,4)
# dfOutputSalesTest["New test price"] = round(dfOutputSalesTest["offer_price"] * (1 - dfOutputSalesTest["New Walmart margin test"]),2)
# dfOutputSalesTest["Price change with new test price %"] = round((dfOutputSalesTest["New test price"] - dfOutputSalesTest["current_nlc_price"])/ dfOutputSalesTest["current_nlc_price"], 3)

# dfOutputSalesTest["Price change with new test price % category"] = pd.cut(dfOutputSalesTest["Price change with new test price %"], bins=[0.01, 0.03,0.06,0.1, np.inf], labels=["1% to 3%", "3% to 6%", "6% to 10%", ">10%"])

# dfOutputSalesTest["Delta price change normal vs test"] = round(dfOutputSalesTest["Price change with new test price %"] - dfOutputSalesTest["Final price change %"], 3)

# dfOutputSalesTest["Delta price change normal vs test category"] = pd.cut(dfOutputSalesTest["Delta price change normal vs test"], bins=[0.01, 0.03,0.06,0.1, np.inf], labels=["1% to 3%", "3% to 6%", "6% to 10%", ">10%"])


# dfOutputSalesTest["New test price category"] = np.where(dfOutputSalesTest["Delta price change normal vs test"] < 0, "Decrease",
#                                                      np.where(dfOutputSalesTest["Delta price change normal vs test"] > 0, "Increase", "No change"))

# dfOutputSalesTest["New Walmart margin test 2"] = round(dfOutputSalesTest["Total margin"]*0.4,4)
# dfOutputSalesTest["New test price 2"] = round(dfOutputSalesTest["offer_price"] * (1 - dfOutputSalesTest["New Walmart margin test 2"]),2)
# dfOutputSalesTest["Price change with new test price % 2"] = round((dfOutputSalesTest["New test price 2"] - dfOutputSalesTest["current_nlc_price"])/ dfOutputSalesTest["current_nlc_price"], 3)



# dfTest = dfOutputSalesTest[(dfOutputSalesTest["Delta price change normal vs test"] > 0.01) &
#                            (dfOutputSalesTest["Delta price change normal vs test"] < 0.1) &
#                            (dfOutputSalesTest["Price change with new test price %"] > 0.01) &
#                            (dfOutputSalesTest["Delta price change normal vs test"] < 0.1) &
#                            (dfOutputSalesTest["current_nlc_margin"]> 0.08)].copy()


# dfTest = dfTest.merge(dfSalesSKUNodeGrouped, how="left", on=["SKU-Node"])

# print("Total SKU-Nodes above 13% wm margin:", len(dfTest))
# display(dfTest["Final target"].value_counts())
# display(dfTest["current_nlc_margin category"].value_counts())
# print("Mean current Tires Easy NLC (node level cost) margin:", dfTest["current_nlc_margin"].mean().round(4))
# print("Mean price change with new test price %:", dfTest["Price change with new test price %"].mean().round(4))
# print("Mean price change with new test price % 2:", dfTest["Price change with new test price % 2"].mean().round(4))

# print("Mean delta price change normal vs test:", dfTest["Delta price change normal vs test"].mean().round(4))
# print("Revenue contribution:", dfTest["total_inv_amount"].sum()/total_rev)
# display(dfTest["Final price change category"].value_counts())

In [128]:
# usefulCols = ["Product Code", "Identifier","SKU-Node", "total_inv_amount", "Sales category","Current walmart margin at NLC category","current_nlc_margin category",
#               "Price change with new test price % category","current_nlc_price","Final price change category","Final node level cost", "New test price", "New test price 2",
#               "Node type", "Min units"]
# dfTestSelection = dfTest[usefulCols].copy()
# dfTestSelection

In [129]:
# import pandas as pd
# import numpy as np

# # Copy
# df = dfTestSelection.copy()

# # ----------------------------
# # 1) Build SKU-level table
# # ----------------------------
# # Use Product Code as the SKU key
# # For stratification, keep the main categorical dimensions at SKU level.
# # Since a SKU can have multiple SKU-Nodes, we summarize numeric weight and
# # take the most representative category using weighted mode by inventory amount.

# def weighted_mode(x, weights):
#     tmp = pd.DataFrame({"x": x, "w": weights})
#     tmp = tmp.dropna(subset=["x"])
#     if len(tmp) == 0:
#         return np.nan
#     return tmp.groupby("x", dropna=False)["w"].sum().idxmax()

# sku_summary = (
#     df.groupby("Product Code", as_index=False)
#       .apply(lambda g: pd.Series({
#           "sku_weight": g["total_inv_amount"].sum(),
#           "Sales category": weighted_mode(g["Sales category"], g["total_inv_amount"]),
#           "Current walmart margin at NLC category": weighted_mode(g["Current walmart margin at NLC category"], g["total_inv_amount"]),
#           "current_nlc_margin category": weighted_mode(g["current_nlc_margin category"], g["total_inv_amount"]),
#           "Price change with new test price % category": weighted_mode(g["Price change with new test price % category"], g["total_inv_amount"]),
#           "num_sku_nodes": g["SKU-Node"].nunique()
#       }))
#       .reset_index(drop=True)
# )

# # ----------------------------
# # 2) Create stratification key
# # ----------------------------
# # This keeps balance across your important dimensions.
# strat_cols = [
#     "Sales category",
#     "Current walmart margin at NLC category",
#     "current_nlc_margin category",
#     "Price change with new test price % category"
# ]

# sku_summary["stratum"] = sku_summary[strat_cols].astype(str).agg(" | ".join, axis=1)

# # ----------------------------
# # 3) Assign groups within each stratum
# # ----------------------------
# # Group sizes:
# #   group 1 = 50%
# #   group 2 = 25%
# #   group 3 = 25%
# #
# # We randomize at SKU level, not SKU-Node level.

# rng = np.random.default_rng(42)

# def assign_groups_within_stratum(g):
#     g = g.sample(frac=1, random_state=42).copy()  # shuffle rows
    
#     n = len(g)
#     n_g1 = int(round(n * 0.50))
#     n_g2 = int(round(n * 0.25))
#     n_g3 = n - n_g1 - n_g2
    
#     groups = (
#         ["group_1"] * n_g1 +
#         ["group_2"] * n_g2 +
#         ["group_3"] * n_g3
#     )
    
#     # In case rounding creates mismatch
#     groups = groups[:n]
#     if len(groups) < n:
#         groups += ["group_3"] * (n - len(groups))
    
#     g["test_group"] = groups
#     return g

# sku_assignment = (
#     sku_summary.groupby("stratum", group_keys=False)
#                .apply(assign_groups_within_stratum)
#                .reset_index(drop=True)
# )

# # ----------------------------
# # 4) Merge assignment back to SKU-Nodes
# # ----------------------------
# dfTestSelectionGrouped = df.merge(
#     sku_assignment[["Product Code", "test_group"]],
#     on="Product Code",
#     how="left"
# )

# # ----------------------------
# # 5) Quick checks
# # ----------------------------
# print("SKU counts by group:")
# print(
#     sku_assignment["test_group"].value_counts(dropna=False)
# )

# print("\nSKU share by group:")
# print(
#     sku_assignment["test_group"].value_counts(normalize=True, dropna=False)
# )

# print("\nWeighted inventory share by group:")
# print(
#     sku_assignment.groupby("test_group")["sku_weight"].sum() /
#     sku_assignment["sku_weight"].sum()
# )

# print("\nSKU-Node counts by group:")
# print(
#     dfTestSelectionGrouped.groupby("test_group")["SKU-Node"].nunique()
# )

# dfTestSelectionGrouped.head()

In [130]:
# for col in strat_cols:
#     print(f"\nDistribution for {col}")
#     display(pd.crosstab(sku_assignment["test_group"], sku_assignment[col], normalize="index"))

In [131]:
# dfTestSelectionGrouped["Current walmart margin at NLC category"].value_counts()

In [132]:
# import numpy as np

# df = dfTestSelectionGrouped.copy()

# # ----------------------------
# # Sub-group labels
# # ----------------------------
# df["Sub-group"] = np.select(
#     [
#         df["test_group"] == "group_1",
#         df["test_group"] == "group_2",
#         df["test_group"] == "group_3",
#     ],
#     [
#         "Baseline",
#         "50% split",
#         "60% split",
#     ],
#     default=np.nan
# )

# # ----------------------------
# # Test price logic
# # group_1 -> baseline price only when baseline says Increase
# #            otherwise keep current price
# # group_2 -> New test price
# # group_3 -> New test price 2
# # ----------------------------
# df["Test price"] = np.select(
#     [
#         (df["test_group"] == "group_1") & (df["Final price change category"] == "Increase"),
#         (df["test_group"] == "group_1") & (df["Final price change category"] != "Increase"),
#         df["test_group"] == "group_2",
#         df["test_group"] == "group_3",
#     ],
#     [
#         df["Final node level cost"],   # baseline price
#         df["current_nlc_price"],       # keep current price when baseline is not Increase
#         df["New test price"],          # 50% split
#         df["New test price 2"],        # 60% split
#     ],
#     default=np.nan
# )

# # ----------------------------
# # Test price change %
# # vs current price
# # ----------------------------
# df["Test price change %"] = (df["Test price"] / df["current_nlc_price"]) - 1

# # Optional: percentage points / percent formatting helper
# df["Test price change % pct"] = df["Test price change %"] * 100

# # ----------------------------
# # Quick checks
# # ----------------------------
# print(df[["test_group", "Sub-group"]].drop_duplicates().sort_values("test_group"))

# print("\nCounts by Sub-group:")
# print(df["Sub-group"].value_counts(dropna=False))

# print("\nTest price summary by Sub-group:")
# print(
#     df.groupby("Sub-group")["Test price change %"]
#       .agg(["count", "mean", "median", "min", "max"])
#       .sort_index()
# )

# df

In [133]:
# df["Test price change % category"] =np.where(df["Test price change %"] < 0, "Decrease",
#                                                      np.where(df["Test price change %"] > 0, "Increase", "No change"))

# df["Test price change % category"].value_counts()

In [134]:
# #df by Sub-group I want split of "Test price change % category" within each Sub-group + average of "Test price change %"
# summary = (
#     df.groupby("Sub-group")
#       .agg(
#           count=("Test price change % category", "count"),
#           increases=("Test price change % category", lambda x: (x == "Increase").sum()),
#           decreases=("Test price change % category", lambda x: (x == "Decrease").sum()),
#           no_change=("Test price change % category", lambda x: (x == "No change").sum()),
#           avg_test_price_change_pct=("Test price change %", "mean")
#       )
#       .reset_index()
# )
# summary

In [135]:
# dfTest = df.copy()
# dfTest

In [136]:
# dfTestDSV= dfTest[["Product Code", "Identifier","Test price","SKU-Node"]].rename(columns = {"Product Code": "SKU", "Identifier": "Source", "Test price": "Price"})
# dfTestDSV

In [137]:
# usecols = ["Product Code", "Identifier","Sub-group", "Test price", "Final price change category", "Test price change %", "Node type", "Min units",
#            "Sales category"]
# dfTestTracker = dfTest[usecols].copy()

# dfTestTracker["Final target"] = "Wm margin split test"

# dfTestTracker = dfTestTracker.rename(columns={
#     "Group": "Final price category",
#     "Final price change category": "Final price change category final",
#     "Test price change %": "Final delta vs current %",
#     "Test price": "Final price"})
# dfTestTracker["SKU-Node"] = dfTestTracker["Product Code"] + "-" + dfTestTracker["Identifier"].astype(str)
# dfTestTracker["Min units"] = dfTestTracker["Min units"].astype(int)
# dfTestTracker["Start date"] = date_str
# dfTestTracker

In [138]:
# dfTestCheckMAP = dfTest.merge(dfMAP, how = "left", on = "Product Code")

In [139]:
# dfTestCheckMAP["New price above MAP"] = dfTestCheckMAP["Test price"] > dfTestCheckMAP["MAP"] 

# dfTestCheckMAP["New price +5% above MAP"] = dfTestCheckMAP["Test price"]*1.05 > dfTestCheckMAP["MAP"] 

# dfTestCheckMAP["New price +5% above MAP"].value_counts()

In [140]:
# dfTestCheckMAP[dfTestCheckMAP["MAP"].notna()]

## Cost + shipping

In [141]:
# test = True
# dfOutputCostNode = dfOutput[dfOutput["Identifier"].isin(dfCostNode["Identifier"].unique())].copy()
# dfOutputCostNode = dfOutputCostNode.merge(dfCostNode, how="left", on="Identifier")

# cost_column = "Final node level cost + shipping"

# dfOutputCostNode[cost_column] = dfOutputCostNode["Final node level cost"] + dfOutputCostNode["Shipping cost"]

# price_change_col = "Final price change %"
# price_change_cat_col = "Final price change category"

# dfOutputCostNode[price_change_col] = round((dfOutputCostNode[cost_column] - dfOutputCostNode["current_nlc_price"])/dfOutputCostNode["current_nlc_price"],4)
# dfOutputCostNode[price_change_cat_col] = np.where(dfOutputCostNode[price_change_col] < 0, "Decrease",
#                                                         np.where(dfOutputCostNode[price_change_col] > 0, "Increase", "No change"))
# dfOutputCostNode[price_change_col].mean()

# dfTest = dfOutputCostNode.copy()

In [142]:
# dfTestDSV= dfTest[["Product Code", "Identifier",cost_column,"SKU-Node"]].rename(columns = {"Product Code": "SKU", "Identifier": "Source", cost_column: "Price"})
# dfTestDSV

In [143]:
# usecols = ["Product Code", "Identifier",cost_column, price_change_cat_col, price_change_col, 
#            "Node type", "Min units","Final node level cost category"]
# dfTestTracker = dfTest[usecols].copy()

# dfTestTracker["Final target"] = "Shipping cost added"

# dfTestTracker = dfTestTracker.rename(columns={
#     "Final node level cost category":"Final price category",
#     price_change_cat_col: "Final price change category final",
#     price_change_col: "Final delta vs current %",
#     cost_column: "Final price"})
# dfTestTracker["SKU-Node"] = dfTestTracker["Product Code"] + "-" + dfTestTracker["Identifier"].astype(str)
# dfTestTracker["Min units"] = dfTestTracker["Min units"].astype(int)
# dfTestTracker["Start date"] = date_str
# dfTestTracker

# Regular updates

## Walmart margin split test update

In [155]:
dfWmMarginSplit = dfOutput[dfOutput["Final target"] == "Wm margin split test"].copy()
dfWmMarginSplit["Sub-group"].value_counts()
# dfWmMarginSplit create column Price where if Sub-group = 60% split then do Price margin split 60%, if Sub-group = 50% split then do Price margin split 50%, if Sub-group = Baseline then Price = Final node level cost when Final price change category = Increase, else Price = current_nlc_price
dfWmMarginSplit["Price"] = np.where(dfWmMarginSplit["Sub-group"] == "60% split", dfWmMarginSplit["Price margin split 60%"],
                                    np.where(dfWmMarginSplit["Sub-group"] == "50% split", dfWmMarginSplit["Price margin split 50%"],
                                             np.where((dfWmMarginSplit["Sub-group"] == "Baseline") & (dfWmMarginSplit["Final price change category"] == "Increase"), dfWmMarginSplit["Final node level cost"],
                                                      dfWmMarginSplit["current_nlc_price"])))

dfWmMarginSplit["Price change %"] = round((dfWmMarginSplit["Price"] - dfWmMarginSplit["current_nlc_price"])/dfWmMarginSplit["current_nlc_price"],4)
dfWmMarginSplit["Price change category"] = np.where(dfWmMarginSplit["Price change %"] < 0, "Decrease",
                                                     np.where(dfWmMarginSplit["Price change %"] > 0, "Increase", "No change"))

# dfWmMarginSplitDSVUpdate only where Price change % in aboslute is greater than 1%
dfWmMarginSplitUpdate = dfWmMarginSplit[abs(dfWmMarginSplit["Price change %"]) >= 0.01].copy()

print("Total SKU-Nodes to update in DSV:", len(dfWmMarginSplitUpdate))

#But do avg price change % only for the ones where Price change category is Increase and the same for decreases
summary = (
    dfWmMarginSplitUpdate.groupby("Sub-group")
                   .agg(
                       count=("Price change category", "count"),
                       increases=("Price change category", lambda x: (x == "Increase").sum()),
                       decreases=("Price change category", lambda x: (x == "Decrease").sum()),
                       no_change=("Price change category", lambda x: (x == "No change").sum()),
                       avg_price_change_pct_increase=("Price change %", lambda x: x[dfWmMarginSplit["Price change category"] == "Increase"].mean()),
                       avg_price_change_pct_decrease=("Price change %", lambda x: x[dfWmMarginSplit["Price change category"] == "Decrease"].mean())
                   )
                   .reset_index()
)
summary

Total SKU-Nodes to update in DSV: 2019


,Sub-group,count,increases,decreases,no_change,avg_price_change_pct_increase,avg_price_change_pct_decrease
0,50% split,523,474,49,0,0.037233,-0.024463
1,60% split,511,473,38,0,0.043330,-0.028600
2,Baseline,985,985,0,0,0.057543,NaN


In [156]:
dfWmMarginSplitDSV = dfWmMarginSplitUpdate[["Product Code", "Identifier","Price","SKU-Node"]].rename(columns = {"Product Code": "SKU", "Identifier": "Source", "Price": "Price"})
dfWmMarginSplitDSV

,SKU,Source,Price,SKU-Node
24810,ARRO0011721555WXL,6283261553,60.60,ARRO0011721555WXL-6283261553
24821,ARRO0011721555WXL,628326799,60.29,ARRO0011721555WXL-628326799
25374,ARRO0011824555W,628326996,102.66,ARRO0011824555W-628326996
26852,ARRO0012227550HXL,6283261553,128.99,ARRO0012227550HXL-6283261553
28054,ARRO0071721565H,6283261553,74.90,ARRO0071721565H-6283261553
...,...,...,...,...
2237098,TOYO3711726570T,628326896,234.31,TOYO3711726570T-628326896
2237381,TOYO3711728570Q,6283261005,268.86,TOYO3711728570Q-6283261005
2237397,TOYO3711728570Q,6283261319,268.86,TOYO3711728570Q-6283261319
2237402,TOYO3711728570Q,6283261414,268.86,TOYO3711728570Q-6283261414


In [157]:
dfWmMarginSplitTracker = dfWmMarginSplitUpdate[["SKU-Node"]].copy()
dfWmMarginSplitTracker = dfWmMarginSplitTracker.merge(dfCurrentTestsAll, on="SKU-Node", how="left")
dfWmMarginSplitTracker["Last price update"] = today_str
dfWmMarginSplitTracker

,SKU-Node,Product Code,Identifier,Final target,Final price category,Final price,Final delta vs current %,Final price change category final,Category inventory,Node type,Start date,Min units,Sub-group,Is in stock?,current_nlc_margin,Current walmart margin at NLC category,sku_sales_category,current_nlc_margin_date,Last price update
0,ARRO0011721555WXL-6283261553,ARRO0011721555WXL,6283261553,Wm margin split test,NaN,59.85,0.077214,Increase,NaN,HUB,2026-03-09,8,50% split,Yes,0.1546,15% to 20%,10-20%,2026-03-17,2026-03-18
1,ARRO0011721555WXL-628326799,ARRO0011721555WXL,628326799,Wm margin split test,NaN,61.35,0.010708,Decrease,NaN,HUB,2026-03-09,8,50% split,Yes,0.1394,10% to 15%,10-20%,2026-03-17,2026-03-18
2,ARRO0011824555W-628326996,ARRO0011824555W,628326996,Wm margin split test,NaN,102.69,0.026900,Decrease,NaN,HUB,2026-03-09,8,50% split,Yes,0.1431,15% to 20%,50-60%,2026-03-17,2026-03-18
3,ARRO0012227550HXL-6283261553,ARRO0012227550HXL,6283261553,Wm margin split test,NaN,127.30,0.000000,No change,NaN,HUB,2026-03-09,8,Baseline,Yes,0.1143,15% to 20%,50-60%,2026-03-17,2026-03-18
4,ARRO0071721565H-6283261553,ARRO0071721565H,6283261553,Wm margin split test,NaN,71.47,0.061015,No change,NaN,HUB,2026-03-09,8,60% split,Yes,0.1766,20% to 25%,50-60%,2026-03-17,2026-03-18
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2014,TOYO3711726570T-628326896,TOYO3711726570T,628326896,Wm margin split test,NaN,231.42,0.055941,Decrease,NaN,HUB,2026-03-09,4,60% split,Yes,0.1747,10% to 15%,70-80%,2026-03-17,2026-03-18
2015,TOYO3711728570Q-6283261005,TOYO3711728570Q,6283261005,Wm margin split test,NaN,264.87,0.103579,Increase,NaN,SPOKE,2026-03-09,8,60% split,Yes,0.1874,10% to 15%,50-60%,2026-03-17,2026-03-18
2016,TOYO3711728570Q-6283261319,TOYO3711728570Q,6283261319,Wm margin split test,NaN,264.75,0.092203,Decrease,NaN,HUB,2026-03-09,8,60% split,Yes,0.1870,10% to 15%,50-60%,2026-03-17,2026-03-18
2017,TOYO3711728570Q-6283261414,TOYO3711728570Q,6283261414,Wm margin split test,NaN,264.75,0.092203,Decrease,NaN,HUB,2026-03-09,8,60% split,Yes,0.1870,10% to 15%,50-60%,2026-03-17,2026-03-18


## Brands margin test update

In [158]:
dfMarginTest = dfOutput[(dfOutput["Final target"] == "Margin test") & (dfOutput["Start date"].isin(["2026-03-12"]))].copy()

# dfMarginTest create column Price where if Sub-group = 11% then do Node level cost - 11% margin, if Sub-group = 12% then do Node level cost - 12% margin, if Sub-group = 13% then do Node level cost - 13% margin, if Sub-group = 14% then do Node level cost - 14% margin, else do current_nlc_price , if 15%  then do Node level cost - 15% margin
dfMarginTest["Price"] = np.where(dfMarginTest["Sub-group"] == "11%", dfMarginTest["Node level cost - 11% margin"],
                                 np.where(dfMarginTest["Sub-group"] == "12%", dfMarginTest["Node level cost - 12% margin"],
                                         np.where(dfMarginTest["Sub-group"] == "13%", dfMarginTest["Node level cost - 13% margin"],
                                                 np.where(dfMarginTest["Sub-group"] == "14%", dfMarginTest["Node level cost - 14% margin"],
                                                         np.where(dfMarginTest["Sub-group"] == "15%", dfMarginTest["Node level cost - 15% margin"], dfMarginTest["current_nlc_price"])))))


dfMarginTest["Price change %"] = round((dfMarginTest["Price"] - dfMarginTest["current_nlc_price"])/dfMarginTest["current_nlc_price"],4)
dfMarginTest["Price change category"] = np.where(dfMarginTest["Price change %"] < 0, "Decrease",
                                                     np.where(dfMarginTest["Price change %"] > 0, "Increase", "No change"))

# dfWmMarginSplitDSVUpdate only where Price change % in aboslute is greater than 1%
dfMarginTestUpdate = dfMarginTest[abs(dfMarginTest["Price change %"]) >= 0.01].copy()

print("Total SKU-Nodes to update in DSV:", len(dfMarginTestUpdate))

#But do avg price change % only for the ones where Price change category is Increase and the same for decreases
summary = (
    dfMarginTestUpdate.groupby("Sub-group")
                   .agg(
                       count=("Price change category", "count"),
                       increases=("Price change category", lambda x: (x == "Increase").sum()),
                       decreases=("Price change category", lambda x: (x == "Decrease").sum()),
                       no_change=("Price change category", lambda x: (x == "No change").sum()),
                       avg_price_change_pct_increase=("Price change %", lambda x: x[dfMarginTestUpdate["Price change category"] == "Increase"].mean()),
                       avg_price_change_pct_decrease=("Price change %", lambda x: x[dfMarginTestUpdate["Price change category"] == "Decrease"].mean())
                   )
                   .reset_index()
)
summary

Total SKU-Nodes to update in DSV: 7656


,Sub-group,count,increases,decreases,no_change,avg_price_change_pct_increase,avg_price_change_pct_decrease
0,11%,3223,2838,385,0,0.073429,-0.051572
1,13%,2587,2292,295,0,0.070199,-0.047074
2,14%,1846,1589,257,0,0.065151,-0.049384


In [159]:
dfMarginTestDSV = dfMarginTestUpdate[["Product Code", "Identifier","Price","SKU-Node"]].rename(columns = {"Product Code": "SKU", "Identifier": "Source", "Price": "Price"})
dfMarginTestDSV

,SKU,Source,Price,SKU-Node
333,ACCE0221520565V,628326984,59.76,ACCE0221520565V-628326984
397,ACCE0221521570H,628326984,67.67,ACCE0221521570H-628326984
552,ACCE0221622560WXL,628326984,72.28,ACCE0221622560WXL-628326984
610,ACCE0261625570S,628326984,102.76,ACCE0261625570S-628326984
825,ACCE0321821555VXL,628326984,84.09,ACCE0321821555VXL-628326984
...,...,...,...,...
2076105,NEXE1631722555WXL,6283261419,135.36,NEXE1631722555WXL-6283261419
2076107,NEXE1631722555WXL,6283261448,135.36,NEXE1631722555WXL-6283261448
2076128,NEXE1631722555WXL,62832681,135.36,NEXE1631722555WXL-62832681
2078377,NEXE1632025555VXL,6283261417,186.39,NEXE1632025555VXL-6283261417


In [160]:
dfMarginTestTracker = dfMarginTestUpdate[["SKU-Node"]].copy()
dfMarginTestTracker = dfMarginTestTracker.merge(dfCurrentTestsAll, on="SKU-Node", how="left")
dfMarginTestTracker["Last price update"] = today_str
dfMarginTestTracker

,SKU-Node,Product Code,Identifier,Final target,Final price category,Final price,Final delta vs current %,Final price change category final,Category inventory,Node type,Start date,Min units,Sub-group,Is in stock?,current_nlc_margin,Current walmart margin at NLC category,sku_sales_category,current_nlc_margin_date,Last price update
0,ACCE0221520565V-628326984,ACCE0221520565V,628326984,Margin test,11%,60.89,0.0018,Increase,NaN,HUB,2026-03-12,8,11%,Yes,0.11,10% to 15%,80-90%,2026-03-17,2026-03-18
1,ACCE0221521570H-628326984,ACCE0221521570H,628326984,Margin test,11%,68.80,-0.0061,Decrease,NaN,HUB,2026-03-12,8,11%,Yes,0.11,0% to 10%,80-90%,2026-03-17,2026-03-18
2,ACCE0221622560WXL-628326984,ACCE0221622560WXL,628326984,Margin test,14%,73.44,0.0153,Increase,NaN,HUB,2026-03-12,8,14%,Yes,0.14,0% to 10%,60-70%,2026-03-17,2026-03-18
3,ACCE0261625570S-628326984,ACCE0261625570S,628326984,Margin test,11%,103.89,-0.0289,Decrease,NaN,HUB,2026-03-12,8,11%,Yes,0.11,15% to 20%,70-80%,2026-03-17,2026-03-18
4,ACCE0321821555VXL-628326984,ACCE0321821555VXL,628326984,Margin test,14%,85.26,0.0000,No change,NaN,HUB,2026-03-12,8,14%,Yes,0.14,10% to 15%,70-80%,2026-03-17,2026-03-18
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7651,NEXE1631722555WXL-6283261419,NEXE1631722555WXL,6283261419,Margin test,14%,110.13,0.0349,Increase,NaN,SPOKE,2026-03-12,4,14%,Yes,0.14,25% to 30%,90-95%,2026-03-17,2026-03-18
7652,NEXE1631722555WXL-6283261448,NEXE1631722555WXL,6283261448,Margin test,14%,110.13,0.0349,Increase,NaN,SPOKE,2026-03-12,4,14%,Yes,0.14,25% to 30%,90-95%,2026-03-17,2026-03-18
7653,NEXE1631722555WXL-62832681,NEXE1631722555WXL,62832681,Margin test,14%,110.13,0.0349,Increase,NaN,HUB,2026-03-12,4,14%,Yes,0.14,25% to 30%,90-95%,2026-03-17,2026-03-18
7654,NEXE1632025555VXL-6283261417,NEXE1632025555VXL,6283261417,Margin test,11%,150.02,0.0000,No change,NaN,SPOKE,2026-03-12,4,11%,Yes,0.11,20% to 25%,80-90%,2026-03-17,2026-03-18


## Low price updates (to show inv/correct our profit.)

In [169]:
dfCurrDSVNLCCheck = dfOutput[["Product Code", "Identifier","Purchase Price+FET","Node level cost - 6% margin", "Final target",
                                                 "Final node level cost","Final node level cost category","Node type", "Min units","current_nlc_margin"]].copy()

dont_update_final_targets = ["Margin test", "Wm margin split test", "Shipping cost added"]

dfCurrDSVNLCCPriceUpdates = dfCurrDSVNLCCheck[(dfCurrDSVNLCCheck["current_nlc_margin"]< min_margin_update_prices)
                                              & (~dfCurrDSVNLCCheck["Final target"].isin(dont_update_final_targets))].copy()


dfCurrDSVNLCCPriceUpdates["Category inventory"] = np.where(dfCurrDSVNLCCPriceUpdates["current_nlc_margin"]< 0.04, "Not showing inventory", "Below 6% margin")
dfCurrDSVNLCCPriceUpdates["SKU-Node"] = dfCurrDSVNLCCPriceUpdates["Product Code"] + "-" + dfCurrDSVNLCCPriceUpdates["Identifier"].astype(str)


if test:
    dfCurrDSVNLCCPriceUpdates = dfCurrDSVNLCCPriceUpdates[dfCurrDSVNLCCPriceUpdates["SKU-Node"].isin(dfTest["SKU-Node"]) == False].copy()
# dfCurrDSVNLCCPriceUpdates = dfCurrDSVNLCCPriceUpdates[dfCurrDSVNLCCPriceUpdates["SKU-Node"].isin(dfTrackerLessSalesChng["SKU-Node"]) == False].copy()


# dfCurrDSVNLCCPriceUpdates = dfCurrDSVNLCCPriceUpdates[~dfCurrDSVNLCCPriceUpdates["Product Code"].str.contains("MICH|BFGO")].copy()


dfCurrDSVNLCCPriceUpdates

,Product Code,Identifier,Purchase Price+FET,Node level cost - 6% margin,Final target,Final node level cost,Final node level cost category,Node type,Min units,current_nlc_margin,Category inventory,SKU-Node
625,ACCE0261628575E,628326984,134.17,142.73,Added,145.84,8%,HUB,8,0.0547,Below 6% margin,ACCE0261628575E-628326984
3314,ACCE0581822560HXL,628326984,96.88,103.06,Added,103.06,6%,HUB,8,0.0539,Below 6% margin,ACCE0581822560HXL-628326984
18636,APLS0061924555WXL,628326959,85.96,91.45,Added,91.45,6%,HUB,8,0.0576,Below 6% margin,APLS0061924555WXL-628326959
25394,ARRO0011825535WXL,628326993,77.00,81.91,Less sales,86.52,11%,HUB,8,0.0554,Below 6% margin,ARRO0011825535WXL-628326993
28811,ARRO0101824560H,628326855,74.07,78.80,Normal strategy,83.22,11%,HUB,8,0.0222,Not showing inventory,ARRO0101824560H-628326855
...,...,...,...,...,...,...,...,...,...,...,...,...
2260582,WEST0911723560H,628326791,82.68,87.96,Normal strategy,92.90,11%,HUB,4,-0.0331,Not showing inventory,WEST0911723560H-628326791
2260878,WEST1031628575E,628326558,136.12,144.81,Added,152.94,11%,HUB,4,0.0579,Below 6% margin,WEST1031628575E-628326558
2264863,YOKO0261822540YXL,6283261134,157.42,174.47,Updated,174.47,6%,HUB,4,0.0576,Below 6% margin,YOKO0261822540YXL-6283261134
2265889,YOKO0262026535YXL,6283261405,253.64,269.83,Added,269.83,6%,HUB,4,0.0362,Not showing inventory,YOKO0262026535YXL-6283261405


In [170]:
usecols = ["Product Code", "Identifier", "Final node level cost", "Final node level cost category", "Category inventory", "Node type", "Min units"]
dfCurrDSVNLCCPriceUpdatesforTracker = dfCurrDSVNLCCPriceUpdates[usecols].copy()
dfCurrDSVNLCCPriceUpdatesforTracker["Final target"] = "Updated"

dfCurrDSVNLCCPriceUpdatesforTracker = dfCurrDSVNLCCPriceUpdatesforTracker.rename(columns={
    "Final node level cost category": "Final price category",
    "Final node level cost": "Final price"})
dfCurrDSVNLCCPriceUpdatesforTracker["SKU-Node"] = dfCurrDSVNLCCPriceUpdatesforTracker["Product Code"] + "-" + dfCurrDSVNLCCPriceUpdatesforTracker["Identifier"].astype(str)
dfCurrDSVNLCCPriceUpdatesforTracker["Min units"] = dfCurrDSVNLCCPriceUpdatesforTracker["Min units"].astype(int)
dfCurrDSVNLCCPriceUpdatesforTracker["Start date"] = date_str
dfCurrDSVNLCCPriceUpdatesforTracker

,Product Code,Identifier,Final price,Final price category,Category inventory,Node type,Min units,Final target,SKU-Node,Start date
625,ACCE0261628575E,628326984,145.84,8%,Below 6% margin,HUB,8,Updated,ACCE0261628575E-628326984,2026-03-18
3314,ACCE0581822560HXL,628326984,103.06,6%,Below 6% margin,HUB,8,Updated,ACCE0581822560HXL-628326984,2026-03-18
18636,APLS0061924555WXL,628326959,91.45,6%,Below 6% margin,HUB,8,Updated,APLS0061924555WXL-628326959,2026-03-18
25394,ARRO0011825535WXL,628326993,86.52,11%,Below 6% margin,HUB,8,Updated,ARRO0011825535WXL-628326993,2026-03-18
28811,ARRO0101824560H,628326855,83.22,11%,Not showing inventory,HUB,8,Updated,ARRO0101824560H-628326855,2026-03-18
...,...,...,...,...,...,...,...,...,...,...
2260582,WEST0911723560H,628326791,92.90,11%,Not showing inventory,HUB,4,Updated,WEST0911723560H-628326791,2026-03-18
2260878,WEST1031628575E,628326558,152.94,11%,Below 6% margin,HUB,4,Updated,WEST1031628575E-628326558,2026-03-18
2264863,YOKO0261822540YXL,6283261134,174.47,6%,Below 6% margin,HUB,4,Updated,YOKO0261822540YXL-6283261134,2026-03-18
2265889,YOKO0262026535YXL,6283261405,269.83,6%,Not showing inventory,HUB,4,Updated,YOKO0262026535YXL-6283261405,2026-03-18


In [171]:
dfCurrDSVNLCCPriceUpdates["Brand code"] = dfCurrDSVNLCCPriceUpdates["Product Code"].str[:4]
dfCurrDSVNLCCPriceUpdates["Brand code"].value_counts()

Brand code
GOYR    1691
KUMH    1249
MICK    1118
NITT    1027
BRID     973
        ... 
APLS       1
MAXX       1
PRED       1
DORA       1
ROYA       1
Name: count, Length: 69, dtype: int64

In [172]:
dfCurrDSVNLCCPriceUpdates["Min units"].value_counts()

Min units
8    9653
4    3407
Name: count, dtype: int64

In [173]:
dfNLCPriceUpdatesFinal = dfCurrDSVNLCCPriceUpdates[["Product Code", "Identifier", "Final node level cost"]].rename(
                        columns={"Identifier": "Source","Final node level cost":"Price","Product Code":"SKU"})

dfNLCPriceUpdatesFinal["SKU-Node"] = dfNLCPriceUpdatesFinal["SKU"] + "-" + dfNLCPriceUpdatesFinal["Source"]
dfNLCPriceUpdatesFinal

,SKU,Source,Price,SKU-Node
625,ACCE0261628575E,628326984,145.84,ACCE0261628575E-628326984
3314,ACCE0581822560HXL,628326984,103.06,ACCE0581822560HXL-628326984
18636,APLS0061924555WXL,628326959,91.45,APLS0061924555WXL-628326959
25394,ARRO0011825535WXL,628326993,86.52,ARRO0011825535WXL-628326993
28811,ARRO0101824560H,628326855,83.22,ARRO0101824560H-628326855
...,...,...,...,...
2260582,WEST0911723560H,628326791,92.90,WEST0911723560H-628326791
2260878,WEST1031628575E,628326558,152.94,WEST1031628575E-628326558
2264863,YOKO0261822540YXL,6283261134,174.47,YOKO0261822540YXL-6283261134
2265889,YOKO0262026535YXL,6283261405,269.83,YOKO0262026535YXL-6283261405


In [174]:
dfCurrentTestsToUpdate = dfCurrentTests[dfCurrentTests["SKU-Node"].isin(dfNLCPriceUpdatesFinal["SKU-Node"])].copy()
dfCurrentTestsToUpdate["Final target"].value_counts()

Final target
Added                       6167
Decreased - margin > 20%    1806
Normal strategy             1768
Updates test                1152
Updated                     1054
No sales test               1025
Less sales                    49
Less sales back               39
Name: count, dtype: int64

In [175]:
# dfCurrentTestsToUpdate["Brand code"] = dfCurrentTestsToUpdate["SKU-Node"].str[:4]
# dfCurrentTestsToUpdate[dfCurrentTestsToUpdate["Final target"] == "Margin test"]#["Brand code"].value_counts()

## High price updates

In [176]:
dfCurrDSVNLCCheck = dfOutput[["Product Code", "Identifier","Purchase Price+FET","Node level cost - 20% margin","Node type", "Min units",
                              "current_nlc_margin","current_nlc_price","Final target","Target for node level cost? - 20% margin","Sub-group"]].copy()
dfCurrDSVNLCCPriceUpdatesDecr= dfCurrDSVNLCCheck[dfCurrDSVNLCCheck["current_nlc_margin"]> max_margin_update_prices].copy()
dfCurrDSVNLCCPriceUpdatesDecr["Category inventory"] = ""

dont_update_final_targets = ["Margin test", "Wm margin split test", "Shipping cost added"]

dfCurrDSVNLCCPriceUpdatesDecr["SKU-Node"] = dfCurrDSVNLCCPriceUpdatesDecr["Product Code"] + "-" + dfCurrDSVNLCCPriceUpdatesDecr["Identifier"].astype(str)


dfCurrDSVNLCCPriceUpdatesDecr =dfCurrDSVNLCCPriceUpdatesDecr[~dfCurrDSVNLCCPriceUpdatesDecr["Final target"].isin(dont_update_final_targets)].copy()

if test:
    dfCurrDSVNLCCPriceUpdatesDecr = dfCurrDSVNLCCPriceUpdatesDecr[dfCurrDSVNLCCPriceUpdatesDecr["SKU-Node"].isin(dfTest["SKU-Node"]) == False].copy()

# dfCurrDSVNLCCPriceUpdates = dfCurrDSVNLCCPriceUpdates[dfCurrDSVNLCCPriceUpdates["SKU-Node"].isin(dfTrackerLessSalesChng["SKU-Node"]) == False].copy()
# dfCurrDSVNLCCPriceUpdates = dfCurrDSVNLCCPriceUpdates[~dfCurrDSVNLCCPriceUpdates["Product Code"].str.contains("MICH|BFGO")].copy()

dfCurrDSVNLCCPriceUpdatesDecr["Final delta vs current %"] = round((dfCurrDSVNLCCPriceUpdatesDecr["Node level cost - 20% margin"] - dfCurrDSVNLCCPriceUpdatesDecr["current_nlc_price"])/dfCurrDSVNLCCPriceUpdatesDecr["current_nlc_price"],4)

dfCurrDSVNLCCPriceUpdatesDecr["Final price change category final"] = np.where(dfCurrDSVNLCCPriceUpdatesDecr["Final delta vs current %"] < 0, "Decrease",
                                                                np.where(dfCurrDSVNLCCPriceUpdatesDecr["Final delta vs current %"] > 0, "Increase", "No change"))
dfCurrDSVNLCCPriceUpdatesDecr

,Product Code,Identifier,Purchase Price+FET,Node level cost - 20% margin,Node type,Min units,current_nlc_margin,current_nlc_price,Final target,Target for node level cost? - 20% margin,Sub-group,Category inventory,SKU-Node,Final delta vs current %,Final price change category final
419,ACCE0221522560V,628326984,60.41901,75.52,HUB,8,0.2577,81.39,Updated,Yes,8%,,ACCE0221522560V-628326984,-0.0721,Decrease
1800,ACCE0322231530WXL,628326984,147.85000,184.81,HUB,8,0.2054,186.06,Decreased - margin > 20%,Yes,20%,,ACCE0322231530WXL-628326984,-0.0067,Decrease
2180,ACCE0342025540YXL,628326984,92.46000,115.57,HUB,8,0.2037,116.11,Added,Yes,11%,,ACCE0342025540YXL-628326984,-0.0047,Decrease
4596,ADVA08822529575G,628326959,230.38000,287.97,HUB,8,0.2079,290.85,Decreased - margin > 20%,Yes,20%,,ADVA08822529575G-628326959,-0.0099,Decrease
4657,ADVA21722511H,628326959,241.63000,302.04,HUB,8,0.2160,308.22,Decreased - margin > 20%,Yes,20%,,ADVA21722511H-628326959,-0.0201,Decrease
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2290646,YOKO3961728570T,628326916,203.51000,254.39,HUB,4,0.2245,262.43,Added,Yes,11%,,YOKO3961728570T-628326916,-0.0306,Decrease
2290751,YOKO3961729570E,628326916,242.58000,303.23,HUB,4,0.2173,309.92,Added,Yes,11%,,YOKO3961729570E-628326916,-0.0216,Decrease
2290919,YOKO3961823560HXL,628326916,181.52000,226.90,HUB,4,0.2148,231.19,Added,Yes,11%,,YOKO3961823560HXL-628326916,-0.0186,Decrease
2291185,YOKO3961826565H,628326916,198.02000,247.52,HUB,4,0.2149,252.21,Added,Yes,11%,,YOKO3961826565H-628326916,-0.0186,Decrease


In [177]:
usecols = ["Product Code", "Identifier", "Node level cost - 20% margin", "Category inventory", "Node type", "Min units",
           "Final delta vs current %", "Final price change category final","Final target","SKU-Node","Sub-group"]
dfUpdateAboveMarginTracker = dfCurrDSVNLCCPriceUpdatesDecr[usecols].copy()

# where Final target = Wm margin split test, then leave it as is, otherwise put "Updated - above max margin" 
dfUpdateAboveMarginTracker["Final target"] = np.where(dfUpdateAboveMarginTracker["Final target"] == "Wm margin split test", "Wm margin split test", 
                                                      np.where(dfUpdateAboveMarginTracker["Final target"] == "Margin test","Margin test",
                                                               "Decreased - margin > 20%"))


dfUpdateAboveMarginTracker = dfUpdateAboveMarginTracker.rename(columns={
    "Final node level cost category": "Final price category",
    "Node level cost - 20% margin": "Final price"})

dfUpdateAboveMarginTracker["Final price category"] = "20%"

dfUpdateAboveMarginTracker["SKU-Node"] = dfUpdateAboveMarginTracker["Product Code"] + "-" + dfUpdateAboveMarginTracker["Identifier"].astype(str)
dfUpdateAboveMarginTracker["Min units"] = dfUpdateAboveMarginTracker["Min units"].astype(int)
dfUpdateAboveMarginTracker["Start date"] = date_str
dfUpdateAboveMarginTracker

,Product Code,Identifier,Final price,Category inventory,Node type,Min units,Final delta vs current %,Final price change category final,Final target,SKU-Node,Sub-group,Final price category,Start date
419,ACCE0221522560V,628326984,75.52,,HUB,8,-0.0721,Decrease,Decreased - margin > 20%,ACCE0221522560V-628326984,8%,20%,2026-03-18
1800,ACCE0322231530WXL,628326984,184.81,,HUB,8,-0.0067,Decrease,Decreased - margin > 20%,ACCE0322231530WXL-628326984,20%,20%,2026-03-18
2180,ACCE0342025540YXL,628326984,115.57,,HUB,8,-0.0047,Decrease,Decreased - margin > 20%,ACCE0342025540YXL-628326984,11%,20%,2026-03-18
4596,ADVA08822529575G,628326959,287.97,,HUB,8,-0.0099,Decrease,Decreased - margin > 20%,ADVA08822529575G-628326959,20%,20%,2026-03-18
4657,ADVA21722511H,628326959,302.04,,HUB,8,-0.0201,Decrease,Decreased - margin > 20%,ADVA21722511H-628326959,20%,20%,2026-03-18
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2290646,YOKO3961728570T,628326916,254.39,,HUB,4,-0.0306,Decrease,Decreased - margin > 20%,YOKO3961728570T-628326916,11%,20%,2026-03-18
2290751,YOKO3961729570E,628326916,303.23,,HUB,4,-0.0216,Decrease,Decreased - margin > 20%,YOKO3961729570E-628326916,11%,20%,2026-03-18
2290919,YOKO3961823560HXL,628326916,226.90,,HUB,4,-0.0186,Decrease,Decreased - margin > 20%,YOKO3961823560HXL-628326916,11%,20%,2026-03-18
2291185,YOKO3961826565H,628326916,247.52,,HUB,4,-0.0186,Decrease,Decreased - margin > 20%,YOKO3961826565H-628326916,11%,20%,2026-03-18


In [178]:
# dfUpdateAboveMarginTracker["Final target"].value_counts() also want the average of "Final delta vs current %" by "Final target"
summary = (
    dfUpdateAboveMarginTracker.groupby("Final target")
    .agg(
        count=("Final target", "count"),
        avg_final_delta_vs_current_pct=("Final delta vs current %", "mean")
    )
    .reset_index()
)
summary

,Final target,count,avg_final_delta_vs_current_pct
0,Decreased - margin > 20%,1831,-0.043882


In [179]:
dfNLCPriceDecreasesDSV = dfCurrDSVNLCCPriceUpdatesDecr[["Product Code", "Identifier", "Node level cost - 20% margin"]].rename(
                        columns={"Identifier": "Source","Node level cost - 20% margin":"Price","Product Code":"SKU"})

dfNLCPriceDecreasesDSV["SKU-Node"] = dfNLCPriceDecreasesDSV["SKU"] + "-" + dfNLCPriceDecreasesDSV["Source"]
dfNLCPriceDecreasesDSV

,SKU,Source,Price,SKU-Node
419,ACCE0221522560V,628326984,75.52,ACCE0221522560V-628326984
1800,ACCE0322231530WXL,628326984,184.81,ACCE0322231530WXL-628326984
2180,ACCE0342025540YXL,628326984,115.57,ACCE0342025540YXL-628326984
4596,ADVA08822529575G,628326959,287.97,ADVA08822529575G-628326959
4657,ADVA21722511H,628326959,302.04,ADVA21722511H-628326959
...,...,...,...,...
2290646,YOKO3961728570T,628326916,254.39,YOKO3961728570T-628326916
2290751,YOKO3961729570E,628326916,303.23,YOKO3961729570E-628326916
2290919,YOKO3961823560HXL,628326916,226.90,YOKO3961823560HXL-628326916
2291185,YOKO3961826565H,628326916,247.52,YOKO3961826565H-628326916


In [180]:
check = dfNLCPriceDecreasesDSV.copy()
check["Brand code"] = check["SKU"].str[:4]
check["Brand code"].value_counts()

Brand code
TOYO    397
FALK    245
KUMH    140
GOYR    128
NITT    125
       ... 
RADA      1
DCEN      1
DELI      1
NATI      1
ARMS      1
Name: count, Length: 73, dtype: int64

In [181]:
dfCurrDSVNLCCPriceUpdatesDecr["Final target"].value_counts()

Final target
Updated                     534
Added                       514
Normal strategy             351
Decreased - margin > 20%    243
No sales test               132
Updates test                 49
Less sales back               4
Less sales                    4
Name: count, dtype: int64

In [182]:
check = dfCurrDSVNLCCPriceUpdatesDecr[dfCurrDSVNLCCPriceUpdatesDecr["Final target"]== "Margin test"]
tests_check = dfCurrentTestsAll.copy()
tests_check["SKU-Node"] = tests_check["Product Code"] + "-" + tests_check["Identifier"].astype(str)
tests_check[tests_check["SKU-Node"].isin(check["SKU-Node"])]["Start date"].value_counts()

Series([], Name: count, dtype: int64)

## New sku-nodes

In [183]:
dfNewSKUNodes = dfOutput[dfOutput["New NLC"] == "Yes"][["Product Code", "Identifier", "Final node level cost","Final node level cost category","Node type", "Min units"]].copy()
dfNewSKUNodes["Final target"] = "Added"
dfNewSKUNodes["Final price change category final"] = dfNewSKUNodes["Final target"]
dfNewSKUNodes["SKU-Node"] = dfNewSKUNodes["Product Code"] + "-" + dfNewSKUNodes["Identifier"].astype(str)
#SKU-Node not in dfTest SKU NODE

if test:
    dfNewSKUNodes = dfNewSKUNodes[~dfNewSKUNodes["SKU-Node"].isin(dfTest["SKU-Node"])].copy()

dfNewSKUNodesFinal = dfNewSKUNodes.rename(columns={"Product Code":"SKU","Final node level cost": "Price", "Identifier":"Source"}).drop(columns=["Final node level cost category"])


dfNewSKUNodesFinal


,SKU,Source,Price,Node type,Min units,Final target,Final price change category final,SKU-Node
82,ACCE0221417565H,628326984,47.31,HUB,8,Added,Added,ACCE0221417565H-628326984
98,ACCE0221417570T,628326984,50.16,HUB,8,Added,Added,ACCE0221417570T-628326984
680,ACCE0281316565T,6283261264,64.53,HUB,8,Added,Added,ACCE0281316565T-6283261264
926,ACCE0321826560VXL,6283261264,96.54,HUB,8,Added,Added,ACCE0321826560VXL-6283261264
976,ACCE0321924555Y,628326659,99.94,HUB,8,Added,Added,ACCE0321924555Y-628326659
...,...,...,...,...,...,...,...,...
2291551,YOKO3961829570E,628326916,317.84,HUB,4,Added,Added,YOKO3961829570E-628326916
2291765,YOKO39618351250E,628326916,350.91,HUB,4,Added,Added,YOKO39618351250E-628326916
2292269,YOKO3962029560E,628326570,377.12,HUB,4,Added,Added,YOKO3962029560E-628326570
2292293,YOKO3962029560E,628326916,401.87,HUB,4,Added,Added,YOKO3962029560E-628326916


In [184]:
dfNewSKUNodesTracker = dfNewSKUNodes.rename(columns={
    "Final node level cost category": "Final price category",
    "Final node level cost": "Final price"})
dfNewSKUNodesTracker["Start date"] = date_str
# dfNewSKUNodesTracker["SKU-Node"] = dfNewSKUNodesTracker["Product Code"] + "-" + dfNewSKUNodesTracker["Identifier"].astype(str)

dfNewSKUNodesTracker

,Product Code,Identifier,Final price,Final price category,Node type,Min units,Final target,Final price change category final,SKU-Node,Start date
82,ACCE0221417565H,628326984,47.31,11%,HUB,8,Added,Added,ACCE0221417565H-628326984,2026-03-18
98,ACCE0221417570T,628326984,50.16,6%,HUB,8,Added,Added,ACCE0221417570T-628326984,2026-03-18
680,ACCE0281316565T,6283261264,64.53,11%,HUB,8,Added,Added,ACCE0281316565T-6283261264,2026-03-18
926,ACCE0321826560VXL,6283261264,96.54,11%,HUB,8,Added,Added,ACCE0321826560VXL-6283261264,2026-03-18
976,ACCE0321924555Y,628326659,99.94,11%,HUB,8,Added,Added,ACCE0321924555Y-628326659,2026-03-18
...,...,...,...,...,...,...,...,...,...,...
2291551,YOKO3961829570E,628326916,317.84,8%,HUB,4,Added,Added,YOKO3961829570E-628326916,2026-03-18
2291765,YOKO39618351250E,628326916,350.91,11%,HUB,4,Added,Added,YOKO39618351250E-628326916,2026-03-18
2292269,YOKO3962029560E,628326570,377.12,11%,HUB,4,Added,Added,YOKO3962029560E-628326570,2026-03-18
2292293,YOKO3962029560E,628326916,401.87,11%,HUB,4,Added,Added,YOKO3962029560E-628326916,2026-03-18


In [185]:
dfNewSKUNodes["Final node level cost category"].value_counts()

Final node level cost category
11%    9065
8%      323
6%      187
Name: count, dtype: int64

# Other - Price changes check (no)

In [ ]:
# dfOutputChange = dfOutput[(~dfOutput["SKU-Node"].isin(dfNLCPriceUpdatesFinal["SKU-Node"])) & 
#                           (dfOutput["New NLC"] == "No")].copy()

# display(dfOutputChange["Final price change category"].value_counts())

# dontChange = []  #["Margin test", "Less sales","Updates test","Less sales back"]

# change = ["Less sales","Less sales back"]

# dfOutputChange = dfOutputChange[dfOutputChange["Final target"].isin(dontChange) == False].copy()
# dfOutputChange = dfOutputChange[dfOutputChange["Final target"].isin(change) == True].copy()

# display(dfOutputChange["Final price change category"].value_counts())


Final price change category
Decrease     1065027
No change     578253
Increase      445039
Name: count, dtype: int64

Final price change category
Decrease     35428
Increase     28681
No change     9488
Name: count, dtype: int64

In [ ]:
# wmMargin = 0.25

# dfHighWMMargin = dfOutput[(dfOutput["Current walmart margin at NLC"] > wmMargin) &  (~dfOutput["Final target"].isin(dontChange))].copy()
# display(dfHighWMMargin["current_nlc_margin category"].value_counts())
# dfHighWMMargin["Current walmart margin at NLC category"].value_counts()

current_nlc_margin category
10.8% to <11.2%    145512
8% to <10.8%        81060
13% to <15%         57415
11.2% to <13%       48722
>=17%               30696
15% to <17%         25730
6% to <8%           19799
0% to <6%            3301
<0%                   535
Name: count, dtype: int64

Current walmart margin at NLC category
25% to 30%    257715
>=30%         155055
<-10%              0
-10% to 0%         0
0% to 10%          0
10% to 15%         0
15% to 20%         0
20% to 25%         0
Name: count, dtype: int64

In [ ]:
# dfOutputChangeWithSales = dfOutputChange[(dfOutputChange["Product Code"].isin(dfSalesLastXDaysUniqueSKUs["Product Code"])) &
#                           (~dfOutputChange["Final target"].isin(dontChange))].copy()

# # dfOutputChangeWithSales = dfOutputChangeWithSales.merge(dfSalesSKUNodeFeb, how="left", on="SKU-Node")

# display(dfOutputChangeWithSales["NLC Price change category"].value_counts())

NLC Price change category
Decrease     147732
No change    128183
Increase      84181
Name: count, dtype: int64

In [ ]:
# dfOutputChangeWithSalesChange = dfOutputChangeWithSales[(dfOutputChangeWithSales["NLC Price change category"] != "No change") &
#                                                         (dfOutputChangeWithSales["NLC Price change %"].abs() > 0.01)&
#                                                         (dfOutputChangeWithSales["Start date"].isin(["2026-01-05","2026-01-20"]))].copy()
# display(dfOutputChangeWithSalesChange["NLC Price change category"].value_counts())
# display(dfOutputChangeWithSalesChange["Final target"].value_counts())
# display(dfOutputChangeWithSalesChange["Start date"].value_counts())

# dfCheck = dfOutputChangeWithSalesChange.groupby(["Final target", "Start date"]).agg({"Product Code": "count"}).reset_index()
# dfCheck = dfCheck.rename(columns={"Product Code": "count"})
# dfCheck

NLC Price change category
Decrease    35203
Increase    18846
Name: count, dtype: int64

Final target
Normal strategy    43935
Added               5161
No sales test       4404
Updated              549
Name: count, dtype: int64

Start date
2026-01-05    48339
2026-01-20     5710
Name: count, dtype: int64

,Final target,Start date,count
0,Added,2026-01-20,5161
1,No sales test,2026-01-05,4404
2,Normal strategy,2026-01-05,43935
3,Updated,2026-01-20,549


## Test update prices

In [ ]:
# dfTestSKUs = dfOutputChangeWithSalesChange.groupby("Product Code").agg({"total_inv_amount": "sum"}).reset_index()
# dfTestSKUs = split_skus_into_groups(dfTestSKUs,2,"Product Code","total_inv_amount")
# total_inv_test = dfTestSKUs["total_inv_amount"].sum()
# dfTestSKUs = dfTestSKUs[["Product Code", "group"]].copy()
# dfTestSKUs

In [ ]:
# total_inv_test/totalFebInvAmt

In [ ]:
# dfTestUpdate = dfOutputChangeWithSalesChange.merge(dfTestSKUs, how="left", on="Product Code")
# dfTestUpdate

In [ ]:
# dfTest = dfTestUpdate.copy()
# dfTest = dfTest[(dfTest["group"].notna()) & (dfTest["current_nlc_price"].notna())].copy()

# print("Total rows in test:", len(dfTest))

# dfTest["Group"] = np.where(dfTest["group"] == 1, "Update", np.where(dfTest["group"] == 2, "No change","Other"))

# dfTest["Test price"]= np.where(dfTest["group"] == 1, dfTest["Final node level cost"],
#                                       np.where(dfTest["group"] == 2, dfTest["current_nlc_price"], np.nan))

# dfTest["Delta test to current NLC %"] = round((dfTest["Test price"] - dfTest["current_nlc_price"])/dfTest["current_nlc_price"],4)
# dfTest["Delta test to current NLC category"] = np.where(dfTest["Delta test to current NLC %"] < 0, "Decrease",
#                                                                 np.where(dfTest["Delta test to current NLC %"] > 0, "Increase", "No change"))

# relevantCols = ["Product Code", "Identifier","SKU-Node", "Group", "Test price", "Delta test to current NLC %", "Delta test to current NLC category",
#                     "Node type", "Min units","current_nlc_margin","Final node level cost category"]



# dfTest = dfTest[relevantCols].copy().rename(columns = {"current_nlc_margin": "NLC margin before update"})

# # Now display # of increases, decreases, no change by each Group + average of "Delta test to current NLC %" by group
# summary = (
#     dfTest
#         .groupby("Group")
#         .agg(
#             count=("Delta test to current NLC category", "count"),
#             increases=("Delta test to current NLC category", lambda x: (x == "Increase").sum()),
#             decreases=("Delta test to current NLC category", lambda x: (x == "Decrease").sum()),
#             no_change=("Delta test to current NLC category", lambda x: (x == "No change").sum()),
#             avg_delta_test_to_current_nlc_pct=("Delta test to current NLC %", "mean"),
#         )
#         .reset_index()
# )
# summary

In [ ]:
# dfTestDSV= dfTest[["Product Code", "Identifier","Test price","SKU-Node"]].rename(columns = {"Product Code": "SKU", "Identifier": "Source", "Test price": "Price"})
# dfTestDSV

In [ ]:
# usecols = ["Product Code", "Identifier","Group", "Test price", "Delta test to current NLC category", "Delta test to current NLC %", 
#            "Node type", "Min units","NLC margin before update","Final node level cost category"]
# dfTestTracker = dfTest[usecols].copy()

# dfTestTracker["Final target"] = "Updates test"

# dfTestTracker = dfTestTracker.rename(columns={
#     "Group": "Sub-group",
#     "Final node level cost category":"Final price category",
#     "Delta test to current NLC category": "Final price change category final",
#     "Delta test to current NLC %": "Final delta vs current %",
#     "Test price": "Final price"})
# dfTestTracker["SKU-Node"] = dfTestTracker["Product Code"] + "-" + dfTestTracker["Identifier"].astype(str)
# dfTestTracker["Min units"] = dfTestTracker["Min units"].astype(int)
# dfTestTracker["Start date"] = date_str
# dfTestTracker

In [ ]:
# dfOutputChangeIncrease = dfOutputChange[dfOutputChange["NLC Price change category"] == "Decrease"].copy()
# display(dfOutputChangeIncrease["current_nlc_margin category"].value_counts())

In [ ]:
# len(dfOutputChange) + len(dfNLCPriceUpdatesFinal) + len(dfNewSKUNodesFinal) == len(dfOutput)

# Create new DSV

## Check original dsv + dsv starting point

In [186]:
dfCurrDSVOriginal

,sku,walmart_dsv_price,source
0,IRON0061417565H,40.11,62832635
1,IRON0081417565H,40.11,628326503
2,IRON0061417565H,40.11,628326106
3,IRON0061417565H,40.11,6283261141
4,APLS0031518555V,40.11,628326916
...,...,...,...
3278564,YOKO39618331250E,264.04,628326944
3278565,ZEET0012028555E,144.88,6283261112
3278566,ZEET0012028555E,144.88,628326584
3278567,ZETA0411726570S,104.49,6283261079


In [187]:
dfCurrDSVStart = dfCurrDSVOriginal.rename(columns={"sku": "SKU", "walmart_dsv_price":"Price", "source":"Source"})#.drop(columns=["date"])
dfCurrDSVStart["SKU-Node"] = dfCurrDSVStart["SKU"] + "-" + dfCurrDSVStart["Source"]
dfCurrDSVStart

,SKU,Price,Source,SKU-Node
0,IRON0061417565H,40.11,62832635,IRON0061417565H-62832635
1,IRON0081417565H,40.11,628326503,IRON0081417565H-628326503
2,IRON0061417565H,40.11,628326106,IRON0061417565H-628326106
3,IRON0061417565H,40.11,6283261141,IRON0061417565H-6283261141
4,APLS0031518555V,40.11,628326916,APLS0031518555V-628326916
...,...,...,...,...
3278564,YOKO39618331250E,264.04,628326944,YOKO39618331250E-628326944
3278565,ZEET0012028555E,144.88,6283261112,ZEET0012028555E-6283261112
3278566,ZEET0012028555E,144.88,628326584,ZEET0012028555E-628326584
3278567,ZETA0411726570S,104.49,6283261079,ZETA0411726570S-6283261079


## Update national prices (no)

In [124]:
dfCurrDSVStartNational = dfCurrDSVStart[dfCurrDSVStart["Source"].isna()].copy()
dfCurrDSVStartNational

,SKU,Price,Source,SKU-Node
0,KELL0011418570T,69.57,NaN,NaN
1,WDTO0314410B,4.14,NaN,NaN
2,WDTO073411400B,4.81,NaN,NaN
3,WDTO0734410B,4.84,NaN,NaN
4,OTRO022411400B,17.13,NaN,NaN
...,...,...,...,...
3062816,ZETA0041416555H,37.74,NaN,NaN
3062817,ZETA00517521575H,140.70,NaN,NaN
3062818,ZETA02518331250E,178.00,NaN,NaN
3062819,RADA10820331250E~wm,275.52,NaN,NaN


In [125]:
pathNewPrices = r"H:\Shared drives\07_Finance\07_10_Pricing_department\07_11_01_WalmartB2B\07_10_01_02 Pricing\2026-02\Walmart B2B Item Report 2.20.26.xlsx"
# sheetNewPrices = "DSV prices no increase"
# sheetNewPrices = "Final prices for the DSV"
sheetNewPrices = "National prices"
dfNewPrices = pd.read_excel(pathNewPrices, sheet_name=sheetNewPrices, dtype={"SKU": str}, skiprows=2)
dfNewPrices = dfNewPrices.rename(columns={"Min of Unit cost 3.5": "New Price", "Product Code": "SKU"})
dfNewPrices

,SKU,New Price
0,-,45.00
1,000060094858,180.30
2,00850004743751,42.00
3,00850004743768,45.00
4,00850004743775,45.00
...,...,...
61877,ZETA0411825555VXL,79.57
61878,ZETA0411826560V,82.63
61879,ZETA0411826565H,96.27
61880,ZETA0411827565H,101.21


In [126]:
dfNewNational = dfCurrDSVStartNational.merge(dfNewPrices, how="left", on="SKU")
dfNewNational["Final DSV price"] = np.where(dfNewNational["New Price"].isna(), dfNewNational["Price"], dfNewNational["New Price"])
dfNewNational["Price change %"] = round((dfNewNational["Final DSV price"] - dfNewNational["Price"]) / dfNewNational["Price"],4)
dfNewNational["Price change category"] = np.where(dfNewNational["Price change %"]>0, "Increase", np.where(dfNewNational["Price change %"]<0, "Decrease", "No change"))
display(dfNewNational["Price change category"].value_counts())
dfNewNationalFinal = dfNewNational[["SKU", "Final DSV price"]].rename(columns={"Final DSV price": "Price"})
dfNewNationalFinal

Price change category
No change    50486
Decrease      5843
Increase      3960
Name: count, dtype: int64

,SKU,Price
0,KELL0011418570T,69.57
1,WDTO0314410B,4.14
2,WDTO073411400B,4.81
3,WDTO0734410B,4.84
4,OTRO022411400B,12.85
...,...,...
60284,ZETA0041416555H,37.74
60285,ZETA00517521575H,140.70
60286,ZETA02518331250E,178.00
60287,RADA10820331250E~wm,275.52


In [127]:
dfCurrDSVNLCForDSV = dfCurrDSVNLC.rename(columns={"Product Code": "SKU", "current_nlc_price": "Price","Identifier": "Source"})
dfCurrDSVNLCForDSV

,SKU,Price,Source
50,IRON0061417565H,40.11,62832635
51,IRON0081417565H,40.11,628326503
52,IRON0061417565H,40.11,628326106
53,IRON0061417565H,40.11,6283261141
54,APLS0031518555V,40.11,628326916
...,...,...,...
3134570,PIRE0681826560V,212.25,628326761
3134571,PIRE0712228540YXL,590.66,628326761
3134572,PIRE1032023550YXL,311.36,628326761
3134573,PIRE1032225535YXL2,458.03,628326761


In [128]:
dfDSVNewNational = pd.concat([dfCurrDSVNLCForDSV, dfNewNationalFinal], ignore_index=True)
dfDSVNewNational["Minimum margin"] = "4%"
dfDSVNewNational = dfDSVNewNational[["SKU", "Price", "Minimum margin", "Source"]]
dfDSVNewNational

,SKU,Price,Minimum margin,Source
0,IRON0061417565H,40.11,4%,62832635
1,IRON0081417565H,40.11,4%,628326503
2,IRON0061417565H,40.11,4%,628326106
3,IRON0061417565H,40.11,4%,6283261141
4,APLS0031518555V,40.11,4%,628326916
...,...,...,...,...
3134570,ZETA0041416555H,37.74,4%,NaN
3134571,ZETA00517521575H,140.70,4%,NaN
3134572,ZETA02518331250E,178.00,4%,NaN
3134573,RADA10820331250E~wm,275.52,4%,NaN


In [129]:
dfDSVNewNational[dfDSVNewNational["SKU"].isin(dfRollbacks["Product Code"])]["Source"].unique()

array([nan], dtype=object)

In [ ]:
# outputPath = r"H:\Shared drives\07_Finance\07_10_Pricing_department\07_11_01_WalmartB2B\07_10_01_02 Pricing\2026-01\National pricing submission"
# file_name = f"Walmart B2B NLC Price Updates no Price increases.csv"
# full_path = f"{outputPath}\\{file_name}"
# dfDSVNewNational.to_csv(full_path, index=False)

In [130]:
dfCurrDSVStart = dfDSVNewNational.copy()
dfCurrDSVStart["SKU-Node"] = dfCurrDSVStart["SKU"] + "-" + dfCurrDSVStart["Source"]
dfCurrDSVStart

,SKU,Price,Minimum margin,Source,SKU-Node
0,IRON0061417565H,40.11,4%,62832635,IRON0061417565H-62832635
1,IRON0081417565H,40.11,4%,628326503,IRON0081417565H-628326503
2,IRON0061417565H,40.11,4%,628326106,IRON0061417565H-628326106
3,IRON0061417565H,40.11,4%,6283261141,IRON0061417565H-6283261141
4,APLS0031518555V,40.11,4%,628326916,APLS0031518555V-628326916
...,...,...,...,...,...
3134570,ZETA0041416555H,37.74,4%,NaN,NaN
3134571,ZETA00517521575H,140.70,4%,NaN,NaN
3134572,ZETA02518331250E,178.00,4%,NaN,NaN
3134573,RADA10820331250E~wm,275.52,4%,NaN,NaN


## RBs updates (no)

In [160]:
dfCurrDSVStartNLC = dfCurrDSVStart[dfCurrDSVStart["Source"].notna()].copy()
dfCurrDSVStartNational = dfCurrDSVStart[dfCurrDSVStart["Source"].isna()].copy()

dfCurrDSVStartNLCNoRbs = dfCurrDSVStartNLC[~dfCurrDSVStartNLC["SKU"].isin(dfRollbacks["Product Code"])].copy()
print("Removed", len(dfCurrDSVStartNLC) - len(dfCurrDSVStartNLCNoRbs), "SKUs with rollbacks from the NLC DSV start list")

dfCurrDSVStart = pd.concat([dfCurrDSVStartNLCNoRbs, dfCurrDSVStartNational], ignore_index=True)
dfCurrDSVStart

Removed 0 SKUs with rollbacks from the NLC DSV start list


,SKU,Price,Source,SKU-Node
0,DEES1758350A,5.07,628326607,DEES1758350A-628326607
1,DEES0434350B,5.11,6283261566,DEES0434350B-6283261566
2,DEES0704350B,5.13,6283261566,DEES0704350B-6283261566
3,NANC1876400A,5.21,6283261046,NANC1876400A-6283261046
4,NANC1876400A,5.21,628326765,NANC1876400A-628326765
...,...,...,...,...
2876204,BRID41924528575G,792.68,None,None
2876205,PIRE1032232535YXL3,793.02,None,None
2876206,GOYR03924511H,793.14,None,None
2876207,PIRE1062231535YXL2,793.22,None,None


In [161]:
dfCurrNationalRB = dfCurrDSVStartNational.merge(dfRollbacksSKUPrice, how="left", left_on="SKU", right_on="SKU")
dfCurrNationalRB["Delta price"] = round((dfCurrNationalRB["RB price"] - dfCurrNationalRB["Price"])/dfCurrNationalRB["Price"],4)
dfCurrNationalRB["Delta price category"] = np.where(dfCurrNationalRB["Delta price"] < 0, "Decrease",
                                                     np.where(dfCurrNationalRB["Delta price"] > 0, "Increase", "No change"))
display(dfCurrNationalRB[dfCurrNationalRB["Delta price category"] != "No change"])
display(dfCurrNationalRB["Delta price category"].value_counts())

dfCurrNationalRB["Price"] = np.where(dfCurrNationalRB["RB price"].isna(), dfCurrNationalRB["Price"], dfCurrNationalRB["RB price"])
dfCurrNationalRB = dfCurrNationalRB[["SKU", "Price"]].copy()

,SKU,Price,Source,SKU-Node,RB price,Delta price,Delta price category
48,LION0081519560V,37.48,None,NaN,35.76,-0.0459,Decrease
969,ADVN0221418565H,42.89,None,NaN,44.94,0.0478,Increase
3025,OTRO0281018850B,55.04,None,NaN,62.24,0.1308,Increase
3703,OTAN0071721555WXL,57.48,None,NaN,59.68,0.0383,Increase
4940,ATLA0031721565H,60.13,None,NaN,62.73,0.0432,Increase
4955,DELI0012024535WXL,60.19,None,NaN,67.16,0.1158,Increase
7205,ARRO0101823560VXL,70.36,None,NaN,70.88,0.0074,Increase
8619,OTAN0071823550WXL,72.91,None,NaN,73.35,0.0060,Increase
9124,ADVN0191822555H,73.80,None,NaN,80.11,0.0855,Increase
9318,OTAN0071823555WXL,75.46,None,NaN,76.11,0.0086,Increase


Delta price category
No change    59354
Increase        22
Decrease         2
Name: count, dtype: int64

In [ ]:
# dfCurrDSVStart[dfCurrDSVStart["SKU"].isin(dfRollbacks["Product Code"])]["Source"].unique()
# dfCurrDSVStartNoRbs = dfCurrDSVStart[~dfCurrDSVStart["SKU"].isin(dfRollbacks["Product Code"])].copy()

# dfCurrDSVStartRbsOk = pd.concat([dfCurrDSVStartNoRbs, dfRollbacksSKUPrice], ignore_index=True)
# dfCurrDSVStartRbsOk

In [162]:
dfRollbacksSKUPriceCheck = dfCurrNationalRB.merge(dfCurrDSVStartNational, how="left", on="SKU")
dfRollbacksSKUPriceCheck["Delta price"] = round(dfRollbacksSKUPriceCheck["Price_x"] - dfRollbacksSKUPriceCheck["Price_y"],2)
dfRollbacksSKUPriceCheck["Delta Price category"] = dfRollbacksSKUPriceCheck["Delta price category"] = np.where(dfRollbacksSKUPriceCheck["Delta price"] < 0, "Decrease",
                                                     np.where(dfRollbacksSKUPriceCheck["Delta price"] > 0, "Increase", "No change"))
dfRollbacksSKUPriceCheck["Delta Price category"].value_counts()

Delta Price category
No change    59354
Increase        22
Decrease         2
Name: count, dtype: int64

In [163]:
dfCurrDSVStart = pd.concat([dfCurrDSVStartNLCNoRbs, dfCurrNationalRB.rename(columns={"Price": "Price", "SKU": "SKU", "Source": "Source"})], ignore_index=True)
dfCurrDSVStart

,SKU,Price,Source,SKU-Node
0,DEES1758350A,5.07,628326607,DEES1758350A-628326607
1,DEES0434350B,5.11,6283261566,DEES0434350B-6283261566
2,DEES0704350B,5.13,6283261566,DEES0704350B-6283261566
3,NANC1876400A,5.21,6283261046,NANC1876400A-6283261046
4,NANC1876400A,5.21,628326765,NANC1876400A-628326765
...,...,...,...,...
2876204,BRID41924528575G,792.68,NaN,NaN
2876205,PIRE1032232535YXL3,793.02,NaN,NaN
2876206,GOYR03924511H,793.14,NaN,NaN
2876207,PIRE1062231535YXL2,793.22,NaN,NaN


## Normal update

In [188]:
original_rows = len(dfCurrDSVStart)

list_dfs_updates = [dfWmMarginSplitDSV, dfMarginTestDSV, dfNLCPriceUpdatesFinal, dfNLCPriceDecreasesDSV]

if test:
    list_dfs_updates = list_dfs_updates + [dfTestDSV]


dfUpdatesDSV = pd.concat(list_dfs_updates, ignore_index=True)

rows_to_update = len(dfUpdatesDSV)  

rows_to_add = len(dfNewSKUNodesFinal)

# if test:
#       rows_test = len(dfTest)
#       rows_added_by_test = len(dfTestDSV[~dfTestDSV["SKU-Node"].isin(dfCurrDSVStart["SKU-Node"])])
#       rows_update_test = rows_test - rows_added_by_test
#       rows_to_add+= rows_added_by_test
#       rows_to_update+= rows_update_test

final_rows = original_rows + rows_to_add

print("Original rows in Walmart DSV:", original_rows
      ,"\nRows to update:", rows_to_update
      # ,"\nRows to update (test):", rows_to_update_test
      ,"\nRows to add:", rows_to_add
      ,"\nFinal rows after update and add:", final_rows
      ,"\nTotal rows to update and add:", rows_to_add +rows_to_update) # + rows_to_update_test
final_rows

Original rows in Walmart DSV: 3278569 
Rows to update: 24566 
Rows to add: 9575 
Final rows after update and add: 3288144 
Total rows to update and add: 34141


3288144

In [189]:
dfCurrDSVRemovedUpdates = dfCurrDSVStart[~dfCurrDSVStart["SKU-Node"].isin(dfUpdatesDSV["SKU-Node"])].copy()

dfNewDSV = pd.concat([dfCurrDSVRemovedUpdates, dfUpdatesDSV,dfNewSKUNodesFinal], ignore_index=True).drop(columns=["SKU-Node"])
dfNewDSV["Minimum margin"] = "4%"
dfNewDSV = dfNewDSV[["SKU", "Price", "Minimum margin", "Source"]]
dfNewDSV = dfNewDSV.drop_duplicates()
dfNewDSV

,SKU,Price,Minimum margin,Source
0,IRON0061417565H,40.11,4%,62832635
1,IRON0081417565H,40.11,4%,628326503
2,IRON0061417565H,40.11,4%,628326106
3,IRON0061417565H,40.11,4%,6283261141
4,APLS0031518555V,40.11,4%,628326916
...,...,...,...,...
3288139,YOKO3961829570E,317.84,4%,628326916
3288140,YOKO39618351250E,350.91,4%,628326916
3288141,YOKO3962029560E,377.12,4%,628326570
3288142,YOKO3962029560E,401.87,4%,628326916


In [ ]:
# pathAddSKUsList = [r"H:\Shared drives\07_Finance\07_10_Pricing_department\07_11_01_WalmartB2B\07_10_01_02 Pricing\2026-02\New SKUs\Item Report_Walmart B2B_New SKUs from March 2026.xlsx",
#                    r"H:\Shared drives\07_Finance\07_10_Pricing_department\07_11_01_WalmartB2B\07_10_01_02 Pricing\2026-02\New SKUs\Item Report_Order 66_Priority 1 Phase 2A & 2B 030926.xlsx"]

# dfAddSKUsList = []
# for path in pathAddSKUsList:
#     df = pd.read_excel(path, usecols=["SKU", "Unit Cost","Order 66"])
#     #keep only when Order 66 = "No"
#     df = df[df["Order 66"] == "No"].copy()
#     #Drop col "Order 66"
#     df = df.drop(columns=["Order 66"])
#     df.rename(columns={ "Unit Cost": "Price"}, inplace=True)
#     df["Minimum margin"] = "4%"
#     df["Source"] = np.nan
#     print(len(df), "rows read from file:", path)

#     dfAddSKUsList.append(df)
    
# dfAddSKUs = pd.concat(dfAddSKUsList, ignore_index=True).drop_duplicates()
# dfAddSKUs


1906 rows read from file: H:\Shared drives\07_Finance\07_10_Pricing_department\07_11_01_WalmartB2B\07_10_01_02 Pricing\2026-02\New SKUs\Item Report_Walmart B2B_New SKUs from March 2026.xlsx
18 rows read from file: H:\Shared drives\07_Finance\07_10_Pricing_department\07_11_01_WalmartB2B\07_10_01_02 Pricing\2026-02\New SKUs\Item Report_Order 66_Priority 1 Phase 2A & 2B 030926.xlsx


,Price,SKU,Minimum margin,Source
0,144.41,ACCE0322123550V,4%,NaN
1,50.38,ACCE0531518555HXL,4%,NaN
2,119.12,ACCE0532228535WXL,4%,NaN
3,74.44,AIRL0038201000B,4%,NaN
4,80.10,APLS0151621570H,4%,NaN
...,...,...,...,...
1919,57.17,MATR0011520575C2,4%,NaN
1920,69.39,MATR0011522575D2,4%,NaN
1921,94.62,MATR0011623585E2,4%,NaN
1922,79.43,RADA0101824545WXL,4%,NaN


In [ ]:
# dfNewDSV = dfCurrDSVStart.copy()
# dfAddSKUsAlreadyInDSV = dfAddSKUs[dfAddSKUs["SKU"].isin(dfNewDSV["SKU"])].copy()
# dfAddNewSKUs = dfAddSKUs[~dfAddSKUs["SKU"].isin(dfNewDSV["SKU"])].copy()

# # dfNewDSVRemove = dfNewDSV[dfNewDSV["SKU"].isin(dfAddSKUs["SKU"])].copy()
# # dfNewDSVNoOverlap = dfNewDSV[~dfNewDSV["SKU"].isin(dfAddSKUs["SKU"])].copy()
# print("Number of overlapping SKUs between new DSV and add SKUs:", len(dfAddSKUsAlreadyInDSV))

# print("# SKUs to add:", len(dfAddNewSKUs))

# total_skus = len(dfNewDSV) + len(dfAddNewSKUs)
# print("Total SKUs in new DSV after adding new SKUs:", total_skus)

# dfNewDSV = pd.concat([dfNewDSV, dfAddNewSKUs], ignore_index=True)

# #Drop SKU-Node
# dfNewDSV = dfNewDSV.drop(columns=["SKU-Node"])

# dfNewDSV["Minimum margin"] = "4%"

# dfNewDSV

Number of overlapping SKUs between new DSV and add SKUs: 1
# SKUs to add: 1915
Total SKUs in new DSV after adding new SKUs: 3234726


,SKU,Price,Source,Minimum margin
0,IRON0061417565H,40.11,62832635,4%
1,IRON0081417565H,40.11,628326503,4%
2,IRON0061417565H,40.11,628326106,4%
3,IRON0061417565H,40.11,6283261141,4%
4,APLS0031518555V,40.11,628326916,4%
...,...,...,...,...
3234721,MATR0011520575C2,57.17,NaN,4%
3234722,MATR0011522575D2,69.39,NaN,4%
3234723,MATR0011623585E2,94.62,NaN,4%
3234724,RADA0101824545WXL,79.43,NaN,4%


In [191]:
current_year_month = datetime.now().strftime("%Y-%m")
outputPath = r"H:\Shared drives\07_Finance\07_10_Pricing_department\07_11_01_WalmartB2B\07_10_01_01 Node level costs\DSV Files"
outputPathMonth = os.path.join(outputPath, current_year_month)
if not os.path.exists(outputPathMonth):
    os.makedirs(outputPathMonth)
outputFileName = f"New WalmartB2B DSV to upload {date_str}.csv"
dfNewDSV.to_csv(os.path.join(outputPathMonth, outputFileName), index=False)

In [190]:
#Check we need to do every time:

dfNewDSVCheck = dfNewDSV.copy()
dfNewDSVCheck["Source"] = dfNewDSVCheck["Source"].fillna("National")
dfNewDSVCheck["SKU-Node"] = dfNewDSVCheck["SKU"] + "-" + dfNewDSVCheck["Source"]

dfCurrDSVOriginalToCheck = dfCurrDSVOriginal.copy()
dfCurrDSVOriginalToCheck = dfCurrDSVOriginalToCheck.copy()#.drop(columns=["date"])
dfCurrDSVOriginalToCheck["source"] = dfCurrDSVOriginalToCheck["source"].fillna("National")
dfCurrDSVOriginalToCheck["SKU-Node"] = dfCurrDSVOriginalToCheck["sku"] + "-" + dfCurrDSVOriginalToCheck["source"]


# rows_to_update_test_ok = len(dfTest[dfTest["Delta test to current NLC category"]!= "No change"])
# rows_to_update_test_ok  = len(dfTest)


dfNewDSVCheckMerged = dfNewDSVCheck.merge(dfCurrDSVOriginalToCheck, how="left", on="SKU-Node", suffixes=("_New", "_Old"))
dfNewDSVCheckMerged["Price change"] = round(dfNewDSVCheckMerged["Price"] - dfNewDSVCheckMerged["walmart_dsv_price"],2)
dfNewDSVCheckMerged["Price change category"] = np.where(dfNewDSVCheckMerged["Price change"]>0, "Increase",
                                                      np.where(dfNewDSVCheckMerged["Price change"]<0, "Decrease", 
                                                      np.where(dfNewDSVCheckMerged["walmart_dsv_price"].isna(),"New","No change")))

if test:
    dfNewDSVCheckMerged["Is in Test SKU-Nodes"] = dfNewDSVCheckMerged["SKU-Node"].isin(dfTestDSV["SKU-Node"])
    no_change_test = len(dfTestTracker[dfTestTracker["Final delta vs current %"]== 0])

dfNewDSVCheckMergedChanged = dfNewDSVCheckMerged[dfNewDSVCheckMerged["Price change"] != 0].copy()
print("Should be", rows_to_update + rows_to_add, "rows") #+ rows_to_update_test_ok

if test:
    print("Removing no change in test should be", rows_to_update + rows_to_add - no_change_test, "rows") #+ rows_to_update_test_ok

dfNewDSVCheckMergedChanged["Is national?"] = np.where(dfNewDSVCheckMergedChanged["Source"] == "National", "Yes", "No")
display(dfNewDSVCheckMergedChanged["Is national?"].value_counts())
display(dfNewDSVCheckMergedChanged["Price change category"].value_counts())
dfNewDSVCheckMergedChanged

Should be 34141 rows


Is national?
No    34122
Name: count, dtype: int64

Price change category
Increase    21692
New          9575
Decrease     2855
Name: count, dtype: int64

,SKU,Price,Minimum margin,Source,SKU-Node,sku,walmart_dsv_price,source,Price change,Price change category,Is national?
3254003,ARRO0011721555WXL,60.60,4%,6283261553,ARRO0011721555WXL-6283261553,ARRO0011721555WXL,59.85,6283261553,0.75,Increase,No
3254004,ARRO0011721555WXL,60.29,4%,628326799,ARRO0011721555WXL-628326799,ARRO0011721555WXL,61.35,628326799,-1.06,Decrease,No
3254005,ARRO0011824555W,102.66,4%,628326996,ARRO0011824555W-628326996,ARRO0011824555W,103.84,628326996,-1.18,Decrease,No
3254006,ARRO0012227550HXL,128.99,4%,6283261553,ARRO0012227550HXL-6283261553,ARRO0012227550HXL,127.30,6283261553,1.69,Increase,No
3254007,ARRO0071721565H,74.90,4%,6283261553,ARRO0071721565H-6283261553,ARRO0071721565H,73.56,6283261553,1.34,Increase,No
...,...,...,...,...,...,...,...,...,...,...,...
3288139,YOKO3961829570E,317.84,4%,628326916,YOKO3961829570E-628326916,NaN,NaN,NaN,NaN,New,No
3288140,YOKO39618351250E,350.91,4%,628326916,YOKO39618351250E-628326916,NaN,NaN,NaN,NaN,New,No
3288141,YOKO3962029560E,377.12,4%,628326570,YOKO3962029560E-628326570,NaN,NaN,NaN,NaN,New,No
3288142,YOKO3962029560E,401.87,4%,628326916,YOKO3962029560E-628326916,NaN,NaN,NaN,NaN,New,No


# Update tracker

In [192]:
dfCurrentTestsAllNew = dfCurrentTestsAll.copy()
dfCurrentTestsAllNew["SKU-Node"] = dfCurrentTestsAllNew["Product Code"] + "-" + dfCurrentTestsAllNew["Identifier"]

df_final_to_append_tracker = pd.concat([dfNewSKUNodesTracker, dfCurrDSVNLCCPriceUpdatesforTracker, 
                                            dfUpdateAboveMarginTracker, dfWmMarginSplitTracker, dfMarginTestTracker], ignore_index=True)

if test:
    df_final_to_append_tracker = pd.concat([df_final_to_append_tracker, dfTestTracker], ignore_index=True)

dfCurrentTestsAllNewNotUpdated = dfCurrentTestsAllNew[~dfCurrentTestsAllNew["SKU-Node"].isin(df_final_to_append_tracker["SKU-Node"])].copy()
dfCurrentTestsAllNew = pd.concat([dfCurrentTestsAllNewNotUpdated, df_final_to_append_tracker], ignore_index=True)
dfCurrentTestsAllNew = dfCurrentTestsAllNew.drop(columns=["SKU-Node"])

#Check if therea re duplicates in dfCurrentTestsAllNew by Product Code and Identifier
duplicates = dfCurrentTestsAllNew[dfCurrentTestsAllNew.duplicated(subset=["Product Code", "Identifier"], keep=False)]

if not duplicates.empty:
    print("Warning: There are duplicates in dfCurrentTestsAllNew based on Product Code and Identifier:")
    dfCurrentTestsAllNew = dfCurrentTestsAllNew.drop_duplicates(subset=["Product Code", "Identifier"], keep="last")
    display(duplicates)

# dfCurrentTestsAllNew = dfCurrentTestsAllNew.merge(dfCurrentTestsAllPrev, how="left", on=["Product Code", "Identifier"])

print("Final count of tests:", len(dfCurrentTestsAllNew))
print("Should be:", len(dfCurrentTestsAll) + len(dfNewSKUNodesTracker))

display(dfCurrentTestsAllNew["Final target"].value_counts().sort_index())
display(dfCurrentTestsAllNew["Start date"].value_counts().sort_index())

dfCurrentTestsAllNew

Final count of tests: 3281314
Should be: 3281405


Final target
Added                       1261444
Decreased - margin > 20%     133267
Less sales                    15808
Less sales back               15513
Margin test                  430874
No sales test                290673
Normal strategy              496537
Shipping cost added           31921
Updated                      361799
Updates test                 147495
Wm margin split test          95983
Name: count, dtype: int64

Start date
2026-01-05    748710
2026-01-16    117323
2026-01-20    231073
2026-01-27     61154
2026-01-28     16514
2026-01-29      4573
2026-01-30    419692
2026-02-03     91240
2026-02-04     18159
2026-02-05     16018
2026-02-06    148142
2026-02-09     31050
2026-02-11     46595
2026-02-12    177616
2026-02-16     48195
2026-02-17     16803
2026-02-18     24710
2026-02-19     32013
2026-02-20     11495
2026-02-24     23017
2026-02-25     55947
2026-02-26     11076
2026-02-27      9944
2026-03-02     18632
2026-03-03     41069
2026-03-04     19070
2026-03-05      9188
2026-03-06     13660
2026-03-07     30793
2026-03-09    106397
2026-03-10     64551
2026-03-11       430
2026-03-12    323921
2026-03-13    118329
2026-03-16     39124
2026-03-17    110625
2026-03-18     24466
Name: count, dtype: int64

,Product Code,Identifier,Final target,Final price category,Final price,Final delta vs current %,Final price change category final,Category inventory,Node type,Start date,Min units,Sub-group,Is in stock?,current_nlc_margin,Current walmart margin at NLC category,sku_sales_category,current_nlc_margin_date,Last price update
0,ACCE0131318560V,6283261261,Normal strategy,11%,70.06,0.0000,No change,NaN,HUB,2026-01-05,8,11%,No,NaN,NaN,No sales,2026-03-17,NaN
1,ACCE0131318560V,6283261264,Normal strategy,11%,70.76,0.0000,No change,NaN,HUB,2026-01-05,8,11%,No,NaN,NaN,No sales,2026-03-17,NaN
2,ACCE0131318560V,6283261391,Normal strategy,11%,67.87,NaN,New,NaN,HUB,2026-01-05,8,11%,No,NaN,NaN,No sales,2026-03-17,NaN
3,ACCE0131418555V,6283261391,Normal strategy,11%,56.13,-0.0200,Decrease,NaN,HUB,2026-01-05,8,11%,No,NaN,NaN,No sales,2026-03-17,NaN
4,ACCE0221316580T,6283261113,Normal strategy,11%,43.25,0.0300,Increase,NaN,HUB,2026-01-05,8,11%,Yes,0.1188,0% to 10%,95-100%,2026-03-17,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3281309,NEXE1631722555WXL,6283261419,Margin test,14%,110.13,0.0349,Increase,NaN,SPOKE,2026-03-12,4,14%,Yes,0.1400,25% to 30%,90-95%,2026-03-17,2026-03-18
3281310,NEXE1631722555WXL,6283261448,Margin test,14%,110.13,0.0349,Increase,NaN,SPOKE,2026-03-12,4,14%,Yes,0.1400,25% to 30%,90-95%,2026-03-17,2026-03-18
3281311,NEXE1631722555WXL,62832681,Margin test,14%,110.13,0.0349,Increase,NaN,HUB,2026-03-12,4,14%,Yes,0.1400,25% to 30%,90-95%,2026-03-17,2026-03-18
3281312,NEXE1632025555VXL,6283261417,Margin test,11%,150.02,0.0000,No change,NaN,SPOKE,2026-03-12,4,11%,Yes,0.1100,20% to 25%,80-90%,2026-03-17,2026-03-18


In [193]:
pathTrackerBk = r"H:\Shared drives\07_Finance\07_10_Pricing_department\07_11_01_WalmartB2B\07_10_01_01 Node level costs\Bk tracker\Final node level costs tracker_bk.csv"
pathTrackerBk = pathTracker.replace("Final node level costs tracker.csv", f"Bk tracker\Final node level costs tracker_{date_str}.csv")
dfCurrentTestsAllNew.to_csv(pathTracker, index=False)
print("Tracker updated")
dfCurrentTestsAllNew.to_csv(pathTrackerBk, index=False)
print("Backup created")

Tracker updated
Backup created


In [ ]:
dfCurrentTestsAllNew["Final target"].value_counts()

Final target
Added                       1250567
Normal strategy              519356
Margin test                  434451
No sales test                304616
Updated                      295688
Updates test                 158038
Decreased - margin > 20%     133360
Wm margin split test          96364
Less sales                    16540
Less sales back               15691
Shipping cost added            1274
Name: count, dtype: int64

In [250]:
dfCurrentTestsAllNew["Start date"].value_counts()

Start date
2026-01-05    783721
2026-01-30    431045
2026-03-12    307670
2026-01-20    235972
2026-02-12    188927
2026-02-06    150238
2026-03-13    138988
2026-01-16    122514
2026-03-09     95617
2026-02-03     94933
2026-03-10     68813
2026-01-27     63966
2026-02-25     58428
2026-02-16     52108
2026-03-16     49245
2026-02-11     48423
2026-03-03     44488
2026-02-19     33558
2026-02-09     32153
2026-02-18     25652
2026-02-24     24138
2026-03-02     19876
2026-03-04     19697
2026-02-04     19129
2026-02-17     18174
2026-01-28     17132
2026-02-05     16794
2026-03-06     14411
2026-02-26     12132
2026-02-20     12124
2026-02-27     10416
2026-03-05     10251
2026-01-29      4745
2026-03-11       467
Name: count, dtype: int64

In [ ]:
# dfPrevDSVOriginalFix = dfPrevDSVOriginal.copy()
# dfPrevDSVOriginalFix["SKU-Node"] = dfPrevDSVOriginalFix["sku"] + "-" + dfPrevDSVOriginalFix["source"]
# dfPrevDSVOriginalFix["SKU-Node-Price"] = dfPrevDSVOriginalFix["sku"] + "-" + dfPrevDSVOriginalFix["source"] + "-" + dfPrevDSVOriginalFix["walmart_dsv_price"].astype(str)

# dfPrevDSVOriginalFix = dfPrevDSVOriginalFix.rename(columns={"walmart_dsv_price": "prev_nlc_price", "sku": "Product Code", "source": "Identifier"}).drop(columns=["date"])
# dfPrevDSVOriginalFix

,Product Code,prev_nlc_price,Identifier,SKU-Node,SKU-Node-Price
0,PIRE0672232535VXL2,663.79,628326948,PIRE0672232535VXL2-628326948,PIRE0672232535VXL2-628326948-663.79
1,TOYO08522512H2,663.82,62832617,TOYO08522512H2-62832617,TOYO08522512H2-62832617-663.82
2,CONT51222511G,663.88,None,NaN,NaN
3,PIRE0712029545Y,663.95,628326996,PIRE0712029545Y-628326996,PIRE0712029545Y-628326996-663.95
4,CONT57022511H,663.97,628326542,CONT57022511H-628326542,CONT57022511H-628326542-663.97
...,...,...,...,...,...
3004221,GOYR64324511H,621.85,628326946,GOYR64324511H-628326946,GOYR64324511H-628326946-621.85
3004222,PIRE1182227535WXL,621.90,628326772,PIRE1182227535WXL-628326772,PIRE1182227535WXL-628326772-621.9
3004223,PIRE1182227535WXL,621.90,6283261142,PIRE1182227535WXL-6283261142,PIRE1182227535WXL-6283261142-621.9
3004224,GOYR00522371350F,622.00,628326772,GOYR00522371350F-628326772,GOYR00522371350F-628326772-622.0


In [ ]:
# dfCurrDSVOriginalFix = dfCurrDSVOriginal.copy()
# dfCurrDSVOriginalFix["SKU-Node"] = dfCurrDSVOriginalFix["sku"] + "-" + dfCurrDSVOriginalFix["source"]
# dfCurrDSVOriginalFix["SKU-Node-Price"] = dfCurrDSVOriginalFix["sku"] + "-" + dfCurrDSVOriginalFix["source"] + "-" + dfCurrDSVOriginalFix["walmart_dsv_price"].astype(str)

# dfCurrDSVOriginalFix["Start date"] = '2026-02-24'
# dfCurrDSVOriginalFix["Min units"] = 8
# dfCurrDSVOriginalFix["Node type"] = "HUB"
# dfCurrDSVOriginalFix["Final price category"] = "11%"
# dfCurrDSVOriginalFix = dfCurrDSVOriginalFix.rename(columns={"walmart_dsv_price": "Final price","sku": "Product Code", "source": "Identifier"}).drop(columns=["date"])
# dfCurrDSVOriginalFix["Final target"] = ""
# dfCurrDSVOriginalFix["Category inventory"] = ""
# dfCurrDSVOriginalFix

,Product Code,Final price,Identifier,SKU-Node,SKU-Node-Price,Start date,Min units,Node type,Final price category,Final target,Category inventory
0,KELL0011418570T,69.57,None,NaN,NaN,2026-02-24,8,HUB,11%,,
1,WDTO0314410B,4.14,None,NaN,NaN,2026-02-24,8,HUB,11%,,
2,WDTO073411400B,4.81,None,NaN,NaN,2026-02-24,8,HUB,11%,,
3,WDTO0734410B,4.84,None,NaN,NaN,2026-02-24,8,HUB,11%,,
4,OTRO022411400B,17.13,None,NaN,NaN,2026-02-24,8,HUB,11%,,
...,...,...,...,...,...,...,...,...,...,...,...
3027608,MICH63422530570L,986.13,628326261,MICH63422530570L-628326261,MICH63422530570L-628326261-986.13,2026-02-24,8,HUB,11%,,
3027609,MICH63422530570L,986.13,62832625,MICH63422530570L-62832625,MICH63422530570L-62832625-986.13,2026-02-24,8,HUB,11%,,
3027610,BKTA37324195LF,986.61,628326895,BKTA37324195LF-628326895,BKTA37324195LF-628326895-986.61,2026-02-24,8,HUB,11%,,
3027611,BKTA37324195LF,986.61,6283261393,BKTA37324195LF-6283261393,BKTA37324195LF-6283261393-986.61,2026-02-24,8,HUB,11%,,


In [ ]:
# dfNewSKUNodesTrackerPrev = dfCurrDSVOriginalFix[~dfCurrDSVOriginalFix["SKU-Node"].isin(dfPrevDSVOriginalFix["SKU-Node"])].copy()
# dfNewSKUNodesTrackerPrev["Final target"] = "Added"
# dfNewSKUNodesTrackerPrev

,Product Code,Final price,Identifier,SKU-Node,SKU-Node-Price,Start date,Min units,Node type,Final price category,Final target,Category inventory
773,CROP00115500C,40.83,628326916,CROP00115500C-628326916,CROP00115500C-628326916-40.83,2026-02-24,8,HUB,11%,Added,
777,CROP00115500C,40.83,6283261207,CROP00115500C-6283261207,CROP00115500C-6283261207-40.83,2026-02-24,8,HUB,11%,Added,
778,CROP00115500C,40.83,628326460,CROP00115500C-628326460,CROP00115500C-628326460-40.83,2026-02-24,8,HUB,11%,Added,
925,LASP0011518565H,41.01,628326870,LASP0011518565H-628326870,LASP0011518565H-628326870-41.01,2026-02-24,8,HUB,11%,Added,
1080,ARRO0011620545WXL,41.11,628326855,ARRO0011620545WXL-628326855,ARRO0011620545WXL-628326855-41.11,2026-02-24,8,HUB,11%,Added,
...,...,...,...,...,...,...,...,...,...,...,...
3027029,BKTA0012860065DA8,1773.59,628326921,BKTA0012860065DA8-628326921,BKTA0012860065DA8-628326921-1773.59,2026-02-24,8,HUB,11%,Added,
3027033,BKTA0012860065DA8,1773.59,6283261393,BKTA0012860065DA8-6283261393,BKTA0012860065DA8-6283261393-1773.59,2026-02-24,8,HUB,11%,Added,
3027050,FIRE2734248080B2,1867.96,6283261137,FIRE2734248080B2-6283261137,FIRE2734248080B2-6283261137-1867.96,2026-02-24,8,HUB,11%,Added,
3027051,FIRE2734248080B2,1867.96,628326467,FIRE2734248080B2-628326467,FIRE2734248080B2-628326467-1867.96,2026-02-24,8,HUB,11%,Added,


In [ ]:
# dfNewSKUNodesTracker = pd.concat([dfNewSKUNodesTrackerPrev, dfNewSKUNodesTracker], ignore_index=True)
# dfNewSKUNodesTracker["Start date"].value_counts()

Start date
2026-02-24    23387
2026-02-25    10857
Name: count, dtype: int64

In [ ]:
# dfCurrDSVNLCCPriceUpdatesforTrackerPrev = dfCurrDSVOriginalFix[~dfCurrDSVOriginalFix["SKU-Node-Price"].isin(dfPrevDSVOriginalFix["SKU-Node-Price"])].copy()
# dfCurrDSVNLCCPriceUpdatesforTrackerPrev = dfCurrDSVNLCCPriceUpdatesforTrackerPrev[~dfCurrDSVNLCCPriceUpdatesforTrackerPrev["SKU-Node"].isin(dfNewSKUNodesTrackerPrev["SKU-Node"])].copy()

# dfCurrDSVNLCCPriceUpdatesforTrackerPrev["Final target"] = "Updated"
# dfCurrDSVNLCCPriceUpdatesforTrackerPrev

,Product Code,Final price,Identifier,SKU-Node,SKU-Node-Price,Start date,Min units,Node type,Final price category,Final target,Category inventory
531,HIRU0078570D,40.45,6283261308,HIRU0078570D-6283261308,HIRU0078570D-6283261308-40.45,2026-02-24,8,HUB,11%,Updated,
3722,VANT0011518560H,42.70,62832622,VANT0011518560H-62832622,VANT0011518560H-62832622-42.7,2026-02-24,8,HUB,11%,Updated,
4678,IRON0081518560H,43.22,628326113,IRON0081518560H-628326113,IRON0081518560H-628326113-43.22,2026-02-24,8,HUB,11%,Updated,
4735,IRON0081518560H,43.22,6283261114,IRON0081518560H-6283261114,IRON0081518560H-6283261114-43.22,2026-02-24,8,HUB,11%,Updated,
7314,IRON0111519550V,44.26,628326159,IRON0111519550V-628326159,IRON0111519550V-628326159-44.26,2026-02-24,8,HUB,11%,Updated,
...,...,...,...,...,...,...,...,...,...,...,...
3025649,TOYO07822531580L,747.12,6283261566,TOYO07822531580L-6283261566,TOYO07822531580L-6283261566-747.12,2026-02-24,8,HUB,11%,Updated,
3025750,MICH2982131530YXL2,755.93,628326301,MICH2982131530YXL2-628326301,MICH2982131530YXL2-628326301-755.93,2026-02-24,8,HUB,11%,Updated,
3027173,PIRE1062231535YXL2,955.80,62832616,PIRE1062231535YXL2-62832616,PIRE1062231535YXL2-62832616-955.8,2026-02-24,8,HUB,11%,Updated,
3027533,PRIM0403235526,3588.79,628326833,PRIM0403235526-628326833,PRIM0403235526-628326833-3588.79,2026-02-24,8,HUB,11%,Updated,


In [ ]:
# dfCurrDSVNLCCPriceUpdatesforTracker = pd.concat([dfCurrDSVNLCCPriceUpdatesforTrackerPrev, dfCurrDSVNLCCPriceUpdatesforTracker], ignore_index=True)
# dfCurrDSVNLCCPriceUpdatesforTracker["Start date"].value_counts()

Start date
2026-02-25    52053
2026-02-24     7281
Name: count, dtype: int64

In [ ]:
# dfTargets = dfCurrentTestsAll[["Final target"]].drop_duplicates().copy()
# dfTargets.to_excel("Targets mapping.xlsx", index=False)
# dfTargetsFull = pd.read_excel("Targets mapping.xlsx")
# dfTargetsFull
# dfCurrentTestsAllNew[["Final target","Old final target","Start date"]].value_counts()
# dfCurrentTestsAllNew[["Final target", "Old final target", "Start date", "Min units", "Node type"]].value_counts().reset_index(name="count")
# 

In [ ]:
# dfCurrentTestsAllPrev = dfCurrentTestsAll.copy()
# dfCurrentTestsAllPrev["Final target prev"] = dfCurrentTestsAllPrev["Final target"]
# dfCurrentTestsAllPrev["Start date prev"] = dfCurrentTestsAllPrev["Start date"]
# dfCurrentTestsAllPrev = dfCurrentTestsAllPrev[["Product Code", "Identifier", "Final target prev", "Start date prev"]].copy()
# dfCurrentTestsAllPrev

,Product Code,Identifier,Final target prev,Start date prev
0,ACCE0131318560V,6283261261,Normal strategy,2026-01-05
1,ACCE0131318560V,6283261264,Normal strategy,2026-01-05
2,ACCE0131318560V,6283261391,Normal strategy,2026-01-05
3,ACCE0131418555V,6283261391,Normal strategy,2026-01-05
4,ACCE0221316580T,6283261113,Normal strategy,2026-01-05
...,...,...,...,...
3010990,IRON0121723550,628326173,Updated,2026-02-20
3010991,IRON0061823545V,628326185,Updated,2026-02-20
3010992,IRON0061823545V,6283261413,Updated,2026-02-20
3010993,NEXE0761520575S,628326275,Updated,2026-02-20


In [ ]:
# dfCurrentTestsAllNew["SKU-Node"] = dfCurrentTestsAllNew["Product Code"] + "-" + dfCurrentTestsAllNew["Identifier"]
# dfCurrentTestsAllNew = dfCurrentTestsAllNew.merge(dfOutput[["SKU-Node","current_nlc_margin"]], how = "left", on="SKU-Node")
# dfCurrentTestsAllNew["current_nlc_margin_date"] = today_str
# dfCurrentTestsAllNew = dfCurrentTestsAllNew.drop(columns=["SKU-Node"])

In [ ]:
# dfCurrentTestsAllNew = dfCurrentTestsAll.copy()

# #dfCurrentTestsAllNew["Final target prev"] is a copy of Final target

# dfCurrentTestsAllNew["SKU-Node"] = dfCurrentTestsAllNew["Product Code"] + "-" + dfCurrentTestsAllNew["Identifier"]

# dfCurrentTestsAllNewNotUpdated = dfCurrentTestsAllNew[~dfCurrentTestsAllNew["SKU-Node"].isin(dfDSVChanges["SKU-Node"])].copy()
# dfCurrentTestsAllNew = pd.concat([dfCurrentTestsAllNewNotUpdated, dfDSVChanges], ignore_index=True)


# # dfCurrentTestsAllNew["Node type"] = dfCurrentTestsAllNew["Node type"].fillna("HUB")
# dfCurrentTestsAllNew = dfCurrentTestsAllNew.drop(columns=["SKU-Node"])

# #Check if therea re duplicates in dfCurrentTestsAllNew by Product Code and Identifier
# duplicates = dfCurrentTestsAllNew[dfCurrentTestsAllNew.duplicated(subset=["Product Code", "Identifier"], keep=False)]

# if not duplicates.empty:
#     print("Warning: There are duplicates in dfCurrentTestsAllNew based on Product Code and Identifier:")
#     dfCurrentTestsAllNew = dfCurrentTestsAllNew.drop_duplicates(subset=["Product Code", "Identifier"], keep="last")
#     display(duplicates)


# Errors in price updates checks

Wait 3 hours since we uploaded the DSV file into hybris before running this. Flag to Fran if the % 

In [318]:
creds_path = ftp_server.create_path_cred(file_name = "ftp_credentials_fran.json")
host, port, username, password = ftp_server.read_credentials(creds_path)
ftp = ftp_server.connect_ftp(host, port, username, password)
path = "/external-merchants/WalmartB2B/price"

import pandas as pd

def ftp_list_names_only(ftp, path):
    ftp.cwd(path)
    names = ftp.nlst()
    return pd.DataFrame({"filename": names})

df = ftp_list_names_only(ftp,path)
# 1) extract datetime string
dt_str = df["filename"].str.extract(r"_(\d{8}-\d{6})_", expand=False)

# 2) convert to datetime
df["file_datetime"] = pd.to_datetime(dt_str, format="%Y%m%d-%H%M%S")

# 3) tag response files
df["is_response"] = df["filename"].str.contains("_response", case=False, na=False)


df["File date"] = df["file_datetime"].dt.date
df

Connected to FTP server: upload.tires-easy.com


,filename,file_datetime,is_response,File date
0,price_feed_20251015-023028_dc7714a8-3de7-4c77-...,2025-10-15 02:30:28,False,2025-10-15
1,price_feed_20251015-023028_dc7714a8-3de7-4c77-...,2025-10-15 02:30:28,True,2025-10-15
2,price_feed_20251015-024020_0f6d7c24-926c-4f05-...,2025-10-15 02:40:20,False,2025-10-15
3,price_feed_20251015-024020_0f6d7c24-926c-4f05-...,2025-10-15 02:40:20,True,2025-10-15
4,price_feed_20251015-025018_469e0cde-1578-4b2e-...,2025-10-15 02:50:18,False,2025-10-15
...,...,...,...,...
7326,price_feed_20260317-105341_485ed14c-097b-47c7-...,2026-03-17 10:53:41,True,2026-03-17
7327,price_feed_20260317-121512_e3959246-2aea-4eb4-...,2026-03-17 12:15:12,False,2026-03-17
7328,price_feed_20260317-121512_e3959246-2aea-4eb4-...,2026-03-17 12:15:12,True,2026-03-17
7329,price_feed_20260317-121712_3f2efe8d-8dd1-4d11-...,2026-03-17 12:17:12,False,2026-03-17


In [319]:
df["File date"].value_counts().sort_index(ascending=False)

File date
2026-03-17      56
2026-03-16      24
2026-03-13      44
2026-03-12     112
2026-03-11       4
2026-03-10      23
2026-03-09      32
2026-03-06      14
2026-03-05    1485
2026-03-04       8
2026-03-03      12
2026-03-02       8
2026-02-27       4
2026-02-26       6
2026-02-25      16
2026-02-24      10
2026-02-20       6
2026-02-19      31
2026-02-18      12
2026-02-17       9
2026-02-16      22
2026-02-13       2
2026-02-12     143
2026-02-11      96
2026-02-09      16
2026-02-06     110
2026-02-05      17
2026-02-04      33
2026-02-03    1290
2026-02-02    1283
2026-01-30     318
2026-01-29      28
2026-01-28       8
2026-01-26      29
2026-01-20     230
2026-01-19     182
2026-01-16      74
2026-01-05     927
2025-12-16       2
2025-12-15     156
2025-11-26      22
2025-11-25     263
2025-11-17      88
2025-11-12       2
2025-11-01      40
2025-10-30      12
2025-10-16      12
2025-10-15      10
Name: count, dtype: int64

In [320]:
dfUseful = df[df["File date"] >= pd.to_datetime(today_str).date()].copy()
dfUseful = dfUseful[dfUseful["is_response"] == True].copy()
dfUseful

,filename,file_datetime,is_response,File date
7276,price_feed_20260317-104531_82ecf54f-c542-43bb-...,2026-03-17 10:45:31,True,2026-03-17
7278,price_feed_20260317-104536_4a33900a-aa76-400c-...,2026-03-17 10:45:36,True,2026-03-17
7281,price_feed_20260317-104539_77b81e98-7b04-4557-...,2026-03-17 10:45:39,True,2026-03-17
7283,price_feed_20260317-104540_6ca36d0e-f25e-4a2c-...,2026-03-17 10:45:40,True,2026-03-17
7285,price_feed_20260317-104541_5c3dcf19-cc3e-4159-...,2026-03-17 10:45:41,True,2026-03-17
7287,price_feed_20260317-104733_345cccc4-9307-4ecc-...,2026-03-17 10:47:33,True,2026-03-17
7289,price_feed_20260317-104733_b9540d12-3392-450d-...,2026-03-17 10:47:33,True,2026-03-17
7291,price_feed_20260317-104734_2425464b-10e7-4954-...,2026-03-17 10:47:34,True,2026-03-17
7293,price_feed_20260317-104740_2f4f1988-4b90-45eb-...,2026-03-17 10:47:40,True,2026-03-17
7295,price_feed_20260317-104741_3218007b-34cd-470c-...,2026-03-17 10:47:41,True,2026-03-17


In [321]:
import os

folder_price_updates = r"H:\Shared drives\07_Finance\07_10_Pricing_department\07_11_01_WalmartB2B\07_10_01_01 Node level costs\Price updates check"
folder_today = os.path.join(folder_price_updates, today_str)
folder_xml= os.path.join(folder_today, "response_files")
os.makedirs(folder_xml, exist_ok=True)

ftp.cwd(path) #"/external-merchants/WalmartB2B/price"

count = 0
for fname in dfUseful["filename"]:
    local_path = os.path.join(folder_xml, fname)

    with open(local_path, "wb") as f:
        ftp.retrbinary(f"RETR {fname}", f.write)

    count += 1
    if count % 10 == 0:
        print(f"Downloaded {count}/{len(dfUseful)}")

print(f"Downloaded {len(dfUseful)} files to {folder_xml}")

Downloaded 10/27
Downloaded 20/27
Downloaded 27 files to H:\Shared drives\07_Finance\07_10_Pricing_department\07_11_01_WalmartB2B\07_10_01_01 Node level costs\Price updates check\2026-03-17\response_files


In [322]:
def read_xml_file(file_path_xml):
    ns_uri = 'http://walmart.com/'
    ns = {'ns': ns_uri}
    tag_item_status = f'{{{ns_uri}}}itemIngestionStatus'

    records = []

    for event, elem in ET.iterparse(file_path_xml, events=('end',)):
        if elem.tag == tag_item_status:
            index_elem = elem.find('ns:index', ns)
            product_id_elem = elem.find('.//ns:productId', ns)
            ship_node = elem.find('ns:shipNode', ns)
            status_elem = elem.find('ns:ingestionStatus', ns)

            # ---- error handling ----
            error_elem = elem.find('.//ns:ingestionError', ns)

            error_type = error_elem.find('ns:type', ns).text if error_elem is not None and error_elem.find('ns:type', ns) is not None else None
            error_code = error_elem.find('ns:code', ns).text if error_elem is not None and error_elem.find('ns:code', ns) is not None else None
            error_field = error_elem.find('ns:field', ns).text if error_elem is not None and error_elem.find('ns:field', ns) is not None else None
            error_description = error_elem.find('ns:description', ns).text if error_elem is not None and error_elem.find('ns:description', ns) is not None else None

            records.append({
                'index': index_elem.text if index_elem is not None else None,
                'productId': product_id_elem.text if product_id_elem is not None else None,
                'shipNode': ship_node.text if ship_node is not None else None,
                'ingestionStatus': status_elem.text if status_elem is not None else None,
                'error_type': error_type,
                'error_code': error_code,
                'error_field': error_field,
                'error_description': error_description
            })

            elem.clear()  # free memory

    df_xml = pd.DataFrame(records)
    return df_xml

In [323]:
df_xml_all = pd.DataFrame()
count = 0
for file in os.listdir(folder_xml):
    file_path_xml = os.path.join(folder_xml, file)
    df_xml = read_xml_file(file_path_xml)
    # print(f"File: {file}, Rows: {len(df_xml)}")
    # display(df_xml["ingestionStatus"].value_counts())
    df_xml_all = pd.concat([df_xml_all, df_xml], ignore_index=True)

    count += 1
    #every 10 print
    if count % 10 == 0:
        print(f"Processed {count} files")

display(df_xml_all)

print("Overall ingestion status counts:")
display(df_xml_all["ingestionStatus"].value_counts())


df_xml_all["SKU-Node"] = df_xml_all["productId"] + "-" + df_xml_all["shipNode"]
df_errors = df_xml_all[df_xml_all["ingestionStatus"] != "SUCCESS"].copy()
df_success = df_xml_all[df_xml_all["ingestionStatus"] == "SUCCESS"].copy()
df_errors_not_success = df_errors[~df_errors["SKU-Node"].isin(df_success["SKU-Node"])].copy()
df_errors_not_success_unique = df_errors_not_success.drop_duplicates(subset=["SKU-Node"], keep="last")
df_errors_not_success_unique["ingestionStatus"].value_counts()

df_xml_all_unique = pd.concat([df_success, df_errors_not_success_unique], ignore_index=True)

print("Ingestion status counts for unique SKU-Nodes:")
display(df_xml_all_unique["ingestionStatus"].value_counts())
# df_errors_not_success_unique[["error_field", "error_description"]].value_counts().reset_index(name="count")

Processed 10 files
Processed 20 files


,index,productId,shipNode,ingestionStatus,error_type,error_code,error_field,error_description
0,0,GTRA0881826560H,628326122,SUCCESS,None,None,None,None
1,1,GTRA0881826560H,628326466,SUCCESS,None,None,None,None
2,2,NITT0582029545VXL,6283261459,SUCCESS,None,None,None,None
3,3,NITT0582029545VXL~wm,6283261459,SUCCESS,None,None,None,None
4,4,NITT0582029545VXL,628326693,SUCCESS,None,None,None,None
...,...,...,...,...,...,...,...,...
260841,9795,GOYR2042127545H,6283261260,SUCCESS,None,None,None,None
260842,9796,GOYR2042127545H,6283261259,SUCCESS,None,None,None,None
260843,9797,GOYR2042127545H,6283261104,SUCCESS,None,None,None,None
260844,9798,GOYR2042127545H,6283261261,SUCCESS,None,None,None,None


Overall ingestion status counts:


ingestionStatus
SUCCESS       258123
DATA_ERROR      2723
Name: count, dtype: int64

Ingestion status counts for unique SKU-Nodes:


ingestionStatus
SUCCESS       258123
DATA_ERROR      2723
Name: count, dtype: int64

In [324]:
# Total not successes
not_success = len(df_xml_all_unique[df_xml_all_unique["ingestionStatus"] != "SUCCESS"])
total_success = len(df_xml_all_unique[df_xml_all_unique["ingestionStatus"] == "SUCCESS"])

perc_failure = round(not_success / (not_success + total_success),4)
print(f"Failure rate: {perc_failure:.2%} with {not_success} failures and {total_success} successes.")
if perc_failure > perc_failure_flag:
    print(f"ALERT: High failure rate of {perc_failure:.2%} with {not_success} failures and {total_success} successes.")
    #Flag to Fran.

Failure rate: 1.04% with 2723 failures and 258123 successes.


In [325]:
df_success_by_sku = df_success.groupby("productId").size().reset_index(name="count").sort_values(by="count", ascending=False)

# df_errors_content_sku = df_errors_not_success_unique[df_errors_not_success_unique["error_field"] == "content.SKU"].copy()
# df_errors_content_sku = df_errors_not_success_unique[df_errors_not_success_unique["error_field"] == "unitCost.currencyAmount"].copy()
df_errors_content_sku = df_errors_not_success_unique.copy()

df_errors_content_sku_agg = df_errors_content_sku.groupby(["productId","error_field"]).size().reset_index(name="count").sort_values(by="count", ascending=False)
df_errors_content_sku_agg_with_success = df_errors_content_sku_agg.merge(df_success_by_sku, how="left", on="productId", suffixes=("_errors", "_success"))

df_errors_content_sku_agg_with_success["count_success"] = df_errors_content_sku_agg_with_success["count_success"].fillna(0).astype(int)
df_errors_content_sku_agg_with_success["Has success"] = np.where(df_errors_content_sku_agg_with_success["count_success"]>0, "Yes", "No")
df_errors_content_sku_agg_with_success["% errors"] = round(df_errors_content_sku_agg_with_success["count_errors"] / (df_errors_content_sku_agg_with_success["count_errors"] + df_errors_content_sku_agg_with_success["count_success"]),4)


df_success_by_node = df_success.groupby("shipNode").size().reset_index(name="count").sort_values(by="count", ascending=False)
df_errors_content_node = df_errors_not_success_unique[df_errors_not_success_unique["error_field"] == "content.legacyDistributorId"].copy()
# df_errors_content_sku = df_errors_not_success_unique[df_errors_not_success_unique["error_field"] == "unitCost.currencyAmount"].copy()

# df_errors_content_node = df_errors_not_success_unique.copy()

#Group by productId and count rows
df_errors_content_node_agg = df_errors_content_node.groupby("shipNode").size().reset_index(name="count").sort_values(by="count", ascending=False)

df_errors_content_node_agg_with_success = df_errors_content_node_agg.merge(df_success_by_node, how="left", on="shipNode", suffixes=("_errors", "_success"))


df_errors_content_node_agg_with_success["count_success"] = df_errors_content_node_agg_with_success["count_success"].fillna(0).astype(int)
df_errors_content_node_agg_with_success["Has success"] = np.where(df_errors_content_node_agg_with_success["count_success"]>0, "Yes", "No")
df_errors_content_node_agg_with_success["% errors"] = round(df_errors_content_node_agg_with_success["count_errors"] / (df_errors_content_node_agg_with_success["count_errors"] + df_errors_content_node_agg_with_success["count_success"]),4)
df_errors_content_node_agg_with_success = df_errors_content_node_agg_with_success.sort_values(by="% errors", ascending=False)
df_errors_content_node_agg_with_success

,shipNode,count_errors,count_success,Has success,% errors
74,6283261198,1,11,Yes,0.0833
90,6283261230,1,17,Yes,0.0556
1,6283261554,2,34,Yes,0.0556
29,628326931,1,18,Yes,0.0526
38,628326435,1,20,Yes,0.0476
...,...,...,...,...,...
80,6283261269,1,1187,Yes,0.0008
42,628326973,1,1185,Yes,0.0008
25,6283261347,1,1320,Yes,0.0008
26,6283261337,1,1316,Yes,0.0008


In [326]:
#send to excel, one sheet with summary of df_xml_all["ingestionStatus"].value_counts(), other sheet the df_xml_all where ingestionStatus != "SUCCESS"
output_excel_path = os.path.join(folder_today, f"NLC Price update response summary {today_str}.xlsx")
with pd.ExcelWriter(output_excel_path, engine='xlsxwriter') as writer:
    df_summary = df_xml_all_unique["ingestionStatus"].value_counts().reset_index()
    df_summary["percent"] = (
        df_summary["count"] / df_summary["count"].sum()
    )


    df_summary.columns = ["ingestionStatus", "Counts", "Percentage"]
    df_summary.to_excel(writer, sheet_name='Summary', index=False)

    df_errors_content_sku_agg_with_success.to_excel(writer, sheet_name='Errors by SKU-Reason', index=False)

    df_summary_errors = df_errors_not_success_unique["error_description"].value_counts().reset_index()
    df_summary_errors.columns = ["error_description", "Counts"]

    df_summary_errors.to_excel(writer, sheet_name='Error Summary', index=False)

    df_errors_not_success_unique.to_excel(writer, sheet_name='Errors list', index=False)